In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
# ============================================================================================
# HF BOOTSTRAP — run FIRST.  (generalized: works for ANY timm encoder, tagged or bare)
#
# FACT: Colab gives every new notebook a FRESH VM. Local caches do NOT persist. So the cache
# must LIVE on Drive (HF_HOME), not be copied to it after the fact. Download succeeds ONCE,
# ever, anywhere -> every future runtime finds the weights already on Drive and goes straight
# offline without a single network call (no download, no revalidation ping).
#
# NOTHING about the encoder is hardcoded below. The HF repo id, the on-disk cache directory
# and the expected file size are all DERIVED from timm's own pretrained cfg. Two bugs this
# kills permanently:
#   (a) the cache dir carries the PRETRAINED TAG, not the bare arch name --
#       'tf_efficientnetv2_l' lives in models--timm--tf_efficientnetv2_l.in21k_ft_in1k/
#   (b) the expected file size is per-encoder (v2_l ~474MB vs v2_xl ~832MB), so a size floor
#       copied from another encoder rejects a perfectly good download as a "partial".
# Change ENC_TIMM and nothing else.
# ============================================================================================
import os, sys, time, glob, subprocess

ENC_TIMM = 'tf_efficientnetv2_l.in21k_ft_in1k'   # tagged is preferred: pins the weights so a
                                                 # timm upgrade can't silently change the default
DRIVE_HF = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/hf_home'
N_TRIES  = 6

# --- 0. the cache IS the Drive folder (must be set before any huggingface_hub import) --------
os.makedirs(DRIVE_HF, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF
for _v in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
    os.environ.pop(_v, None)                     # public repo: send NO auth (stale tokens 403)

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import timm
except ImportError:
    _pip('timm'); import timm
from timm.models import get_pretrained_cfg

# --- 1. DERIVE everything from timm's own cfg -----------------------------------------------
_ARCH = ENC_TIMM.split('.')[0]                   # arch name without the pretrained tag
_cfg  = get_pretrained_cfg(ENC_TIMM)
HF_REPO = getattr(_cfg, 'hf_hub_id', None)
assert HF_REPO, (
    f'{ENC_TIMM!r} resolves to tag {getattr(_cfg, "tag", None)!r} with NO hf_hub_id -> this timm '
    f'build has no pretrained weights for it. Beware the un-prefixed EfficientNetV2 names '
    f'(efficientnetv2_l etc.) — they are architecture-only and would train from RANDOM init '
    f'without ever raising. Use the tf_* variants.')

CACHE_DIR = 'models--' + HF_REPO.replace('/', '--')
SNAP_GLOB = os.path.join(DRIVE_HF, 'hub', CACHE_DIR, 'snapshots', '*')
_WEIGHT_FILES = ('model.safetensors', 'pytorch_model.bin')

# Expected size from the actual parameter count (fp32), with slack for fp16 storage + metadata.
# Costs a few seconds of CPU and ZERO network — no weights are fetched by pretrained=False.
_N_PARAM  = sum(p.numel() for p in timm.create_model(_ARCH, pretrained=False).parameters())
MIN_BYTES = int(_N_PARAM * 4 * 0.45)             # 0.45 clears fp16 (~0.5x) yet rejects truncation

print(f'encoder : {ENC_TIMM}')
print(f'  tag   : {_cfg.tag}   params {_N_PARAM/1e6:.1f}M  (fp32 ~{_N_PARAM*4/1e6:.0f}MB)')
print(f'  repo  : {HF_REPO}')
print(f'  cache : {os.path.join(DRIVE_HF, "hub", CACHE_DIR)}')
print(f'  floor : {MIN_BYTES/1e6:.0f}MB')

def _found_files():
    out = []
    for fn in _WEIGHT_FILES:
        out += glob.glob(os.path.join(SNAP_GLOB, fn))
    return out

def _weights_on_drive():
    return any(os.path.getsize(f) > MIN_BYTES for f in _found_files())

def _go_offline():
    """Offline must be applied to the LIVE huggingface_hub too: the library snapshots
    HF_HUB_OFFLINE into constants at import time, so setting only the env var has no effect
    on an already-imported hub (exactly the case after a download)."""
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    try:
        from huggingface_hub import constants as _C
        _C.HF_HUB_OFFLINE = True
    except Exception:
        pass

# --- 2. OFFLINE-FIRST: if the weights are already on Drive, never contact HF at all ----------
if _weights_on_drive():
    _go_offline()
    print('Weights found on Drive -> HF hub OFFLINE from the start. Zero network calls.')
else:
    os.environ.pop('HF_HUB_OFFLINE', None)
    print('Weights not on Drive yet -> one-time download (this is the LAST time ever).')
    _ok, _last = False, None
    for _try in range(N_TRIES):
        # alternate transports: even tries = classic HTTP bridge, odd tries = native Xet
        _path = 'classic-HTTP' if _try % 2 == 0 else 'xet'
        if _try % 2 == 0:
            os.environ['HF_HUB_DISABLE_XET'] = '1'
        else:
            os.environ.pop('HF_HUB_DISABLE_XET', None)
        try:
            _m = timm.create_model(ENC_TIMM, pretrained=True); del _m
            _ok = True
            print(f'Download OK on attempt {_try+1} ({_path} path) -> saved to Drive cache')
            break
        except Exception as _e:
            _last = f'{type(_e).__name__}: {str(_e)[:200]}'
            print(f'  attempt {_try+1}/{N_TRIES} ({_path}) failed: {_last[:110]}')
            time.sleep(10 * (_try + 1))

    # Two DISTINCT failures with opposite remedies — never collapse them into one message.
    assert _ok, (
        f'All {N_TRIES} attempts failed — HF unreachable from this region right now. Re-run this '
        f'cell in a few minutes; ONE success is permanent (cache lives on {DRIVE_HF}).\n'
        f'Last error: {_last}')
    assert _weights_on_drive(), (
        f'Download reported SUCCESS but the detector found no qualifying file.\n'
        f'  looked for : {_WEIGHT_FILES} under {SNAP_GLOB}\n'
        f'  size floor : {MIN_BYTES/1e6:.0f}MB\n'
        f'  files seen : {[(os.path.basename(f), f"{os.path.getsize(f)/1e6:.0f}MB") for f in _found_files()] or "NONE"}\n'
        f'  dirs in cache: {sorted(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_HF, "hub", "models--*")))}\n'
        f'This is a DETECTOR bug, not a network failure — the weights are probably on Drive. '
        f'Check that CACHE_DIR ({CACHE_DIR}) matches a directory listed above.')
    _go_offline()
    print('HF hub OFFLINE for the rest of the run.')

# --- 3. prove the exact training-loop call works offline from the Drive cache ----------------
try:
    import segmentation_models_pytorch as smp
except ImportError:
    _pip('segmentation_models_pytorch'); import segmentation_models_pytorch as smp

ENC_SMP = 'tu-' + ENC_TIMM          # the string to pass as encoder_name everywhere downstream
_probe = smp.Unet(encoder_name=ENC_SMP, encoder_weights='imagenet', in_channels=5, classes=1)
_enc_ch = _probe.encoder.out_channels
del _probe
print(f'VERIFIED: smp.Unet(encoder_name={ENC_SMP!r}, encoder_weights="imagenet") builds OFFLINE '
      f'from the Drive cache.')
print(f'  encoder out_channels: {_enc_ch}')
print('The training loop cannot hang or 403 on model creation — it never touches the network.')

In [ ]:

import os, zipfile, time
ZIP_PATH = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/rogii-wellbore-geology-prediction.zip'
UNZIP_DIR = '/content/rogii'
if IN_COLAB:
    if os.path.exists(UNZIP_DIR) and len(os.listdir(UNZIP_DIR)) > 0:
        print(f'Dataset already at {UNZIP_DIR}.')
    else:
        os.makedirs(UNZIP_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf: zf.extractall(UNZIP_DIR)
    DATA_ROOT = UNZIP_DIR
else:
    DATA_ROOT = '/path/to/rogii-wellbore-geology-prediction'

# === Single artifact folder for FT26 ===

############# Group A #############
#  timm-efficientnet-b8, tu-tf_efficientnetv2_l, tu-tf_efficientnetv2_xl.in21k_ft_in1k, tu-tf_efficientnet_b7.ns_jft_in1k


# === Locked configuration ===
SEED = 34
SEEDS = [SEED, SEED * 2, SEED * 3, SEED * 4, SEED * 5, SEED * 6, SEED * 7]

ENCODER      = 'tu-tf_efficientnetv2_l.in21k_ft_in1k'
ENCODER_SHORT = 'effnetv2_l'                # SANITY: set both back to b7 + STACK_N=1 -> reproduce 6.541.
DROP_CONNECT_RATE = 0.2
drop_path_rate=DROP_PATH_RATE = 0.2

DECODER_CHANNELS = (512, 256, 128, 64, 32)    # (768,384,192,96,48),   (384,224,160,144,144),   (512, 256, 128, 64, 32)
DECODER_INTERP = "nearest"   # native SMP decoder upsampling: "nearest" (default) or "bilinear". PixelShuffle removed from this fork.

# === POSTERIOR-PLANE MD DILATION (trajectory-consistency on the stage-1 handoff) ===
POSTERIOR_DILATION        = False  # insert a residual MD-only dilated 1-D stack on the stage-1 posterior plane,
#   BEFORE stage 2 ingests it. Rows-as-channels + dilations grow the MD receptive field at FULL row resolution
#   -> long-range trajectory consistency / de-drift. Zero-init residual: False (skip) is bit-identical baseline;
#   True starts == the bare handoff and learns the correction. Set True to experiment.
POSTERIOR_DILATION_FREEZE = False   # freeze the synth-pretrained dilation stack during
#   FINETUNE (synth learns the MD-consistency prior on 15k wells; finetune can't re-overfit it on 620). Toggle vs unfrozen.
POST_DIL_RATES            = (1, 2, 4, 8, 16)  # MD dilations (exp. RF growth, ~+-31 cols). Extend for longer range.
POST_DIL_EMBED            = 128                 # rows(256)->embed bottleneck for the MD stack (cheaper, less overfit surface)

# === MULTI-SCALE POSTERIOR INJECTION (stage-2 decoder) ===
# post1 currently enters stage-2 ONCE at the stem; stem stride-2 conv + 4 more downsamples (stride 32)
# blur away the row-precision that is the whole point of the posterior (mass in a few rows/column). The
# decoder rebuilds spatial size only from encoder features (which never saw the sharp posterior), so the
# lost row detail can't be recovered. This re-supplies post1 at EACH decoder scale: area-pool it to the
# block's resolution and concat onto that block's conv1 input (conv1 widened by 1 plane, ZERO-INIT ->
# starts bit-identical, learns on top). Area-pool (not bilinear) marginalises row mass per bin. Targets
# DRIFT (within-well row shape), NOT datum (the unanchored offset). False -> stem-only = baseline.
MULTISCALE_POST_INJECT = False

# === AMP (mixed precision) — DEFAULT OFF = bit-for-bit identical to the current fp32 path ===
USE_AMP   = True        # True -> autocast the TRAIN forward/backward (conv/matmul in low precision; softmax,
AMP_DTYPE = 'bfloat16'    # 'float16' (any tensor-core GPU; uses GradScaler) or 'bfloat16' (Ampere+ A100/L4)

STACK_N               = 2   # 2 = stacked (2 UNets, posterior handoff); 1 = single UNet.
STACK_DEEP_SUP_WEIGHT = 0.5
STACK_DETACH          = True

CORRUPTION_MODE = 'clean'
N_SYNTH_PRETRAIN = 25000

# === [EVEX-A] Event-expansion pack knobs (ported VERBATIM from the validated A2BETRAY build) ===
# CORRESPONDENCE-PRESERVING geometry class: labels and lookup move together -> HW<->TW match_corr
# untouched by construction (the gate is the class detector). Pinch ships DORMANT (A2-class,
# reclassified by the 2026-07-11 real-data pregate; do not arm in this pack). v4 rates of record:
# dipbrk 0.35 / repeat 0.12 / standard 20-80% window. EVEX-B's 0.45/0.18/95% dose is TOMBSTONED;
# TOEEV (toe-shifted placement) retired 2026-07-19 as a wash -> the standard window STAYS.
EVENT_PACK = 'v3_expA'            # 'v3' = bit-identical pack-free baseline | 'v3_expA' = armed (v4 pool of record)
DIPBREAK_PROB, DIPBREAK_SECOND_P, DIPBREAK_FAULT_COUPLE_P = 0.35, 0.30, 0.30
DIPBREAK_DSLOPE_MIN, DIPBREAK_DSLOPE_MAX = 0.008, 0.035    # ft per md-step, signed
PINCH_PROB, PINCH_H_MIN, PINCH_H_MAX = 0.00, 10.0, 30.0    # DORMANT (A2-class) — machinery banked, rate 0
PINCH_C_MIN, PINCH_C_MAX = 0.05, 0.20
REPEAT_PROB, REPEAT_SHIFT_MIN, REPEAT_SHIFT_MAX = 0.12, 10.0, 40.0
REPEAT_DAMP_MIN, REPEAT_DAMP_MAX, REPEAT_L_MIN, REPEAT_L_MAX = 0.2, 0.5, 150, 400
EXPA_MAX_ABS_DIP = 0.050    # ft/step cap on the pack's ADDED |dTVT| excl. discrete jumps (real p99 = 0.070)
EXPA_EXC_CAP_FT  = 320.0    # excursion cap (pregate-confirmed); FZ pair guard reuses it
EXPA_HASH = 3714159261      # pack RNG stream (registered, ledger §21.16)
print('EVENT PACK: ' + ('v3_expA [ARMED] dipbrk=%.2f repeat=%.2f (pinch dormant=%.2f, A2-class)'
      % (DIPBREAK_PROB, REPEAT_PROB, PINCH_PROB) if EVENT_PACK == 'v3_expA' else 'v3 (bit-identical baseline)'))

# === [FZ] FAULT-ZONE ARCHITECTURE bundle — THE ARM ============================================
# One CORRESPONDENCE-PRESERVING invariance class, three configurations the generator has never
# emitted (verified 2026-07-19: native faults are constant-throw single-row jumps, <=2/well,
# independent-uniform-placed):
#   [FZ-1] growth throw    — throw ceiling scales with lateral position (the play's documented
#                            growth-fault mechanics: Karnes Trough / Charlotte-Jourdanton / NLOFZ).
#   [FZ-2] antithetic pair — graben/horst: opposite-sign close-spaced faults, section partially-
#                            to-fully recovers (the section-recovery alias the model has never seen).
#   [FZ-3] drag ramps      — throw accrued over a finite smoothstep ramp (fault<->monocline
#                            ambiguity; the 1-D import of the FaultSeg displacement-gradient recipe).
# All LABEL-SIDE geometry, lookup untouched -> match_corr flat is the class gate. RNG: OWN stream
# (FZ_HASH) consumed only when armed -> FZ_ENABLE=False is bit-identical to the v4 generator with
# ZERO extra base-stream draws. LIMIT IDENTITY (smoke-asserted): armed with (PAIR_PROB=0,
# RAMP_SHARP_P=1, GROWTH_FLOOR=1) == off, byte-for-byte.
# PRE-MARKED INTENTIONAL DEVIATION (freeze into the ledger at launch): |dTVT| p99 rises on RAMP
# COLUMNS ONLY (jump-class carve-out; max discrete slope 1.5*throw/width <= FZ_RAMP_MAX_SLOPE).
# Pooled excursion stays pinned: the pair is guarded vs max(base-with-events, EXPA_EXC_CAP_FT),
# redraw<=6 then drop+count (echoed at pool build).
FZ_ENABLE = False                # master; False = bit-identical v4 generator
FZ_HASH   = 1414213562          # NEW RNG stream (verified unused vs EXPA/BETRAY/bump/B2/unroll streams; register in ledger §21.16 on merge)
FZ_GROWTH_ENABLE = True         # [FZ-1] |mag| draw U(min,max) -> U(min, min+(max-min)*s(pos)), s = FLOOR..1 across the window
FZ_GROWTH_FLOOR  = 0.40         #        throw ceiling at the window start (x45+5 = 23 ft); full 50 ft at the window end
FZ_PAIR_PROB = 0.25             # [FZ-2] per-well rate, independent of FAULT_PROB -> ~50% of wells carry some fault (harder-than-real)
FZ_PAIR_SPACING_MIN, FZ_PAIR_SPACING_MAX = 40, 200      #        md-steps between members (inside the 20-80% window)
FZ_PAIR_RECOVER_MIN, FZ_PAIR_RECOVER_MAX = 0.60, 1.00   #        fraction of throw #1 recovered by throw #2 (opposite sign)
FZ_RAMP_SHARP_P = 0.40          # [FZ-3] prob an event stays a single-row jump (current behavior)
FZ_RAMP_W_MIN, FZ_RAMP_W_MAX = 20, 150   #       ramp width draw (md-steps)
FZ_RAMP_MAX_SLOPE = 4.0         #        ft/step envelope for ramp columns (1.5*50/20 = 3.75 by construction)
print('FZ bundle: ' + (f'ARMED  growth(floor={FZ_GROWTH_FLOOR}) | pair(p={FZ_PAIR_PROB}, '
      f'spacing {FZ_PAIR_SPACING_MIN}-{FZ_PAIR_SPACING_MAX}, recover {FZ_PAIR_RECOVER_MIN}-{FZ_PAIR_RECOVER_MAX}) | '
      f'ramp(sharp_p={FZ_RAMP_SHARP_P}, w {FZ_RAMP_W_MIN}-{FZ_RAMP_W_MAX}, max_slope {FZ_RAMP_MAX_SLOPE})'
      if FZ_ENABLE else 'off (bit-identical v4 generator)'))
_pack_tag = 'expa_' if EVENT_PACK == 'v3_expA' else ''
_fz_tag   = 'fz_'   if FZ_ENABLE else ''


# === Teacher / KD wiring: REMOVED (was the KD fork; stripped after §31.54) =============
# The fold-4 KD gain was the OOF estimator edge, not knowledge transfer — every leak-free
# wiring killed it (nested 5.55 / shuffle 5.66 / LBO 5.63 vs baseline 5.605). This notebook
# is the clean 5-channel baseline: IN_CH = IN_CH_BASE = 5, no teachers, no KD term.
_T_DRIVE = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore'
N_TEACH = 0

# Image dimensions
# H = 224; W = 448; BIN_SIZE = 256/224; K_MAX = 12
H = 256; W = 512; BIN_SIZE = 256/256; K_MAX = 12    # Default
#H = 288; W = 576; BIN_SIZE = 256/288; K_MAX = 12

EXPERIMENT_NAME = f'5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt_H{H}_W{W}_bin{BIN_SIZE}'

LOCAL_ARTIFACT_DIR = f'/content/{EXPERIMENT_NAME}'
os.makedirs(LOCAL_ARTIFACT_DIR, exist_ok=True)

# Backward-compat aliases
OUT_DIR = LOCAL_ARTIFACT_DIR
CKPT_DIR = LOCAL_ARTIFACT_DIR
# [REUSE_PT ckpt fallback] If the in-memory cache AND pretrain_reuse_f{fold}.pt are both absent,
# reuse the fold donor's saved per-epoch pretrain checkpoint (ep PRETRAIN_EPOCHS) so a re-run (e.g.
# the frozen-upsampler finetune) SKIPS pretrain instead of re-pretraining. Loud print on load.
# Set False to force fresh pretrain; delete pretrain_*_ep*.pt to invalidate a stale reuse. Mismatched
# architectures fail loudly at load_state_dict (key/shape check), so cross-mode reuse can't go silent.
REUSE_PRETRAIN_EPOCH_CKPT = True
PRETRAIN_CKPT_SEARCH_DIRS = [CKPT_DIR]   # also searched for pretrain_f{fold}_s{donor}_..._ep{N}.pt; append a mounted dataset dir if prior checkpoints live elsewhere

# === EXTERNAL PRETRAINED DIR — reuse a PRIOR run's per-(fold,seed) pretrain checkpoints ===

PRETRAINED_DIR = f'/content/drive/MyDrive/Kaggle/ROGII-Wellbore/{EXPERIMENT_NAME}'


if IN_COLAB:
    DRIVE_ARTIFACT_DIR = f'/content/drive/MyDrive/Kaggle/ROGII-Wellbore/{EXPERIMENT_NAME}'
    os.makedirs(DRIVE_ARTIFACT_DIR, exist_ok=True)
    DRIVE_OUT = DRIVE_ARTIFACT_DIR
    DRIVE_CKPT = DRIVE_ARTIFACT_DIR

print(f'Local artifact dir: {LOCAL_ARTIFACT_DIR}')
if IN_COLAB:
    print(f'Drive artifact dir: {DRIVE_ARTIFACT_DIR}')


In [ ]:
# === HF AUTH — MUST run before timm/smp import (cell 16 downloads the 836MB XL encoder) ===
# Unauthenticated HF pulls get rate-limited on Colab's shared egress IP -> hangs / truncated blobs.
# Preference: explicit token -> Colab secret -> unauthenticated (still runs; never crashes).
import os as _os_hf
HF_TOKEN = ''   # <<<<<< PASTE YOUR hf_... TOKEN HERE. Leave '' to try the Colab secret instead.
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        HF_TOKEN = ''
if HF_TOKEN:
    _os_hf.environ['HF_TOKEN'] = HF_TOKEN
    _os_hf.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print('HF auth: token set')
else:
    print('HF auth: NONE (unauthenticated - may rate-limit; paste a token above if downloads hang)')

import subprocess, sys, copy
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation_models_pytorch'])
    import segmentation_models_pytorch as smp
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm
print(f'smp {smp.__version__}, timm {timm.__version__}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


# === REUSE_PT toggle =========================================================
# False -> bit-for-bit identical to the original notebook: every (fold, seed) job runs
#          its own synth pretrain, then finetune.
# True  -> pretrain ONCE per fold (on the first seed that runs = SEEDS[0] on a fresh run)
#          and REUSE those post-pretrain weights as the finetune start for every other seed
#          in that fold. The synth pretrain pool is keyed to the GLOBAL seed (identical
#          across seeds), so this drops only pretrain-path + decoder-init diversity; finetune
#          keeps full per-seed diversity (aug RNG ft_base_seed=seed+1e6, per-seed loader
#          generator, per-seed drop_path). Reuse artifacts are tagged '_reusePT' so they never
#          collide with non-reuse runs that share EXPERIMENT_NAME. Saves ~(n_seeds-1)/n_seeds
#          of the per-fold pretrain compute.
REUSE_PT = False
# =============================================================================

# ALL_FOLDS is set in cell 6 (after fold splits computed); we override the FT24 probe restriction there

# === Single deterministic recipe (ARMS removed: training is fully deterministic) ===
IN_CH_BASE   = 5            # baseline 4ch + dZ-per-column channel
IN_CH        = IN_CH_BASE   # 5-channel baseline (KD stripped)
USE_PRETRAIN = True         # synth pretrain -> real finetune (locked recipe)

CV_TO_BEAT          = 5.5
CV_PASS_MARGIN      = 0.1
CV_FOLD_REGRESS_TOL = 0.50   # per-fold regression cap (METHODOLOGY §1)
# === dZ channel synth-generation parameters (see cell 8) ===
DZ_SYNTH_DIP_FRAC   = 0.10   # smooth formation-dip term scale (frac of well's TVT range)
DZ_SYNTH_DIP_AUTOCORR_FRAC = 0.20   # autocorr length = this * n_md
DZ_SYNTH_NOISE_STD  = 0.5    # small drilling deviation per row (ft); decouples synth Z<->TVT slightly

# Phase budgets (FT24 baseline — locked)
PRETRAIN_EPOCHS = 5
FINETUNE_EPOCHS = 35

# === v3 synth config — LOCKED ===
V3_WOBBLE_TOTAL_PCT = 0.15
V3_WOBBLE_SPLIT = (0.50, 0.30, 0.20)
V3_WAVELENGTH_DIVISORS = (1.0, 5.0, 20.0)

# === Corruption presets (THE single change vs the locked a_v3 baseline) ===
_CORRUPTION_PRESETS = {
    'baseline':  dict(sigma_drift=8.0, sigma_tvt_offset=3.5, sigma_meas=1.5, sigma_calib=0.06),  # match_corr ~0.27
    'reduced_B': dict(sigma_drift=3.0, sigma_tvt_offset=1.0, sigma_meas=1.0, sigma_calib=0.06),  # match_corr ~0.69 ≈ real 0.64
    'reduced_C': dict(sigma_drift=1.5, sigma_tvt_offset=0.5, sigma_meas=0.7, sigma_calib=0.06),  # match_corr ~0.88  (cleaner than real)
    'clean': dict(sigma_drift=0.5, sigma_tvt_offset=0.2, sigma_meas=0.5, sigma_calib=0.06),  # match_corr ~0.976
}

_cp = _CORRUPTION_PRESETS[CORRUPTION_MODE]
SIGMA_DRIFT      = _cp['sigma_drift']
SIGMA_TVT_OFFSET = _cp['sigma_tvt_offset']
SIGMA_MEAS       = _cp['sigma_meas']
SIGMA_CALIB      = _cp['sigma_calib']
print(f'Corruption arm: {CORRUPTION_MODE} -> drift={SIGMA_DRIFT}, tvt_offset={SIGMA_TVT_OFFSET}, '
      f'meas={SIGMA_MEAS}, calib={SIGMA_CALIB}  (full 5-fold grid)')

# === EMA config — LOCKED ===
EMA_DECAY = 0.997
WARMUP_EPOCHS = 0
USE_EMA = True

# === Integral-regression loss (soft-argmax / DSNT-style) — finetune-only ===
# Adds Huber(decoded expected_row - true_row) to hard_ce, directly optimizing the
# scored quantity (RMSE of the softmax-expected row). 0.0 = bit-identical baseline.
# CE stays the load-bearing shape term; this is a metric-alignment nudge on top.
# Magnitudes: L_CE ~3; the Huber term ~10-40, so weight ~0.05 makes them comparable.
# DELTA = Huber transition in ft (BIN_SIZE=1): quadratic (RMSE-like) below it, linear above
# to keep the toe outliers from dominating. Large delta -> MSE; small -> MAE.
# WEIGHT forms:
#   scalar w        -> STATIC weight every finetune epoch (current behavior; 0.0 = bit-identical baseline)
#   [w0, w1]        -> LINEAR RAMP over finetune epochs [START_EXP_RAMP, END_EXP_RAMP] (1-based,
#                      inclusive): w0 before START, linear w0->w1 across the window, w1 after END.
#                      [0.0, w1] makes epochs 1..START-1 bit-identical to baseline (log-comparable).
# RAMP ANCHOR: absolute PLANNED finetune-epoch indices (not fraction-of-actual-run) — every
# (fold,seed) sees the identical loss-vs-epoch schedule; early-stopped seeds see a truncated ramp,
# and phase_info records exp_w_at_best / exp_w_at_stop per seed for post-hoc stratification.
# WINDOW GUIDANCE (measured, this pipeline): EMA decay 0.997 @ ~78 steps/ep -> ~4.3-ep lag before a
# behavior change reaches the EVALUATED (shadow) weights; cosine lr ~50% @ ep15, ~10% @ ep25; seeds'
# bests cluster ~ep 11-18. END should sit >=5 ep before typical stop or the full-weight regime never
# reaches the selected checkpoint. Default window 10->20 targets the CE/TVT divergence onset (~ep15).
EXP_LOSS_WEIGHT = 0.0  # inert. Static probe: 0.05. First ramp arm: [0.0, 0.05].
EXP_LOSS_DELTA  = 8.0     # ft
START_EXP_RAMP  = 10      # 1-based finetune epoch where the ramp leaves w0 (list form only)
END_EXP_RAMP    = 20      # 1-based finetune epoch where the ramp reaches w1 (list form only)

# --- normalize the two forms once; everything downstream consumes (_EXP_W0, _EXP_W1, _EXP_RAMP_ON)
if isinstance(EXP_LOSS_WEIGHT, (list, tuple)):
    assert len(EXP_LOSS_WEIGHT) == 2, 'EXP_LOSS_WEIGHT list form is [w_start, w_end]'
    _EXP_W0, _EXP_W1 = float(EXP_LOSS_WEIGHT[0]), float(EXP_LOSS_WEIGHT[1])
    _EXP_RAMP_ON = (_EXP_W0 != _EXP_W1)
    assert 1 <= START_EXP_RAMP <= END_EXP_RAMP, 'need 1 <= START_EXP_RAMP <= END_EXP_RAMP'
else:
    _EXP_W0 = _EXP_W1 = float(EXP_LOSS_WEIGHT)
    _EXP_RAMP_ON = False
_EXP_ARMED = (_EXP_W0 > 0) or (_EXP_W1 > 0)

def exp_loss_weight_at(ft_epoch1):
    """Weight for 1-based finetune epoch. Pure function of the PLANNED schedule (never of val)."""
    if not _EXP_RAMP_ON:
        return _EXP_W1
    if ft_epoch1 <= START_EXP_RAMP: return _EXP_W0
    if ft_epoch1 >= END_EXP_RAMP:   return _EXP_W1
    frac = (ft_epoch1 - START_EXP_RAMP) / float(END_EXP_RAMP - START_EXP_RAMP)
    return _EXP_W0 + frac * (_EXP_W1 - _EXP_W0)

if _EXP_RAMP_ON:
    print(f'Integral-regression loss: RAMP {_EXP_W0}->{_EXP_W1} over ft ep {START_EXP_RAMP}->{END_EXP_RAMP}, '
          f'delta={EXP_LOSS_DELTA} ft (finetune-only)')
else:
    print(f'Integral-regression loss: weight={_EXP_W1} (static), delta={EXP_LOSS_DELTA} ft (finetune-only)')

# === TRAIN-TIME DETACHED RANDOM-K UNROLLING (finetune-only) ===================================
# Close the stage-2 train/inference gap: train stage-2 to refine an ALREADY-refined belief, the job
# it currently does OFF-distribution at inference (k4). On each finetune step, with prob UNROLL_P run
# stage-2 k times (k ~ UNROLL_K_CHOICES, per-step), DETACHING the belief between rounds so gradients
# flow only through the FINAL pass; supervise the final pass only. With prob 1-UNROLL_P the normal
# k=1 pass runs (anchors the base task). T=1 (no inference-temperature inherited — that stays the
# post-hoc diagnostic). Eval/OOF are NEVER unrolled -> the saved checkpoint is the deployed k1 model.
#
# IDENTITY: UNROLL_ENABLE=False (default) or UNROLL_P=0.0 -> train forward == baseline (asserted in
# the cell-20 smoke). Pretrain is never unrolled regardless (finetune-only, like the warps).
UNROLL_ENABLE     = True          # master switch; False = bit-identical baseline
UNROLL_P          = 0.5          # per-step probability of running the unrolled (k>=2) path
UNROLL_K_CHOICES  = (2, 3, 4)      # k sampled per unrolled step (matches the pool's inference depth)
UNROLL_K_WEIGHTS  = (1.0, 1.0, 1.0)  # sampling weights over UNROLL_K_CHOICES (uniform default)
UNROLL_DEEP_SUP_ROUNDS = True    # False = supervise final pass only (bit-identical to ft39l);
                                  #        True = ALSO supervise intermediate unroll rounds 2..k-1 (NEW, untested arm).
UNROLL_ROUND_SUP_WEIGHT = 0.5     # lambda on the MEAN of the intermediate-round CEs (only consumed when the flag is True).
                                   # True  = also CE the intermediate rounds' stage-2 outputs (v2 idea)
UNROLL_RNG_SEED   = 20260707       # per-step k/gate draws (distinct stream; determinism-safe)
assert abs(sum(UNROLL_K_WEIGHTS) - sum(UNROLL_K_WEIGHTS)) == 0 and len(UNROLL_K_WEIGHTS) == len(UNROLL_K_CHOICES)
print(f'Unroll (finetune-only): enable={UNROLL_ENABLE}, p={UNROLL_P}, k~{UNROLL_K_CHOICES} '
      f'w={UNROLL_K_WEIGHTS}, deep_sup_rounds={UNROLL_DEEP_SUP_ROUNDS}' + (f' round_sup_w={UNROLL_ROUND_SUP_WEIGHT}' if UNROLL_DEEP_SUP_ROUNDS else '')
      + ('' if UNROLL_ENABLE else '   [OFF — bit-identical baseline]'))

# === TRAIN-TIME TEMPERATURE at the two belief-transfer sites (both default 1.0 = bit-identical) ===
# REFEED_TRAIN_T : soften the DETACHED stage-2 belief re-fed on unrolled rounds (finetune-only; only
#   active when UNROLL is on). The scored final pass stays untempered. Trains stage-2 to iterate on
#   less-committed beliefs -> the training-side lever on the anti-iteration (>12ft) tail.
# HANDOFF_TRAIN_T: soften the stage-1->stage-2 handoff. Lives in forward(), so it applies at TRAIN
#   AND EVAL (the deployed model consumes the same tempered handoff it trained on). k1-seed effect.
# Grid to sweep (fixed-T, single-axis; sweep ONE site at a time): {1.0, 1.15, 1.3}. Sharpen (<1) not
# expected to help. Temperature-as-augmentation (sample T per step) is a LATER v2 on whichever site wins.
REFEED_TRAIN_T  = 1.0     # >1 softens the re-fed belief on unrolled rounds; 1.0 = identity
HANDOFF_TRAIN_T = 1.5     # >1 softens the stage-1->stage-2 handoff (train+eval); 1.0 = identity
print(f'Train-time temperature: refeed_T={REFEED_TRAIN_T} (unroll refeed), '
      f'handoff_T={HANDOFF_TRAIN_T} (stage1->stage2, train+eval)'
      + ('' if (REFEED_TRAIN_T == 1.0 and HANDOFF_TRAIN_T == 1.0) else '   [ARMED]'))

# === MULTI-K VALIDATION LOGGING (observational; selection stays on k=1) ========================
# Log val TVT-RMSE and val-CE at k=1 AND each depth in DEPLOY_K_LOG every finetune epoch, so the
# k=1-best epoch vs deploy-k-best epoch divergence is visible. Selection/early-stop are UNCHANGED
# (still track k=1). Eval iteration MATCHES DEPLOYMENT: refeed applies REFEED_TRAIN_T and the handoff
# applies HANDOFF_TRAIN_T (both are what this checkpoint deploys with). [] or None -> byte-identical
# to the k=1-only log.
DEPLOY_K_LOG = [2, 3]     # e.g. [3, 4] or [2, 3, 4]; k=1 always logged as the selection metric
print(f'Multi-k val logging: k=1 (selection) + deploy depths {list(DEPLOY_K_LOG) if DEPLOY_K_LOG else "(none)"} '
      f'(observational; refeed_T={REFEED_TRAIN_T}, handoff_T={HANDOFF_TRAIN_T} applied — matches deploy)')

# === CHECKPOINT_K — select best-state + early-stop on a DEPLOY depth (not k=1) ==================
# =1 (default): bit-identical to k=1 selection. >1: select/stop on the deployment-matched RMSE at
# CHECKPOINT_K (must be in DEPLOY_K_LOG so it's computed). Use when deploying with iteration so the
# saved epoch is the one best at the depth you actually run. k=1 will look worse in these runs by
# design; compare deploy-k vs deploy-k.
CHECKPOINT_K = 3
assert CHECKPOINT_K == 1 or CHECKPOINT_K in (DEPLOY_K_LOG or []), \
    f'CHECKPOINT_K={CHECKPOINT_K} must be 1 or in DEPLOY_K_LOG={DEPLOY_K_LOG} (else it is not computed)'
print(f'Checkpoint/early-stop selection metric: k={CHECKPOINT_K}'
      + ('  (k=1 headline — identity)' if CHECKPOINT_K == 1 else
         f'  (DEPLOY depth; refeed_T={REFEED_TRAIN_T}, handoff_T={HANDOFF_TRAIN_T} applied)'))

# === Warm restart probe ===
USE_WARM_RESTARTS    = False
WARM_RESTART_T0      = 10    # first cycle length (epochs)
WARM_RESTART_T_MULT  = 1     # 1 = equal cycles, 2 = doubling cycles
# Patience auto-extends to T_0+5 = 15 epochs in cell 18 to avoid early-stopping
# mid-cycle when val regresses momentarily at restart points.

print(f'Warm restarts: {USE_WARM_RESTARTS}  (T_0={WARM_RESTART_T0}, T_mult={WARM_RESTART_T_MULT})')

np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}' + (f'  GPU: {torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'Seeds: {SEEDS} (single recipe, no arms)')
print(f'EMA: decay={EMA_DECAY}, warmup={WARMUP_EPOCHS}')
print(f'Pretrain {PRETRAIN_EPOCHS} ep + Finetune {FINETUNE_EPOCHS} ep at lr=1e-3 cosine (each phase)')
print(f'Synth pretrain pool: {N_SYNTH_PRETRAIN}/fold')
print(f'CV to beat (b7/reduced_C + MD-warp + BedWarp locked baseline): {CV_TO_BEAT:.3f} ft')
print(f'Encoder: {ENCODER}, in_ch: {IN_CH} (4 baseline + dZ-per-column channel)')
print()
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
torch.use_deterministic_algorithms(False, warn_only=True)  # warn_only because some ops have no det impl

# TF32: let FP32 matmuls/convs use tensor cores at ~FP32 quality (near-free speedup, every precision)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# === Worker / RNG plumbing (determinism patch v1) ===
import random as _random
import os as _os
_os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')  # required for det matmul

def seed_everything(seed):
    """Call at start of each training run with the per-run seed."""
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_loader_generator(seed):
    """torch.Generator that controls DataLoader batch shuffle deterministically."""
    g = torch.Generator()
    g.manual_seed(int(seed) & 0xFFFFFFFF)
    return g

def seed_worker(worker_id):
    """worker_init_fn for DataLoader — reseeds numpy + python random per worker."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    _random.seed(worker_seed)

print('Determinism patch v1 loaded: seed_everything, seed_worker, make_loader_generator')

# === FT39j: synth fault-overlay parameters (locked) ===
FAULT_PROB           = 0.33    # per-well probability of having any fault events
FAULT_MAGNITUDE_MIN  = 5.0     # ft — minimum fault throw
FAULT_MAGNITUDE_MAX  = 50.0    # ft — maximum fault throw
# N_FAULTS_PER_WELL: 1 with P=0.7, 2 with P=0.3 (hardcoded in synth gen)
# FAULT_POSITION: random in 20-80% of post-PS section (hardcoded)
# FAULT_SHARPNESS: single-row jump (hardcoded)
print(f'Fault overlay: prob={FAULT_PROB}, magnitude [{FAULT_MAGNITUDE_MIN:.0f}, {FAULT_MAGNITUDE_MAX:.0f}] ft')

# === FT39k: unconformity overlay parameters (locked) ===
UNCONF_PROB                  = 0.33   # per-well probability of having an unconformity (independent of fault)
UNCONF_MAX_CUMULATIVE_SHIFT  = 30.0   # ft — bounded cumulative TVT shift over post-unconformity section
# UNCONF_POSITION: random in 20-70% of post-PS section (hardcoded; leaves 30%+ of post-PS for the post-unconf slope to develop)
print(f'Unconformity overlay: prob={UNCONF_PROB}, max cumulative shift={UNCONF_MAX_CUMULATIVE_SHIFT} ft')

# === FT39l: variable bed thickness overlay parameters (locked) ===
VAR_BED_PROB              = 0.50    # per-well probability of having variable bed thickness modulation
VAR_BED_AMP_STD           = 0.40    # std of modulation factor around 1.0
VAR_BED_AUTOCORR_LENGTH   = 200     # autocorr length of modulation (rows)
VAR_BED_AMP_MIN           = 0.30    # clip lower bound (thickest beds reduce wobble to 30%)
VAR_BED_AMP_MAX           = 2.00    # clip upper bound (thinnest beds amplify wobble 2x)
print(f'Variable bed thickness overlay: prob={VAR_BED_PROB}, amp_std={VAR_BED_AMP_STD}, '
      f'autocorr={VAR_BED_AUTOCORR_LENGTH} rows, clipped to [{VAR_BED_AMP_MIN}, {VAR_BED_AMP_MAX}]')

# === MD-axis elastic warp (finetune-only) — apparent-dip-rate augmentation ===
MD_WARP_STRENGTH = 0.36 * W / 512    # 0.15 for baseline / 0.22 for stronger / 0.3 for strongest
MD_WARP_AUTOCORR  = W // 4       # 128 default
MD_WARP_P         = 0.5
print(f'MD elastic warp (finetune-only): p={MD_WARP_P}, strength={MD_WARP_STRENGTH}, autocorr={MD_WARP_AUTOCORR}'
      + ('  [OFF — ARM C is jitter-only]' if MD_WARP_STRENGTH == 0.0 else ''))

# === Per-column vertical shear (finetune-only) — "elastic anchor jitter" ===
SHEAR_STD_ROWS = 3.0   # ~3-row typical wobble; 3-sigma = the 9-row clip (clip is a rare rail, not the operating pt)
SHEAR_P        = 0.0      # mirrors MD_WARP_P / BedWarp gate (the proven warp-family p)
SHEAR_AUTOCORR = 128      # field smoothness along W (mirrors MD_WARP_AUTOCORR) -> ~4 smooth lobes over W=512
SHEAR_MAX_ROWS = 9        # hard clip on |d(c)| (inside the anchor-jitter +/-12 precedent)
print(f'Vertical shear (finetune-only): p={SHEAR_P}, std={SHEAR_STD_ROWS} rows, '
      f'autocorr={SHEAR_AUTOCORR}, |d|<= {SHEAR_MAX_ROWS}')


# === BedWarp (finetune-only) — smooth monotone TVT-axis deformation: bed thickness jitter ===
BEDWARP_STRENGTH = 0.36 * H / 256     # 0.12 for baseline / 0.18 for stronger / 0.24 for strongest
BEDWARP_AUTOCORR = H // 4    # default 64
BEDWARP_P        = 0.5     # fraction of finetune samples deformed (mirrors MD_WARP_P)
print(f'BedWarp (finetune-only): p={BEDWARP_P}, strength={BEDWARP_STRENGTH}, autocorr={BEDWARP_AUTOCORR} rows')


# === PhaseJitter (render-stage, finetune real wells only) — ARMED-BUT-INERT ===
PHASE_JITTER_P = 0.0   # <-- set to 0.5 to ARM the PhaseJitter test
print(f'PhaseJitter (finetune, real wells): p={PHASE_JITTER_P}'
      + ('  [INERT — bit-exact lock recipe]' if PHASE_JITTER_P == 0.0 else '  [ARMED]'))


# === ToeCrop (render-stage, finetune real wells only) — WIRED-BUT-INERT ===
TOECROP_P        = 0.0   # <-- set to 0.5 to ARM the ToeCrop test
TOECROP_MAX_FRAC = 0.25   # crop upper bound: c ~ U(0, MAX_FRAC) of post-PS rows
print(f'ToeCrop (finetune, real wells): p={TOECROP_P}, max_frac={TOECROP_MAX_FRAC}'
      + ('  [INERT]' if TOECROP_P == 0.0 else '  [ARMED]'))


# === GR_AUG (render-stage, finetune real wells only) — WIRED-BUT-INERT ===
GR_AUG_P        = 0.0
GR_AUG_GAIN_S   = 0.25   # gain ~ U(1-s, 1+s); ~±4.3 API stretch at median std 17.3 (was 0.15 = negligible ±2.6)
GR_AUG_OFFSET_S = 8.0    # offset ~ U(-s, +s) API; matches observed HW-TW median drift 8.0 (was 10.0, slightly hot)

print(f'GR_AUG (finetune, real wells): p={GR_AUG_P}, gain_s={GR_AUG_GAIN_S}, offset_s={GR_AUG_OFFSET_S}'
      + ('  [INERT]' if GR_AUG_P == 0.0 else '  [ARMED]'))


## 1. Data + 5-fold splits

In [ ]:
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
HW_SUFFIX = '__horizontal_well.csv'
TW_SUFFIX = '__typewell.csv'
def well_hw_path(wid): return os.path.join(TRAIN_DIR, wid + HW_SUFFIX)
def well_tw_path(wid): return os.path.join(TRAIN_DIR, wid + TW_SUFFIX)
def list_wells(wells_dir):
    files = set(os.listdir(wells_dir))
    return [f[:-len(HW_SUFFIX)] for f in sorted(files)
            if f.endswith(HW_SUFFIX) and (f[:-len(HW_SUFFIX)] + TW_SUFFIX) in files]
well_ids_all = list_wells(TRAIN_DIR)
print(f'{len(well_ids_all)} wells.')

rng = np.random.RandomState(SEED)
well_pool = list(well_ids_all); rng.shuffle(well_pool)
well_meta = []
for wid in tqdm(well_pool, desc='scanning'):
    hw = pd.read_csv(well_hw_path(wid), usecols=['MD','TVT','TVT_input'])
    well_meta.append({'well_id': wid, 'n_rows': len(hw),
                      'tvt_range': hw['TVT'].max() - hw['TVT'].min()})
meta_df = pd.DataFrame(well_meta)
N_STRATA = N_FOLDS = 5
meta_df['stratum'] = pd.qcut(meta_df['tvt_range'], q=N_STRATA, labels=list(range(N_STRATA)))
fold_assignment = {}
for s in range(N_STRATA):
    pool = meta_df[meta_df['stratum'] == s]['well_id'].tolist()
    rng2 = np.random.RandomState(SEED + s); rng2.shuffle(pool)
    chunk = len(pool) // N_FOLDS
    for f in range(N_FOLDS):
        lo = f * chunk; hi = (f + 1) * chunk if f < N_FOLDS - 1 else len(pool)
        for wid in pool[lo:hi]: fold_assignment[wid] = f
fold_splits = {}
for f in range(N_FOLDS):
    fold_splits[f] = {
        'val':   [w for w, fa in fold_assignment.items() if fa == f],
        'train': [w for w, fa in fold_assignment.items() if fa != f],
    }
    print(f'Fold {f}: {len(fold_splits[f]["train"])} train, {len(fold_splits[f]["val"])} val')

# === FT26: full 5-fold sweep (no probe restriction) ===
ALL_FOLDS = [0]   # FZ design read: configuration-diversity arm -> stable fold. Control =
#   schedule-matched roundsup fold-0 (ens 5.153); verify the control by its summary-JSON
#   'experiment' field, NEVER by filename (the ft39l collision lesson).
print(f'\nFT26 using all {len(ALL_FOLDS)} folds: {ALL_FOLDS}')

## 2. Synth pool generation (Method A v3, per-fold leakage discipline)

In [ ]:
def make_correlated_noise(n, autocorr_length, target_std, rng):
    phi = np.exp(-1.0 / autocorr_length)
    white = rng.normal(0, 1, n)
    smoothed = np.zeros(n)
    smoothed[0] = white[0]
    for i in range(1, n):
        smoothed[i] = phi * smoothed[i-1] + np.sqrt(1 - phi**2) * white[i]
    return smoothed * target_std

def generate_synthetic_well_method_a_v3(
    real_tw_df, hint_real_dist, seed=None,
    sigma_drift=8.0, sigma_calib=0.06,
    sigma_tvt_offset=3.5, sigma_meas=1.5,
    drift_autocorr=40, tvt_offset_autocorr=200,
    wobble_total_pct=V3_WOBBLE_TOTAL_PCT,
    wobble_split=V3_WOBBLE_SPLIT,
    wavelength_divisors=V3_WAVELENGTH_DIVISORS,
    fault_prob=0.0,                       # FT39j: prob of any fault events
    fault_magnitude_min=5.0,              # FT39j: min throw (ft)
    fault_magnitude_max=50.0,             # FT39j: max throw (ft)
    unconf_prob=0.0,                      # FT39k: prob of unconformity (slope change)
    unconf_max_cumulative_shift=30.0,     # FT39k: bounded cumulative shift (ft)
    var_bed_prob=0.0,                     # FT39l: prob of variable bed thickness
    var_bed_amp_std=0.4,                  # FT39l: std of amplitude modulation around 1.0
    var_bed_autocorr_length=200,          # FT39l: autocorr length of modulation (rows)
    var_bed_amp_min=0.3,                  # FT39l: clip lower bound on modulation
    var_bed_amp_max=2.0,                  # FT39l: clip upper bound on modulation
):
    rng = np.random.RandomState(seed)
    tw_tvt = real_tw_df['TVT'].values
    tw_gr_arr = real_tw_df['GR'].values
    ok = ~np.isnan(tw_tvt) & ~np.isnan(tw_gr_arr)
    tw_tvt = tw_tvt[ok]; tw_gr_arr = tw_gr_arr[ok]
    if len(tw_tvt) < 100: return None
    sort = np.argsort(tw_tvt)
    tw_tvt = tw_tvt[sort]; tw_gr_arr = tw_gr_arr[sort]
    n_md = int(rng.choice(hint_real_dist['n_md']))
    ps_md_pct = float(rng.choice(hint_real_dist['ps_md_pct']))
    tvt_range = float(rng.choice(hint_real_dist['tvt_range']))
    tw_lo, tw_hi = tw_tvt[0], tw_tvt[-1]
    margin = tvt_range * 1.5
    if tw_hi - tw_lo < 2*margin:
        heel_tvt = (tw_lo + tw_hi) / 2
    else:
        heel_tvt = float(rng.uniform(tw_lo + margin, tw_hi - margin))
    md_arr = np.arange(n_md).astype(np.float64)
    lateral_md = n_md * (1 - ps_md_pct/100)
    if lateral_md < 100: return None
    dip_sign = rng.choice([-1, 1])
    dip_rate = dip_sign * tvt_range / max(lateral_md, 1)
    wobble_total_amp = tvt_range * wobble_total_pct * rng.uniform(0.7, 1.3)
    amp_lo = wobble_total_amp * wobble_split[0] * rng.uniform(0.5, 1.5)
    amp_mid = wobble_total_amp * wobble_split[1] * rng.uniform(0.5, 1.5)
    amp_hi = wobble_total_amp * wobble_split[2] * rng.uniform(0.5, 1.5)
    omega_lo = 2 * np.pi / (lateral_md * wavelength_divisors[0] * rng.uniform(0.7, 1.3))
    omega_mid = 2 * np.pi / (lateral_md / wavelength_divisors[1] * rng.uniform(0.7, 1.3))
    omega_hi = 2 * np.pi / (lateral_md / wavelength_divisors[2] * rng.uniform(0.7, 1.3))
    phi_lo = rng.uniform(0, 2*np.pi)
    phi_mid = rng.uniform(0, 2*np.pi)
    phi_hi = rng.uniform(0, 2*np.pi)
    ps_md = int(ps_md_pct/100 * n_md)
    ps_md = max(ps_md, 100); ps_md = min(ps_md, n_md - 100)
    pre_md = md_arr[:ps_md]
    pre_start_tvt = max(tw_lo, heel_tvt - tvt_range * 8)
    pre_t = pre_md / max(ps_md - 1, 1)
    pre_tvt = pre_start_tvt + (heel_tvt - pre_start_tvt) * (0.5 - 0.5 * np.cos(pre_t * np.pi))
    post_md = md_arr[ps_md:] - md_arr[ps_md]

    # === FT39l: variable bed thickness modulation ===
    # Apply smooth modulation to wobble amplitudes across MD. Different mechanism from
    # faults/unconformities (discrete events): continuous amplitude variation. Some MD
    # regions have less wobble (thicker beds), others have more (thinner beds).
    var_bed_info = None
    if var_bed_prob > 0 and rng.uniform() < var_bed_prob:
        var_bed_mod_full = make_correlated_noise(
            n_md, autocorr_length=var_bed_autocorr_length,
            target_std=var_bed_amp_std, rng=rng
        ) + 1.0  # centered at 1.0
        var_bed_mod_full = np.clip(var_bed_mod_full, var_bed_amp_min, var_bed_amp_max).astype(np.float64)
        var_bed_info = {
            'mod_std_observed': float(np.std(var_bed_mod_full)),
            'mod_min': float(var_bed_mod_full.min()),
            'mod_max': float(var_bed_mod_full.max()),
        }
    else:
        var_bed_mod_full = np.ones(n_md, dtype=np.float64)
    var_bed_mod_post = var_bed_mod_full[ps_md:ps_md + len(post_md)]

    post_tvt = (heel_tvt + dip_rate * post_md
                + var_bed_mod_post * amp_lo  * np.sin(omega_lo  * post_md + phi_lo)
                + var_bed_mod_post * amp_mid * np.sin(omega_mid * post_md + phi_mid)
                + var_bed_mod_post * amp_hi  * np.sin(omega_hi  * post_md + phi_hi))
    tvt_full = np.concatenate([pre_tvt, post_tvt])

    # === FT39j fault overlay + [FZ] FAULT-ZONE ARCHITECTURE bundle ===============================
    # Base draws (gate, n_faults, positions, magnitudes) come from the well's own rng in the
    # ORIGINAL order -> byte-identical base-stream consumption whether FZ is armed or not.
    # FZ post-processes the drawn events and adds its own from a SEPARATE stream (FZ_HASH):
    #   [FZ-1] growth throw    : |mag| draw U(min,max) remapped to U(min, min+(max-min)*s(pos)),
    #                            s = FLOOR..1 across the 20-80% window (same base draw, no extra
    #                            base consumption; skipped exactly at s==1.0 for the limit identity).
    #   [FZ-2] antithetic pair : one graben/horst crossing (opposite signs, close-spaced, section
    #                            recovers by RECOVER_FRAC) — excursion-guarded vs
    #                            max(base-with-events, EXPA_EXC_CAP_FT); redraw<=6 then drop+count.
    #   [FZ-3] drag ramps      : each fault-class event draws sharp-vs-ramped; ramped throw accrues
    #                            as a smoothstep over w columns with an EXACT plateau tail (the
    #                            fault<->monocline ambiguity). Ramp columns are JUMP-CLASS for the
    #                            envelope (carve-out; max discrete slope 1.5*|m|/w <= FZ_RAMP_MAX_SLOPE).
    # FZ_ENABLE=False executes the original application lines verbatim (bit-identical v4 generator).
    # Z uses the pre-fault snapshot either way -> the wellbore stays smooth across ALL events.
    tvt_full_for_z = tvt_full.copy()   # snapshot of pre-fault TVT for Z derivation
    fault_info = []
    _fz_on = bool(globals().get('FZ_ENABLE', False))
    _fz_info = None
    _plo_f = ps_md + int(0.20 * (n_md - ps_md))
    _phi_f = ps_md + int(0.80 * (n_md - ps_md))
    _fz_events = []                     # [pos, mag, kind] — collected (not applied) when FZ is on
    if fault_prob > 0 and rng.uniform() < fault_prob:
        n_faults = 1 if rng.uniform() < 0.7 else 2
        post_md_lo = _plo_f
        post_md_hi = _phi_f
        if post_md_hi - post_md_lo >= 50:
            n_faults_eff = min(n_faults, post_md_hi - post_md_lo)
            positions = np.sort(rng.choice(np.arange(post_md_lo, post_md_hi),
                                            size=n_faults_eff, replace=False))
            magnitudes = rng.uniform(fault_magnitude_min, fault_magnitude_max,
                                      size=n_faults_eff) * rng.choice([-1, 1], size=n_faults_eff)
            if not _fz_on:
                for pos, mag in zip(positions, magnitudes):
                    tvt_full[pos:] += mag
                    fault_info.append({'md_idx': int(pos), 'magnitude': float(mag)})
            else:
                for pos, mag in zip(positions, magnitudes):
                    _fz_events.append([int(pos), float(mag), 'native'])
    if _fz_on:
        _fzrng = np.random.RandomState((int(seed if seed is not None else 0) * FZ_HASH) & 0xFFFFFFFF)
        _fz_info = {'n_native': len(_fz_events), 'pair': False, 'pair_dropped': False,
                    'ramped': 0, 'sharp': 0, 'growth_scales': [], 'ramp_ws': []}
        _win = max(_phi_f - _plo_f, 1)
        # --- [FZ-1] growth throw ---------------------------------------------------------------
        if FZ_GROWTH_ENABLE:
            for _ev in _fz_events:
                _s = FZ_GROWTH_FLOOR + (1.0 - FZ_GROWTH_FLOOR) * float(np.clip((_ev[0] - _plo_f) / _win, 0.0, 1.0))
                if _s != 1.0:   # exact skip at s==1 -> the limit identity holds byte-for-byte
                    _ev[1] = float(np.sign(_ev[1]) * (fault_magnitude_min + (abs(_ev[1]) - fault_magnitude_min) * _s))
                _fz_info['growth_scales'].append(float(_s))
        # --- [FZ-2] antithetic pair (graben/horst) ----------------------------------------------
        if _phi_f - _plo_f >= 50 and _fzrng.uniform() < FZ_PAIR_PROB:
            _dlt = np.zeros(n_md - ps_md, dtype=np.float64)
            for _p, _m, _k in _fz_events:
                _dlt[_p - ps_md:] += _m
            _bpost = tvt_full[ps_md:] + _dlt
            _exc_ref = float(np.nanmax(_bpost) - np.nanmin(_bpost))
            for _try in range(6):
                _sp = int(_fzrng.uniform(FZ_PAIR_SPACING_MIN, FZ_PAIR_SPACING_MAX))
                if _phi_f - _sp <= _plo_f:
                    continue
                _p1 = int(_fzrng.choice(np.arange(_plo_f, _phi_f - _sp)))
                _p2 = _p1 + _sp
                _m1 = float(_fzrng.uniform(fault_magnitude_min, fault_magnitude_max)) * (1.0 if _fzrng.uniform() < 0.5 else -1.0)
                if FZ_GROWTH_ENABLE:
                    _s1 = FZ_GROWTH_FLOOR + (1.0 - FZ_GROWTH_FLOOR) * float(np.clip((_p1 - _plo_f) / _win, 0.0, 1.0))
                    if _s1 != 1.0:
                        _m1 = float(np.sign(_m1) * (fault_magnitude_min + (abs(_m1) - fault_magnitude_min) * _s1))
                _rf = float(_fzrng.uniform(FZ_PAIR_RECOVER_MIN, FZ_PAIR_RECOVER_MAX))
                _m2 = -_m1 * _rf
                _cand = _bpost.copy()
                _cand[_p1 - ps_md:] += _m1
                _cand[_p2 - ps_md:] += _m2
                if float(np.nanmax(_cand) - np.nanmin(_cand)) <= max(_exc_ref, EXPA_EXC_CAP_FT):
                    _fz_events.append([_p1, _m1, 'pair'])
                    _fz_events.append([_p2, _m2, 'pair'])
                    _fz_info['pair'] = True
                    _fz_info['pair_spacing'] = _sp
                    _fz_info['pair_recover'] = _rf
                    break
            else:
                _fz_info['pair_dropped'] = True
        # --- [FZ-3] drag ramps + application (position order; EXACT plateau tails) --------------
        _fz_events.sort(key=lambda e: e[0])
        for _p, _m, _k in _fz_events:
            _w = 0 if _fzrng.uniform() < FZ_RAMP_SHARP_P else int(_fzrng.uniform(FZ_RAMP_W_MIN, FZ_RAMP_W_MAX))
            if _w <= 1 or _p + 2 >= n_md:
                tvt_full[_p:] += _m
                _fz_info['sharp'] += 1
                _w = 0
            else:
                _e = min(_p + _w, n_md)
                _t = (np.arange(_e - _p, dtype=np.float64) + 1.0) / (_e - _p)
                tvt_full[_p:_e] += _m * (3.0 * _t**2 - 2.0 * _t**3)   # smoothstep accrual (monocline-shaped)
                tvt_full[_e:] += _m                                    # exact plateau
                _fz_info['ramped'] += 1
                _fz_info['ramp_ws'].append(int(_e - _p))
            fault_info.append({'md_idx': int(_p), 'magnitude': float(_m), 'kind': _k, 'ramp_w': int(_w),
                               'pos_frac': float(np.clip((_p - _plo_f) / _win, 0.0, 1.0))})

    # === FT39k: optional unconformity overlay (slope change) ===
    # Like faults: TVT shifts but Z (computed from tvt_full_for_z) stays smooth — formation
    # dip changes, wellbore doesn't. Cumulative shift bounded to keep the change visible but
    # not dominant relative to natural wobble.
    unconf_info = None
    if unconf_prob > 0 and rng.uniform() < unconf_prob:
        unconf_lo = ps_md + int(0.20 * (n_md - ps_md))
        unconf_hi = ps_md + int(0.70 * (n_md - ps_md))   # leave 30%+ for post-unconf slope to develop
        if unconf_hi - unconf_lo >= 100:
            unconf_pos = int(rng.choice(np.arange(unconf_lo, unconf_hi)))
            lateral_remaining = n_md - unconf_pos
            # Pick a delta_slope such that cumulative shift over remaining MD is within bound
            max_delta_slope = unconf_max_cumulative_shift / max(lateral_remaining, 1)
            unconf_delta_slope = float(rng.uniform(-max_delta_slope, max_delta_slope))
            # Apply: TVT shifts by delta_slope * (md - unconf_pos) for md >= unconf_pos
            md_relative = (np.arange(n_md) - unconf_pos).astype(np.float64)
            md_relative[md_relative < 0] = 0.0
            tvt_full = tvt_full + unconf_delta_slope * md_relative
            unconf_info = {'md_idx': unconf_pos, 'delta_slope': unconf_delta_slope,
                            'cumulative_shift': float(unconf_delta_slope * lateral_remaining)}

    # === [EVEX-A] Event-expansion pack: A1 dip-breaks + A3-lite repeat (pinch DORMANT, A2-class) ===
    # Geometry/thickness-class events only (GR render untouched). All draws from a SEPARATE stream
    # (EXPA_HASH) -> EVENT_PACK='v3' is bit-identical with ZERO extra draws. Applied after the
    # tvt_full_for_z snapshot -> Z stays wellbore-smooth (identity rule, same as faults/unconf).
    # Guards act on the PACK'S MARGINAL EFFECT (delta vs the pack-free well): added |slope| capped
    # at EXPA_MAX_ABS_DIP (brittle dip axis; base wobble spikes are the base's business), and the
    # well's excursion may not be pushed past max(base excursion, EXPA_EXC_CAP_FT) so the pool's
    # excursion p99 stays pinned inside the real envelope (median/p90 MAY rise — added drift
    # regimes are the point; pre-marked in the PREGATE). Breach -> redraw (6 tries) -> drop, count.
    pack_info = None
    _pinch_map_fn = None
    if globals().get('EVENT_PACK', 'v3') == 'v3_expA':
        prng = np.random.RandomState((int(seed if seed is not None else 0) * EXPA_HASH) & 0xFFFFFFFF)
        _plo = ps_md + int(0.20 * (n_md - ps_md)); _phi = ps_md + int(0.80 * (n_md - ps_md))
        _base_tvt = tvt_full.copy()
        _b_post = _base_tvt[ps_md:]
        _b_exc = float(np.nanmax(_b_post) - np.nanmin(_b_post))
        pack_info = {'events': [], 'dropped': False, 'tries': 0}
        for _try in range(6):
            tvt_full = _base_tvt.copy(); _ev = []; _pinch = None; _rep_span = None
            # --- A1 dip-breaks: strong slope regimes, zero throw (FT39k caps cumulative shift at
            #     30 ft; A1 unlocks the "significant dip change" regime under the delta guards).
            if _phi - _plo >= 50 and prng.uniform() < DIPBREAK_PROB:
                _nb = 2 if prng.uniform() < DIPBREAK_SECOND_P else 1
                for _k in range(_nb):
                    _cpl = bool(fault_info) and prng.uniform() < DIPBREAK_FAULT_COUPLE_P
                    if _cpl:
                        _p = int(prng.choice([f['md_idx'] for f in fault_info]))   # drag/rollover analog
                    else:
                        _p = int(prng.choice(np.arange(_plo, _phi)))
                    _ds = float(prng.uniform(DIPBREAK_DSLOPE_MIN, DIPBREAK_DSLOPE_MAX))
                    _ds *= (1.0 if prng.uniform() < 0.5 else -1.0)
                    _rel = (np.arange(n_md) - _p).astype(np.float64); _rel[_rel < 0] = 0.0
                    tvt_full = tvt_full + _ds * _rel
                    _ev.append({'type': 'dipbreak', 'md_idx': _p, 'delta_slope': _ds, 'coupled': _cpl})
            # --- A3-lite repeated section: throw INTO the already-drilled post-PS band + damped-dip
            #     window -> long-lived, physically grounded alias (thrust/low-angle repeat).
            #     Drop-first component. Landing targeted inside the traversed band (0.15 margins).
            if _phi - _plo >= 50 and prng.uniform() < REPEAT_PROB:
                _p = int(prng.choice(np.arange(_plo, _phi)))
                _band = tvt_full[ps_md:_p]
                _blo = float(np.nanmin(_band)); _bhi = float(np.nanmax(_band)); _ext = _bhi - _blo
                if _ext >= max(8.0, REPEAT_SHIFT_MIN):
                    _target = float(prng.uniform(_blo + 0.15 * _ext, _bhi - 0.15 * _ext))
                    _s = float(np.clip(_target - tvt_full[_p], -REPEAT_SHIFT_MAX, REPEAT_SHIFT_MAX))
                    if abs(_s) >= 2.0 and (_blo <= tvt_full[_p] + _s <= _bhi):
                        _L = int(prng.uniform(REPEAT_L_MIN, REPEAT_L_MAX))
                        _dp = float(prng.uniform(REPEAT_DAMP_MIN, REPEAT_DAMP_MAX))
                        tvt_full[_p:] += _s
                        _e = min(_p + _L, n_md)
                        if _e - _p >= 3:
                            _seg = tvt_full[_p:_e].copy()
                            _new = _seg[0] + np.concatenate([[0.0], np.cumsum(np.diff(_seg) * _dp)])
                            _tail = _new[-1] - _seg[-1]
                            tvt_full[_p:_e] = _new
                            tvt_full[_e:] += _tail
                        _rep_span = (_p, _e)
                        _ev.append({'type': 'repeat', 'md_idx': _p, 'shift': _s, 'damp': _dp, 'L': int(_e - _p)})
            # --- B2 pinch-out: interval-selective thinning of the EXPERIENCED column. Lookup-side
            #     only (labels honest, Z untouched): the local interval [t0, t0+c*h] holds the FULL
            #     reference interval [t0, t0+h] -> the tracked marker compresses toward vanishing.
            #     Thickness = the measured tolerant axis -> dosed hot.
            if prng.uniform() < PINCH_PROB:
                _pt = tvt_full[ps_md:]
                _t0r = float(np.nanmin(_pt)); _t1r = float(np.nanmax(_pt))
                _h = float(prng.uniform(PINCH_H_MIN, PINCH_H_MAX))
                if _t1r - _t0r > _h + 2.0:
                    _t0 = float(prng.uniform(_t0r, _t1r - _h)); _c = float(prng.uniform(PINCH_C_MIN, PINCH_C_MAX))
                    # lens/wedge geometry: the reference displacement (1-c)*h TAPERS back to identity
                    # over a recovery band above the lens (monotone: REC >= 1.5*delta), so the
                    # decorrelation stays LOCAL to the crossing instead of a permanent column offset
                    # (permanent offset at p=0.30 would gut pooled match_corr — pregate-measured).
                    _rec = float((1.0 - _c) * _h * prng.uniform(1.2, 2.0))   # monotone (slope >= 0.167); tight so decorrelation stays local
                    _pinch = (_t0, _h, _c, _rec)
                    _ev.append({'type': 'pinch', 't0': _t0, 'h': _h, 'c': _c, 'rec': _rec})
            # --- guards on the pack's MARGINAL effect (accept / reject-and-redraw) ---
            _dd = np.abs(np.diff((tvt_full - _base_tvt)[ps_md:]))   # fault jumps cancel in the delta
            if _rep_span is not None:                                # damping (slope REDUCTION) + the
                _a = max(_rep_span[0] - ps_md - 1, 0)                # repeat jump are masked: the guard
                _b = min(_rep_span[1] - ps_md + 1, len(_dd))         # caps ADDED slope only
                if _b > _a: _dd[_a:_b] = 0.0
            _pack_slope = float(_dd.max()) if len(_dd) else 0.0
            _post = tvt_full[ps_md:]
            _exc = float(np.nanmax(_post) - np.nanmin(_post))
            if _pack_slope <= EXPA_MAX_ABS_DIP and _exc <= max(_b_exc, EXPA_EXC_CAP_FT):
                pack_info['events'] = _ev; pack_info['tries'] = _try + 1
                if _pinch is not None:
                    _t0, _h, _c, _rec = _pinch
                    def _pinch_map_fn(t, t0=_t0, h=_h, c=_c, rec=_rec):
                        u = np.asarray(t, dtype=np.float64).copy()
                        d = (1.0 - c) * h
                        mid = (u > t0) & (u < t0 + c * h)
                        rcv = (u >= t0 + c * h) & (u < t0 + c * h + rec)
                        u[mid] = t0 + (u[mid] - t0) / c
                        u[rcv] = u[rcv] + d * (1.0 - (u[rcv] - (t0 + c * h)) / rec)
                        return u   # above the recovery band: identity (local lens, not a column shift)
                break
        else:
            tvt_full = _base_tvt
            pack_info['dropped'] = True; pack_info['tries'] = 6


    tvt_offset = make_correlated_noise(n_md, autocorr_length=tvt_offset_autocorr,
                                        target_std=sigma_tvt_offset, rng=rng)
    _lk = tvt_full
    if _pinch_map_fn is not None:
        _lk = _pinch_map_fn(_lk)             # DORMANT in this pack (PINCH_PROB = 0, A2-class)
    tvt_for_lookup = np.clip(_lk + tvt_offset, tw_tvt[0], tw_tvt[-1])
    hw_gr_lookup = np.interp(tvt_for_lookup, tw_tvt, tw_gr_arr)
    calib_factor = 1.0 + rng.normal(0, sigma_calib)
    drift_noise = make_correlated_noise(n_md, autocorr_length=drift_autocorr,
                                         target_std=sigma_drift, rng=rng)
    meas_noise = rng.normal(0, sigma_meas, n_md)
    hw_gr_synth = hw_gr_lookup * calib_factor + drift_noise + meas_noise
    hw_gr_synth = np.clip(hw_gr_synth, 10, 250).astype(np.float32)
    tvt_input = np.full(n_md, np.nan)
    tvt_input[:ps_md] = tvt_full[:ps_md]
    # --- synthetic Z (vertical elevation): -TVT + smooth formation dip + small drilling noise ---
    # mirrors real Z<->TVT relationship from the leak findings: tightly correlated but not
    # deterministic. Decoupling preserves the channel as informative input without making it
    # a perfect TVT predictor during pretrain.
    synth_z_dip_amp = tvt_range * DZ_SYNTH_DIP_FRAC * rng.uniform(0.5, 1.5)
    synth_z_dip_acl = max(200, int(n_md * DZ_SYNTH_DIP_AUTOCORR_FRAC))
    synth_z_dip = make_correlated_noise(n_md, autocorr_length=synth_z_dip_acl,
                                         target_std=synth_z_dip_amp, rng=rng)
    synth_z_drill_noise = rng.normal(0, DZ_SYNTH_NOISE_STD, n_md)
    # FT39j: Z uses PRE-fault TVT (tvt_full_for_z) so wellbore stays continuous across faults
    z_full = (-tvt_full_for_z + synth_z_dip + synth_z_drill_noise).astype(np.float32)
    return {
        'MD': md_arr.astype(np.float32),
        'TVT': tvt_full.astype(np.float32),
        'TVT_input': tvt_input.astype(np.float32),
        'GR': hw_gr_synth,
        'Z': z_full,
        'ps_md_idx': ps_md,
        '_tw_tvt': tw_tvt.astype(np.float32),
        '_tw_gr': tw_gr_arr.astype(np.float32),
        '_fault_info': fault_info,
        '_unconf_info': unconf_info,
        '_var_bed_info': var_bed_info,
        '_pack_info': pack_info,
        '_fz_info': _fz_info,
    }

def build_hint_real_dist(train_wids):
    rows = []
    for wid in train_wids:
        hw = pd.read_csv(well_hw_path(wid), usecols=['MD','TVT','TVT_input'])
        n_md = len(hw)
        known = ~hw['TVT_input'].isna()
        if known.sum() < 1 or known.sum() >= n_md: continue
        ps_md_idx = int(known.sum())
        ps_md_pct = 100.0 * ps_md_idx / n_md
        post_tvt = hw['TVT'].values[ps_md_idx:]
        post_ok = ~np.isnan(post_tvt)
        if post_ok.sum() < 10: continue
        tvt_range = float(np.nanmax(post_tvt) - np.nanmin(post_tvt))
        rows.append({'well_id': wid, 'n_md': n_md, 'ps_md_pct': ps_md_pct,
                     'tvt_range': tvt_range})
    df = pd.DataFrame(rows)
    return {'n_md': df['n_md'].values, 'ps_md_pct': df['ps_md_pct'].values,
            'tvt_range': df['tvt_range'].values}, df


In [ ]:
# === §FZ-SMOKE — identities + planted scenarios (data-free; run BEFORE any pool build) =========
# House pattern (G1.1/G1b/G2): (1) EVENT_PACK='v3' byte-identical to the PRISTINE baseline
# generator (embedded verbatim below); (2) LIMIT IDENTITY — FZ armed with null params == FZ off,
# byte-for-byte; (3) RNG ISOLATION — armed vs off at matched seed leaves MD/Z/TVT_input/pre-PS
# TVT/TW byte-equal (base stream untouched); (4) PLANTED GROWTH — throw ceiling scales with
# position per the s(pos) law; (5) PLANTED GRABEN — full-recovery pair restores the section
# exactly beyond fault #2, displaces exactly m1 between; (6) RAMP SHAPE + CARVE-OUT — smoothstep
# accrual, exact plateau, discrete slope <= min(1.6*|m|/w, FZ_RAMP_MAX_SLOPE), ZERO added slope
# outside ramp spans; (7) armed determinism. Knobs restored to launch values at the end.

# --- pristine baseline generator (verbatim copy of the pack-free cell-7 function, renamed) ---
def _pristine_generate_v3(
    real_tw_df, hint_real_dist, seed=None,
    sigma_drift=8.0, sigma_calib=0.06,
    sigma_tvt_offset=3.5, sigma_meas=1.5,
    drift_autocorr=40, tvt_offset_autocorr=200,
    wobble_total_pct=V3_WOBBLE_TOTAL_PCT,
    wobble_split=V3_WOBBLE_SPLIT,
    wavelength_divisors=V3_WAVELENGTH_DIVISORS,
    fault_prob=0.0,                       # FT39j: prob of any fault events
    fault_magnitude_min=5.0,              # FT39j: min throw (ft)
    fault_magnitude_max=50.0,             # FT39j: max throw (ft)
    unconf_prob=0.0,                      # FT39k: prob of unconformity (slope change)
    unconf_max_cumulative_shift=30.0,     # FT39k: bounded cumulative shift (ft)
    var_bed_prob=0.0,                     # FT39l: prob of variable bed thickness
    var_bed_amp_std=0.4,                  # FT39l: std of amplitude modulation around 1.0
    var_bed_autocorr_length=200,          # FT39l: autocorr length of modulation (rows)
    var_bed_amp_min=0.3,                  # FT39l: clip lower bound on modulation
    var_bed_amp_max=2.0,                  # FT39l: clip upper bound on modulation
):
    rng = np.random.RandomState(seed)
    tw_tvt = real_tw_df['TVT'].values
    tw_gr_arr = real_tw_df['GR'].values
    ok = ~np.isnan(tw_tvt) & ~np.isnan(tw_gr_arr)
    tw_tvt = tw_tvt[ok]; tw_gr_arr = tw_gr_arr[ok]
    if len(tw_tvt) < 100: return None
    sort = np.argsort(tw_tvt)
    tw_tvt = tw_tvt[sort]; tw_gr_arr = tw_gr_arr[sort]
    n_md = int(rng.choice(hint_real_dist['n_md']))
    ps_md_pct = float(rng.choice(hint_real_dist['ps_md_pct']))
    tvt_range = float(rng.choice(hint_real_dist['tvt_range']))
    tw_lo, tw_hi = tw_tvt[0], tw_tvt[-1]
    margin = tvt_range * 1.5
    if tw_hi - tw_lo < 2*margin:
        heel_tvt = (tw_lo + tw_hi) / 2
    else:
        heel_tvt = float(rng.uniform(tw_lo + margin, tw_hi - margin))
    md_arr = np.arange(n_md).astype(np.float64)
    lateral_md = n_md * (1 - ps_md_pct/100)
    if lateral_md < 100: return None
    dip_sign = rng.choice([-1, 1])
    dip_rate = dip_sign * tvt_range / max(lateral_md, 1)
    wobble_total_amp = tvt_range * wobble_total_pct * rng.uniform(0.7, 1.3)
    amp_lo = wobble_total_amp * wobble_split[0] * rng.uniform(0.5, 1.5)
    amp_mid = wobble_total_amp * wobble_split[1] * rng.uniform(0.5, 1.5)
    amp_hi = wobble_total_amp * wobble_split[2] * rng.uniform(0.5, 1.5)
    omega_lo = 2 * np.pi / (lateral_md * wavelength_divisors[0] * rng.uniform(0.7, 1.3))
    omega_mid = 2 * np.pi / (lateral_md / wavelength_divisors[1] * rng.uniform(0.7, 1.3))
    omega_hi = 2 * np.pi / (lateral_md / wavelength_divisors[2] * rng.uniform(0.7, 1.3))
    phi_lo = rng.uniform(0, 2*np.pi)
    phi_mid = rng.uniform(0, 2*np.pi)
    phi_hi = rng.uniform(0, 2*np.pi)
    ps_md = int(ps_md_pct/100 * n_md)
    ps_md = max(ps_md, 100); ps_md = min(ps_md, n_md - 100)
    pre_md = md_arr[:ps_md]
    pre_start_tvt = max(tw_lo, heel_tvt - tvt_range * 8)
    pre_t = pre_md / max(ps_md - 1, 1)
    pre_tvt = pre_start_tvt + (heel_tvt - pre_start_tvt) * (0.5 - 0.5 * np.cos(pre_t * np.pi))
    post_md = md_arr[ps_md:] - md_arr[ps_md]

    # === FT39l: variable bed thickness modulation ===
    # Apply smooth modulation to wobble amplitudes across MD. Different mechanism from
    # faults/unconformities (discrete events): continuous amplitude variation. Some MD
    # regions have less wobble (thicker beds), others have more (thinner beds).
    var_bed_info = None
    if var_bed_prob > 0 and rng.uniform() < var_bed_prob:
        var_bed_mod_full = make_correlated_noise(
            n_md, autocorr_length=var_bed_autocorr_length,
            target_std=var_bed_amp_std, rng=rng
        ) + 1.0  # centered at 1.0
        var_bed_mod_full = np.clip(var_bed_mod_full, var_bed_amp_min, var_bed_amp_max).astype(np.float64)
        var_bed_info = {
            'mod_std_observed': float(np.std(var_bed_mod_full)),
            'mod_min': float(var_bed_mod_full.min()),
            'mod_max': float(var_bed_mod_full.max()),
        }
    else:
        var_bed_mod_full = np.ones(n_md, dtype=np.float64)
    var_bed_mod_post = var_bed_mod_full[ps_md:ps_md + len(post_md)]

    post_tvt = (heel_tvt + dip_rate * post_md
                + var_bed_mod_post * amp_lo  * np.sin(omega_lo  * post_md + phi_lo)
                + var_bed_mod_post * amp_mid * np.sin(omega_mid * post_md + phi_mid)
                + var_bed_mod_post * amp_hi  * np.sin(omega_hi  * post_md + phi_hi))
    tvt_full = np.concatenate([pre_tvt, post_tvt])

    # === FT39j: optional fault overlay ===
    # Apply TVT jumps post-PS. Z (computed below) uses PRE-fault TVT so wellbore stays
    # smooth across faults — formation top jumps, wellbore doesn't. GR auto-resamples from
    # post-fault TVT via the existing TW lookup.
    tvt_full_for_z = tvt_full.copy()   # snapshot of pre-fault TVT for Z derivation
    fault_info = []
    if fault_prob > 0 and rng.uniform() < fault_prob:
        n_faults = 1 if rng.uniform() < 0.7 else 2
        post_md_lo = ps_md + int(0.20 * (n_md - ps_md))
        post_md_hi = ps_md + int(0.80 * (n_md - ps_md))
        if post_md_hi - post_md_lo >= 50:
            n_faults_eff = min(n_faults, post_md_hi - post_md_lo)
            positions = np.sort(rng.choice(np.arange(post_md_lo, post_md_hi),
                                            size=n_faults_eff, replace=False))
            magnitudes = rng.uniform(fault_magnitude_min, fault_magnitude_max,
                                      size=n_faults_eff) * rng.choice([-1, 1], size=n_faults_eff)
            for pos, mag in zip(positions, magnitudes):
                tvt_full[pos:] += mag
                fault_info.append({'md_idx': int(pos), 'magnitude': float(mag)})

    # === FT39k: optional unconformity overlay (slope change) ===
    # Like faults: TVT shifts but Z (computed from tvt_full_for_z) stays smooth — formation
    # dip changes, wellbore doesn't. Cumulative shift bounded to keep the change visible but
    # not dominant relative to natural wobble.
    unconf_info = None
    if unconf_prob > 0 and rng.uniform() < unconf_prob:
        unconf_lo = ps_md + int(0.20 * (n_md - ps_md))
        unconf_hi = ps_md + int(0.70 * (n_md - ps_md))   # leave 30%+ for post-unconf slope to develop
        if unconf_hi - unconf_lo >= 100:
            unconf_pos = int(rng.choice(np.arange(unconf_lo, unconf_hi)))
            lateral_remaining = n_md - unconf_pos
            # Pick a delta_slope such that cumulative shift over remaining MD is within bound
            max_delta_slope = unconf_max_cumulative_shift / max(lateral_remaining, 1)
            unconf_delta_slope = float(rng.uniform(-max_delta_slope, max_delta_slope))
            # Apply: TVT shifts by delta_slope * (md - unconf_pos) for md >= unconf_pos
            md_relative = (np.arange(n_md) - unconf_pos).astype(np.float64)
            md_relative[md_relative < 0] = 0.0
            tvt_full = tvt_full + unconf_delta_slope * md_relative
            unconf_info = {'md_idx': unconf_pos, 'delta_slope': unconf_delta_slope,
                            'cumulative_shift': float(unconf_delta_slope * lateral_remaining)}

    tvt_offset = make_correlated_noise(n_md, autocorr_length=tvt_offset_autocorr,
                                        target_std=sigma_tvt_offset, rng=rng)
    tvt_for_lookup = np.clip(tvt_full + tvt_offset, tw_tvt[0], tw_tvt[-1])
    hw_gr_lookup = np.interp(tvt_for_lookup, tw_tvt, tw_gr_arr)
    calib_factor = 1.0 + rng.normal(0, sigma_calib)
    drift_noise = make_correlated_noise(n_md, autocorr_length=drift_autocorr,
                                         target_std=sigma_drift, rng=rng)
    meas_noise = rng.normal(0, sigma_meas, n_md)
    hw_gr_synth = hw_gr_lookup * calib_factor + drift_noise + meas_noise
    hw_gr_synth = np.clip(hw_gr_synth, 10, 250).astype(np.float32)
    tvt_input = np.full(n_md, np.nan)
    tvt_input[:ps_md] = tvt_full[:ps_md]
    # --- synthetic Z (vertical elevation): -TVT + smooth formation dip + small drilling noise ---
    # mirrors real Z<->TVT relationship from the leak findings: tightly correlated but not
    # deterministic. Decoupling preserves the channel as informative input without making it
    # a perfect TVT predictor during pretrain.
    synth_z_dip_amp = tvt_range * DZ_SYNTH_DIP_FRAC * rng.uniform(0.5, 1.5)
    synth_z_dip_acl = max(200, int(n_md * DZ_SYNTH_DIP_AUTOCORR_FRAC))
    synth_z_dip = make_correlated_noise(n_md, autocorr_length=synth_z_dip_acl,
                                         target_std=synth_z_dip_amp, rng=rng)
    synth_z_drill_noise = rng.normal(0, DZ_SYNTH_NOISE_STD, n_md)
    # FT39j: Z uses PRE-fault TVT (tvt_full_for_z) so wellbore stays continuous across faults
    z_full = (-tvt_full_for_z + synth_z_dip + synth_z_drill_noise).astype(np.float32)
    return {
        'MD': md_arr.astype(np.float32),
        'TVT': tvt_full.astype(np.float32),
        'TVT_input': tvt_input.astype(np.float32),
        'GR': hw_gr_synth,
        'Z': z_full,
        'ps_md_idx': ps_md,
        '_tw_tvt': tw_tvt.astype(np.float32),
        '_tw_gr': tw_gr_arr.astype(np.float32),
        '_fault_info': fault_info,
        '_unconf_info': unconf_info,
        '_var_bed_info': var_bed_info,
    }

_FZ_KNOBS = ['EVENT_PACK', 'FZ_ENABLE', 'FZ_GROWTH_ENABLE', 'FZ_GROWTH_FLOOR', 'FZ_PAIR_PROB',
             'FZ_PAIR_SPACING_MIN', 'FZ_PAIR_SPACING_MAX', 'FZ_PAIR_RECOVER_MIN', 'FZ_PAIR_RECOVER_MAX',
             'FZ_RAMP_SHARP_P', 'FZ_RAMP_W_MIN', 'FZ_RAMP_W_MAX',
             'DIPBREAK_PROB', 'REPEAT_PROB', 'PINCH_PROB']
_fz_saved = {k: globals()[k] for k in _FZ_KNOBS}
_ARRK = ['MD', 'TVT', 'TVT_input', 'GR', 'Z', '_tw_tvt', '_tw_gr']

def _fz_fx_tw():
    _t = np.arange(0.0, 400.0, 0.5)
    _gr = 80 + 40 * np.sin(_t / 7.0) + 25 * np.sin(_t / 23.0 + 1.0) + 10 * np.sin(_t / 2.3)
    return pd.DataFrame({'TVT': _t, 'GR': _gr})

_fz_fx_hint = {'n_md': np.array([2200, 2600, 3000]), 'ps_md_pct': np.array([25.0, 30.0]),
               'tvt_range': np.array([25.0, 30.0, 40.0])}

def _fz_gen(seed, fault_prob, fn=None, **kv):
    for _k, _v in kv.items():
        globals()[_k] = _v
    try:
        _f = fn if fn is not None else generate_synthetic_well_method_a_v3
        return _f(_fz_fx_tw(), _fz_fx_hint, seed=seed, fault_prob=fault_prob,
                  unconf_prob=0.33, var_bed_prob=0.5)
    finally:
        for _k in kv:
            globals()[_k] = _fz_saved[_k]

def _fz_eq(a, b):
    if a is None or b is None:
        return a is None and b is None
    return all(np.array_equal(a[k], b[k], equal_nan=True) for k in _ARRK)

_fz_null = dict(FZ_ENABLE=True, FZ_PAIR_PROB=0.0, FZ_RAMP_SHARP_P=1.0, FZ_GROWTH_ENABLE=True,
                FZ_GROWTH_FLOOR=1.0)
_fz_quiet_pack = dict(DIPBREAK_PROB=0.0, REPEAT_PROB=0.0, PINCH_PROB=0.0)

# --- (1) pristine identity: EVENT_PACK='v3' + FZ off == the pack-free baseline generator ---
for _sd in range(120):
    _a = _fz_gen(_sd, 0.5, EVENT_PACK='v3', FZ_ENABLE=False)
    _b = _fz_gen(_sd, 0.5, fn=_pristine_generate_v3, EVENT_PACK='v3', FZ_ENABLE=False)
    assert _fz_eq(_a, _b), f'[FZ-smoke 1] pristine identity FAILED at seed {_sd}'
print('[FZ-smoke 1] PASS — EVENT_PACK=v3 byte-identical to the pristine baseline generator (120 wells)')

# --- (2) limit identity: pack ARMED, FZ null params == FZ off, byte-for-byte ---
for _sd in range(120):
    _a = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', FZ_ENABLE=False)
    _b = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', **_fz_null)
    assert _fz_eq(_a, _b), f'[FZ-smoke 2] limit identity FAILED at seed {_sd}'
print('[FZ-smoke 2] PASS — FZ(null params) == FZ off under the armed pack (120 wells)')

# --- (3) RNG isolation at LAUNCH knobs: base stream untouched; treatment is post-PS-only ---
_n_diff = 0
for _sd in range(120):
    _a = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', FZ_ENABLE=False)
    _b = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', FZ_ENABLE=True)
    if _a is None and _b is None:
        continue
    for _k in ['MD', 'Z', 'TVT_input', '_tw_tvt', '_tw_gr']:
        assert np.array_equal(_a[_k], _b[_k], equal_nan=True), f'[FZ-smoke 3] {_k} moved at seed {_sd}'
    _ps = _a['ps_md_idx']
    assert np.array_equal(_a['TVT'][:_ps], _b['TVT'][:_ps]), f'[FZ-smoke 3] pre-PS TVT moved at seed {_sd}'
    if not np.array_equal(_a['TVT'], _b['TVT']):
        _n_diff += 1
assert _n_diff > 10, f'[FZ-smoke 3] armed path produced almost no treatment ({_n_diff}/120 wells moved)'
print(f'[FZ-smoke 3] PASS — MD/Z/TVT_input/pre-PS/TW byte-equal armed<->off; {_n_diff}/120 wells treated post-PS')

# --- (4) planted growth: ceiling law + monotone position profile ---
_pos, _mag = [], []
for _sd in range(400):
    _w = _fz_gen(_sd, 1.0, EVENT_PACK='v3_expA', FZ_ENABLE=True, FZ_PAIR_PROB=0.0,
                 FZ_RAMP_SHARP_P=1.0, **_fz_quiet_pack)
    if _w is None:
        continue
    for _f in _w['_fault_info']:
        _s = FZ_GROWTH_FLOOR + (1.0 - FZ_GROWTH_FLOOR) * _f['pos_frac']
        assert abs(_f['magnitude']) <= 5.0 + 45.0 * _s + 1e-6, f'[FZ-smoke 4] ceiling law broken at seed {_sd}'
        assert abs(_f['magnitude']) >= 5.0 - 1e-9, f'[FZ-smoke 4] floor broken at seed {_sd}'
        _pos.append(_f['pos_frac']); _mag.append(abs(_f['magnitude']))
_pos = np.array(_pos); _mag = np.array(_mag)
_lo = _mag[_pos <= 0.33].mean(); _hi = _mag[_pos >= 0.67].mean()
assert _hi / _lo >= 1.2, f'[FZ-smoke 4] growth profile too flat: toe/heel mean|throw| = {_hi:.1f}/{_lo:.1f}'
print(f'[FZ-smoke 4] PASS — growth ceiling law holds on {len(_mag)} events; mean|throw| heel {_lo:.1f} -> toe {_hi:.1f} ft')

# --- (5) planted graben: full-recovery antithetic pair ---
_n_pair = 0
for _sd in range(200):
    if _n_pair >= 40:
        break
    _kv = dict(EVENT_PACK='v3_expA', FZ_ENABLE=True, FZ_PAIR_PROB=1.0, FZ_RAMP_SHARP_P=1.0,
               FZ_GROWTH_ENABLE=False, FZ_PAIR_RECOVER_MIN=1.0, FZ_PAIR_RECOVER_MAX=1.0,
               **_fz_quiet_pack)
    _a = _fz_gen(_sd, 0.0, EVENT_PACK='v3_expA', FZ_ENABLE=False, **_fz_quiet_pack)
    _b = _fz_gen(_sd, 0.0, **_kv)
    if _b is None or not _b['_fz_info']['pair']:
        continue
    _pf = [_f for _f in _b['_fault_info'] if _f['kind'] == 'pair']
    assert len(_pf) == 2, '[FZ-smoke 5] pair count != 2'
    _p1, _p2 = _pf[0]['md_idx'], _pf[1]['md_idx']
    _m1, _m2 = _pf[0]['magnitude'], _pf[1]['magnitude']
    _ps = _b['ps_md_idx']; _n = len(_b['TVT'])
    _plo = _ps + int(0.20 * (_n - _ps)); _phi = _ps + int(0.80 * (_n - _ps))
    assert _m2 == -_m1, '[FZ-smoke 5] recover=1.0 pair not exactly antithetic'
    assert FZ_PAIR_SPACING_MIN <= _p2 - _p1 <= FZ_PAIR_SPACING_MAX, '[FZ-smoke 5] spacing out of range'
    assert _plo <= _p1 and _p2 < _phi, '[FZ-smoke 5] pair escaped the 20-80% window'
    _d = _b['TVT'].astype(np.float64) - _a['TVT'].astype(np.float64)
    assert np.abs(_d[_p2:]).max() <= 1e-3, '[FZ-smoke 5] section did not recover beyond fault #2'
    assert np.abs(_d[_p1:_p2] - _m1).max() <= 1e-3, '[FZ-smoke 5] graben interval displacement != m1'
    assert np.abs(_d[:_p1]).max() <= 1e-3, '[FZ-smoke 5] displacement before fault #1'
    _n_pair += 1
assert _n_pair >= 30, f'[FZ-smoke 5] too few planted pairs realized ({_n_pair})'
print(f'[FZ-smoke 5] PASS — {_n_pair} planted grabens: exact interval displacement + exact recovery')

# --- (6) ramp shape + envelope carve-out ---
_n_ramp = 0
for _sd in range(300):
    if _n_ramp >= 40:
        break
    _kv = dict(EVENT_PACK='v3_expA', FZ_ENABLE=True, FZ_PAIR_PROB=0.0, FZ_RAMP_SHARP_P=0.0,
               FZ_GROWTH_ENABLE=False, **_fz_quiet_pack)
    _a = _fz_gen(_sd, 1.0, EVENT_PACK='v3_expA', FZ_ENABLE=False, **_fz_quiet_pack)
    _b = _fz_gen(_sd, 1.0, **_kv)
    if _b is None or len(_b['_fault_info']) != 1 or _b['_fault_info'][0]['ramp_w'] == 0:
        continue
    _f = _b['_fault_info'][0]; _p = _f['md_idx']; _m = _f['magnitude']; _w = _f['ramp_w']
    assert FZ_RAMP_W_MIN <= _w <= FZ_RAMP_W_MAX, '[FZ-smoke 6] ramp width out of range'
    # The off-twin applies the SAME fault as a sharp step (identical base draws) -> the delta's
    # closed form is m*(smoothstep - 1) inside the ramp and EXACTLY ZERO outside (plateau + pre-
    # fault + carve-out all in one elementwise check).
    _d = _b['TVT'].astype(np.float64) - _a['TVT'].astype(np.float64)
    _t = (np.arange(_w, dtype=np.float64) + 1.0) / _w
    _exp = np.zeros(len(_d)); _exp[_p:_p + _w] = _m * (3.0 * _t**2 - 2.0 * _t**3 - 1.0)
    assert np.abs(_d - _exp).max() <= 1e-3, '[FZ-smoke 6] delta != closed-form smoothstep (shape/plateau/carve-out)'
    _dd = np.diff(_d[_p:_p + _w])   # in-ramp increments only (the twin's sharp jump at p is out of frame)
    assert (np.sign(_dd[np.abs(_dd) > 1e-3]) == np.sign(_m)).all(), '[FZ-smoke 6] non-monotone accrual'
    assert np.abs(_dd).max() <= min(1.6 * abs(_m) / _w, FZ_RAMP_MAX_SLOPE) + 1e-3, '[FZ-smoke 6] ramp slope over envelope'
    _n_ramp += 1
assert _n_ramp >= 25, f'[FZ-smoke 6] too few single-fault ramps realized ({_n_ramp})'
print(f'[FZ-smoke 6] PASS — {_n_ramp} planted ramps match the closed-form smoothstep (shape+plateau+carve-out)')

# --- (7) armed determinism + pack-liveness sanity ---
for _sd in (3, 77):
    _a = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', FZ_ENABLE=True)
    _b = _fz_gen(_sd, 0.5, EVENT_PACK='v3_expA', FZ_ENABLE=True)
    assert _fz_eq(_a, _b), '[FZ-smoke 7] armed generator not deterministic'
_n_packev = sum(len((_fz_gen(_sd, 0.33, EVENT_PACK='v3_expA', FZ_ENABLE=True) or {'_pack_info': None})
                    .get('_pack_info', {}).get('events', []) or [])
                for _sd in range(200))
assert _n_packev > 20, f'[FZ-smoke 7] ported pack looks dead ({_n_packev} events / 200 wells)'
print(f'[FZ-smoke 7] PASS — armed determinism holds; ported pack live ({_n_packev} pack events / 200 wells)')

for _k in _FZ_KNOBS:
    globals()[_k] = _fz_saved[_k]
print('\n§FZ-SMOKE COMPLETE — knobs restored to launch values:')
print(f"  EVENT_PACK={EVENT_PACK} | FZ_ENABLE={FZ_ENABLE} | growth(floor={FZ_GROWTH_FLOOR}) | "
      f"pair(p={FZ_PAIR_PROB}) | ramp(sharp_p={FZ_RAMP_SHARP_P}, w {FZ_RAMP_W_MIN}-{FZ_RAMP_W_MAX})")


In [ ]:
# Per-fold synth pools with leakage discipline.
# Two pools per fold:
#   - synth_pools[fold]       : 700 wells for joint arm (locked baseline)
#   - synth_pools_pretrain[fold] : 2000 wells for pretrain arm
#
# Both pools share leakage discipline: TW sources from train-only, hint_real_dist from train-only.
# The two pools are INDEPENDENT draws — the pretrain pool is not a superset of the joint pool
# (different seeds → different TW picks). This is fine because the experiment compares METHODS,
# not pool-identity. Each method gets a pool sized appropriately for its training regime.

synth_pools = {}
synth_pools_pretrain = {}
hint_dists = {}

for fold in ALL_FOLDS:
    train_wells = fold_splits[fold]['train']
    print(f'\n--- Fold {fold}: {len(train_wells)} train wells ---')
    hint_dist, hint_df = build_hint_real_dist(train_wells)
    hint_dists[fold] = hint_dist
    print(f'  hint_real_dist built from {len(hint_df)} train wells')

    # --- Pretrain arm pool (2000) ---
    # Use DIFFERENT seed offset so the pretrain pool is an independent draw
    # (not a superset of the joint pool) — methods differ structurally, this is intentional.
    fold_rng_pre = np.random.RandomState(SEED + 5000 + fold)
    tw_picks_pre = fold_rng_pre.choice(train_wells, size=N_SYNTH_PRETRAIN, replace=True)
    pool_pre = []
    t0 = time.time()
    for i, tw_wid in enumerate(tw_picks_pre):
        tw = pd.read_csv(well_tw_path(tw_wid), usecols=['TVT','GR'])
        syn = generate_synthetic_well_method_a_v3(
            tw, hint_dist, seed=SEED + 7000 + fold*100000 + i,
            sigma_drift=SIGMA_DRIFT, sigma_tvt_offset=SIGMA_TVT_OFFSET,
            sigma_meas=SIGMA_MEAS, sigma_calib=SIGMA_CALIB,
            wobble_total_pct=V3_WOBBLE_TOTAL_PCT,
            wobble_split=V3_WOBBLE_SPLIT,
            wavelength_divisors=V3_WAVELENGTH_DIVISORS,
            fault_prob=FAULT_PROB,
            fault_magnitude_min=FAULT_MAGNITUDE_MIN,
            fault_magnitude_max=FAULT_MAGNITUDE_MAX,
            unconf_prob=UNCONF_PROB,
            unconf_max_cumulative_shift=UNCONF_MAX_CUMULATIVE_SHIFT,
            var_bed_prob=VAR_BED_PROB,
            var_bed_amp_std=VAR_BED_AMP_STD,
            var_bed_autocorr_length=VAR_BED_AUTOCORR_LENGTH,
            var_bed_amp_min=VAR_BED_AMP_MIN,
            var_bed_amp_max=VAR_BED_AMP_MAX,
        )
        if syn is not None:
            syn['_synth_id'] = f'synth_f{fold}_pretrain_{i:04d}'
            syn['_seed_tw_wid'] = tw_wid
            syn['_pool'] = 'pretrain'
            pool_pre.append(syn)
    synth_pools_pretrain[fold] = pool_pre
    print(f'  Pretrain pool: generated {len(pool_pre)} v3 synth wells in {time.time()-t0:.1f}s')
    # FT39j: fault overlay statistics
    n_with_faults = sum(1 for w in pool_pre if w.get('_fault_info'))
    n_1fault = sum(1 for w in pool_pre if len(w.get('_fault_info', [])) == 1)
    n_2faults = sum(1 for w in pool_pre if len(w.get('_fault_info', [])) == 2)
    all_mags = [abs(f['magnitude']) for w in pool_pre for f in w.get('_fault_info', [])]
    if all_mags:
        print(f'  Fault overlay: {n_with_faults}/{len(pool_pre)} ({100*n_with_faults/len(pool_pre):.1f}%) wells faulted '
              f'({n_1fault} with 1, {n_2faults} with 2). '
              f'Magnitude: mean={np.mean(all_mags):.1f}, std={np.std(all_mags):.1f}, range=[{min(all_mags):.1f}, {max(all_mags):.1f}] ft')
    else:
        print(f'  Fault overlay: 0 wells had faults (check FAULT_PROB)')
    # FT39k: unconformity overlay statistics
    n_with_unconf = sum(1 for w in pool_pre if w.get('_unconf_info'))
    unconf_cumulatives = [abs(w['_unconf_info']['cumulative_shift']) for w in pool_pre if w.get('_unconf_info')]
    if unconf_cumulatives:
        print(f'  Unconformity overlay: {n_with_unconf}/{len(pool_pre)} ({100*n_with_unconf/len(pool_pre):.1f}%) wells. '
              f'Cumulative shift: mean={np.mean(unconf_cumulatives):.1f}, max={max(unconf_cumulatives):.1f} ft')
    else:
        print(f'  Unconformity overlay: 0 wells had unconformity (check UNCONF_PROB)')
    # FT39l: variable bed thickness statistics
    n_with_varbed = sum(1 for w in pool_pre if w.get('_var_bed_info'))
    varbed_stds = [w['_var_bed_info']['mod_std_observed'] for w in pool_pre if w.get('_var_bed_info')]
    if varbed_stds:
        print(f'  Variable bed thickness: {n_with_varbed}/{len(pool_pre)} ({100*n_with_varbed/len(pool_pre):.1f}%) wells. '
              f'Modulation std observed: mean={np.mean(varbed_stds):.3f}')
    else:
        print(f'  Variable bed thickness: 0 wells (check VAR_BED_PROB)')
    # Triple-event joint statistics
    n_all_three = sum(1 for w in pool_pre if w.get('_fault_info') and w.get('_unconf_info') and w.get('_var_bed_info'))
    n_any = sum(1 for w in pool_pre if w.get('_fault_info') or w.get('_unconf_info') or w.get('_var_bed_info'))
    n_none = sum(1 for w in pool_pre if not w.get('_fault_info') and not w.get('_unconf_info') and not w.get('_var_bed_info'))

    # [EVEX-A + FZ] realized-dose echo (METHODOLOGY §12: launch evidence from LIVE values)
    if EVENT_PACK == 'v3_expA':
        _pk = [w.get('_pack_info') for w in pool_pre if w.get('_pack_info')]
        _pev = [e for z in _pk for e in z.get('events', [])]
        _pdrp = sum(1 for z in _pk if z.get('dropped'))
        _ndip = sum(1 for e in _pev if e['type'] == 'dipbreak'); _nrep = sum(1 for e in _pev if e['type'] == 'repeat')
        print(f'  [pack] dipbreaks {_ndip} | repeats {_nrep} | guard-dropped wells {_pdrp}/{len(_pk)}')
    if FZ_ENABLE:
        _fzs = [w.get('_fz_info') for w in pool_pre if w.get('_fz_info')]
        _npair = sum(1 for z in _fzs if z['pair']); _ndropp = sum(1 for z in _fzs if z['pair_dropped'])
        _nram = sum(z['ramped'] for z in _fzs); _nshp = sum(z['sharp'] for z in _fzs)
        _ws = [w_ for z in _fzs for w_ in z['ramp_ws']]
        _fev = [(f.get('pos_frac'), abs(f['magnitude'])) for w in pool_pre
                for f in w.get('_fault_info', []) if f.get('pos_frac') is not None]
        _prof = ''
        if _fev:
            _fp = np.array([e[0] for e in _fev]); _fm = np.array([e[1] for e in _fev])
            _prof = (' | mean|throw| by pos tercile: '
                     + '/'.join(f'{_fm[m].mean():.1f}' for m in
                                (_fp <= 0.33, (_fp > 0.33) & (_fp <= 0.67), _fp > 0.67)) + ' ft')
        print(f'  [FZ] pairs {_npair}/{len(pool_pre)} wells ({100*_npair/max(len(pool_pre),1):.1f}%) | '
              f'pair-dropped {_ndropp} | events ramped/sharp {_nram}/{_nshp}'
              + (f' | ramp w mean {np.mean(_ws):.0f}' if _ws else '') + _prof)
    print(f'  Triple joint: {n_all_three}/{len(pool_pre)} ({100*n_all_three/len(pool_pre):.1f}%) have ALL THREE')
    print(f'  Any: {n_any}/{len(pool_pre)} ({100*n_any/len(pool_pre):.1f}%) have at least one variation, '
          f'{n_none}/{len(pool_pre)} ({100*n_none/len(pool_pre):.1f}%) baseline-like')

print(f'\nTotal synth wells generated: {sum(len(p) for p in synth_pools.values()) + sum(len(p) for p in synth_pools_pretrain.values())}')


In [ ]:
# Mandatory leakage audit — hard assert on BOTH pools before proceeding to training
print('=== Leakage audit (both pools, all folds) ===\n')
audit_pass = True
for fold in ALL_FOLDS:
    val_set = set(fold_splits[fold]['val'])
    train_set = set(fold_splits[fold]['train'])
    # Pretrain pool
    pool_p = synth_pools_pretrain[fold]
    seed_tws_p = [s['_seed_tw_wid'] for s in pool_p]
    val_leaked_p = [w for w in seed_tws_p if w in val_set]
    not_in_train_p = [w for w in seed_tws_p if w not in train_set]
    print(f'Fold {fold}:')

    print(f'  Pretrain pool ({len(pool_p)}): {len(set(seed_tws_p))} unique TW seeds')
    print(f'    val wells leaked: {len(val_leaked_p)} ({"FAIL" if val_leaked_p else "OK"})')
    print(f'    non-train TW seeds: {len(not_in_train_p)} ({"FAIL" if not_in_train_p else "OK"})')
    if val_leaked_p or not_in_train_p or val_leaked_p or not_in_train_p:
        audit_pass = False
print(f'\nOverall audit: {"PASS" if audit_pass else "FAIL"}')
assert audit_pass, 'Leakage audit failed; halt'


In [ ]:
# === match_corr sanity gate (on the BUILT production pool) — confirm reduced_B landed at the target ===
# Measures match_corr (HW GR vs TW GR @ truth TVT) on a sample of the actual pretrain pool and on real,
# so the run log records that this arm's synth matching signal really sits near real (~0.64) before training.
import numpy as _np, pandas as _pd
def _mc(hw_gr, hw_tvt, tw_tvt, tw_gr):
    t = _np.interp(hw_tvt, tw_tvt, tw_gr)
    ok = _np.isfinite(hw_gr) & _np.isfinite(t) & _np.isfinite(hw_tvt)
    if ok.sum() < 20: return _np.nan
    a, b = hw_gr[ok], t[ok]
    if a.std() < 1e-6 or b.std() < 1e-6: return _np.nan
    return float(_np.corrcoef(a, b)[0, 1])

_rng = _np.random.RandomState(SEED + 999)
for _f in ALL_FOLDS[:2]:           # check a couple of folds; cheap
    _pool = synth_pools_pretrain[_f]
    _idx = _rng.choice(len(_pool), size=min(300, len(_pool)), replace=False)
    _smc = [_mc(_pool[j]['GR'][_pool[j]['ps_md_idx']:], _pool[j]['TVT'][_pool[j]['ps_md_idx']:],
                _pool[j]['_tw_tvt'], _pool[j]['_tw_gr']) for j in _idx]
    _smc = [v for v in _smc if _np.isfinite(v)]
    # real target for this fold
    _rmc = []
    for _w in list(fold_splits[_f]['train'])[:120]:
        _hw = _pd.read_csv(well_hw_path(_w), usecols=['TVT','TVT_input','GR'])
        _tw = _pd.read_csv(well_tw_path(_w), usecols=['TVT','GR'])
        _o = _np.argsort(_tw['TVT'].values); _po = _hw['TVT_input'].isna().values
        _v = _mc(_hw['GR'].values[_po], _hw['TVT'].values[_po], _tw['TVT'].values[_o], _tw['GR'].values[_o])
        if _np.isfinite(_v): _rmc.append(_v)
    print(f'fold {_f}: synth match_corr = {_np.mean(_smc):.3f}  |  real target = {_np.mean(_rmc):.3f}  '
          f'(arm={CORRUPTION_MODE})')
print('-> synth should sit near real (~0.64). If baseline arm: expect ~0.27.')


## 3. Image construction (real for all 773, synth for all 5 folds)

In [ ]:
# === dZ-per-column channel helpers (FT39e) ===
def _dz_per_col(hw_z, hw_md, bin_edges, hw_mask):
    """Per-column average of dZ/dMD. Returns float32 (W,) with NaN where no data."""
    n = len(hw_z)
    if n < 2:
        return np.full(W, 0.0, dtype=np.float32)
    # uniform 1.0 ft MD step per the EDA report, but compute defensively
    dz_per_row = np.gradient(np.asarray(hw_z, dtype=np.float64),
                              np.asarray(hw_md, dtype=np.float64))
    dz = np.full(W, np.nan, dtype=np.float32)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo:
            continue
        dz[c] = float(np.mean(dz_per_row[lo:hi]))
    return dz

def _dz_channel(dz_per_col, hw_mask):
    """Build the (H, W) dZ channel — per-well z-score + clip ±3 + broadcast across H.
    Neutralize columns without HW data so they don't bias the well z-score / clip."""
    bad = (~hw_mask) | (~np.isfinite(dz_per_col))
    if (~bad).any():
        fill = float(np.nanmean(dz_per_col[~bad]))
    else:
        fill = 0.0
    dz = np.where(bad, fill, dz_per_col).astype(np.float32)
    m, s = dz.mean(), dz.std()
    dz_norm = (dz - m) / s if s > 1e-6 else np.zeros_like(dz)
    dz_norm = np.clip(dz_norm, -3.0, 3.0)
    return np.broadcast_to(dz_norm[None, :], (H, W)).copy().astype(np.float32)

def build_image_from_arrays(hw_md, hw_gr, hw_z, hw_tvt, hw_tvti, tw_tvt, tw_gr,
                            phase_h=0.0, phase_v=0.0):
    n_hw = len(hw_md)
    known_tvt = pd.Series(hw_tvti).dropna().values
    if len(known_tvt) == 0: raise ValueError('No pre-PS rows')
    depth_center = float(known_tvt[-1])
    tvt_lo = depth_center - (H/2) * BIN_SIZE
    if phase_v != 0.0:                       # sub-bin TVT grid phase (PhaseJitter)
        tvt_lo = tvt_lo + float(phase_v) * BIN_SIZE
    def tvt_to_row(t, lo=tvt_lo):
        if np.isnan(t): return -1
        r = int((t - lo) / BIN_SIZE)
        return r if 0 <= r < H else -1
    if phase_h != 0.0:                       # sub-bin MD grid phase (PhaseJitter)
        bin_edges = np.clip(np.linspace(0, n_hw, W + 1)
                            + float(phase_h) * (n_hw / W), 0, n_hw).astype(int)
    else:
        bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
    hw_gr_per_col_raw = np.full(W, np.nan, dtype=np.float32)
    hw_mask = np.zeros(W, dtype=bool)
    truth_row = np.full(W, -1, dtype=np.int32)
    hint_row = np.full(W, -1, dtype=np.int32)
    post_ps_mask = np.zeros(W, dtype=bool)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo: continue
        gr_slc = hw_gr[lo:hi]
        if np.any(~np.isnan(gr_slc)):
            hw_gr_per_col_raw[c] = np.nanmean(gr_slc)
            hw_mask[c] = True
        tvt_slc = hw_tvt[lo:hi]
        if np.any(~np.isnan(tvt_slc)): truth_row[c] = tvt_to_row(np.nanmedian(tvt_slc))
        tvti_slc = hw_tvti[lo:hi]
        is_known = ~np.isnan(tvti_slc)
        if is_known.any(): hint_row[c] = tvt_to_row(np.nanmedian(tvti_slc[is_known]))
        else: post_ps_mask[c] = True
    if np.all(np.isnan(hw_gr_per_col_raw)):
        hw_gr_per_col = np.zeros(W, dtype=np.float32)
    else:
        col_mean = np.nanmean(hw_gr_per_col_raw)
        hw_gr_per_col = hw_gr_per_col_raw.copy()
        hw_gr_per_col[np.isnan(hw_gr_per_col)] = col_mean
    ext_row_tvts = (tvt_lo - K_MAX * BIN_SIZE) + (np.arange(H + 2*K_MAX) + 0.5) * BIN_SIZE
    sort_idx = np.argsort(tw_tvt)
    tw_tvt_s = tw_tvt[sort_idx]; tw_gr_s = tw_gr[sort_idx]
    ok = ~np.isnan(tw_gr_s)
    if ok.sum() < 2:
        tw_gr_per_row_ext = np.zeros(H + 2*K_MAX, dtype=np.float32)
    else:
        tw_gr_per_row_ext = np.interp(ext_row_tvts, tw_tvt_s[ok], tw_gr_s[ok],
                                       left=tw_gr_s[ok][0], right=tw_gr_s[ok][-1]).astype(np.float32)
    tw_gr_per_row = tw_gr_per_row_ext[K_MAX : K_MAX + H]
    ch_tw_raw = np.broadcast_to(tw_gr_per_row[:, None], (H, W)).copy()
    ch_hw_raw = np.broadcast_to(hw_gr_per_col[None, :], (H, W)).copy()
    ch_hint = np.zeros((H, W), dtype=np.float32)
    for c in range(W):
        if hint_row[c] >= 0 and not post_ps_mask[c]:
            ch_hint[hint_row[c], c] = 1.0
    cost_vol = (ch_hw_raw - ch_tw_raw) ** 2
    if hw_mask.any():
        mean_cost = float(cost_vol[:, hw_mask].mean())
    else:
        mean_cost = 0.0
    for c in range(W):
        if not hw_mask[c]:
            cost_vol[:, c] = mean_cost
    ch_tw = ch_tw_raw.copy()
    m, s = ch_tw.mean(), ch_tw.std()
    if s > 1e-6: ch_tw = (ch_tw - m) / s
    ch_hw = ch_hw_raw.copy()
    m, s = ch_hw.mean(), ch_hw.std()
    if s > 1e-6: ch_hw = (ch_hw - m) / s
    cm, cs = cost_vol.mean(), cost_vol.std()
    if cs > 1e-6:
        cost_vol_norm = (cost_vol - cm) / cs
    else:
        cost_vol_norm = np.zeros_like(cost_vol)
    cost_vol_norm = np.clip(cost_vol_norm, -3.0, 3.0)
    # --- dZ-per-column channel (FT39e) — jitter-invariant (per-column scalar, no row shift) ---
    dz_per_col = _dz_per_col(hw_z, hw_md, bin_edges, hw_mask)
    ch_dz = _dz_channel(dz_per_col, hw_mask)
    image_in = np.stack([ch_tw, ch_hw, ch_hint, cost_vol_norm, ch_dz], axis=0).astype(np.float32)
    return dict(image_in=image_in, truth_row=truth_row, hint_row=hint_row,
                post_ps_mask=post_ps_mask, depth_center=depth_center,
                tvt_lo=tvt_lo, hw_mask=hw_mask, dz_per_col=dz_per_col,
                tw_gr_per_row_ext=tw_gr_per_row_ext, hw_gr_per_col=hw_gr_per_col)

def build_image_real(hw_df, tw_df):
    arrs = dict(
        hw_md=hw_df['MD'].values, hw_gr=hw_df['GR'].values,
        hw_z=hw_df['Z'].values,
        hw_tvt=hw_df['TVT'].values, hw_tvti=hw_df['TVT_input'].values,
        tw_tvt=tw_df['TVT'].values, tw_gr=tw_df['GR'].values)
    rec = build_image_from_arrays(**arrs)
    # Raw logs retained EXACTLY as passed (no dtype cast) so render_phase(rec, 0, 0)
    # reproduces the cached build bit-for-bit. Real wells only (~200 MB total);
    # synth recs never carry raws and are never phase-jittered.
    rec['_raw'] = arrs
    return rec


def render_phase(rec, phase_h=0.0, phase_v=0.0):
    """Re-render a real-well rec from its raw logs at grid phase (dh in MD bins, dv in
    TVT ft) through the IDENTICAL pipeline. Returns a fresh rec; the cached one is never
    mutated. No-op (returns rec) if raws are absent (synth)."""
    raw = rec.get('_raw')
    if raw is None:
        return rec
    out = build_image_from_arrays(phase_h=phase_h, phase_v=phase_v, **raw)
    out['well_id'] = rec.get('well_id'); out['_is_synth'] = rec.get('_is_synth', False)
    out['_raw'] = raw
    return out

def build_image_synth(synth_dict):
    return build_image_from_arrays(
        hw_md=synth_dict['MD'], hw_gr=synth_dict['GR'],
        hw_z=synth_dict['Z'],
        hw_tvt=synth_dict['TVT'], hw_tvti=synth_dict['TVT_input'],
        tw_tvt=synth_dict['_tw_tvt'], tw_gr=synth_dict['_tw_gr'])

def render_toecrop(rec, frac):
    """Crop the last `frac` of POST-PS raw rows and re-render through the standard
    pipeline. Pre-PS rows (anchors) are untouchable by construction. Vertical geometry
    is invariant (depth_center comes from the last pre-PS TVT, always kept). Returns a
    fresh rec whose _raw is the CROPPED well, so a subsequent render_phase composes on
    it. No-op (returns rec) if raws are absent or the crop rounds to zero rows."""
    raw = rec.get('_raw')
    if raw is None or frac <= 0.0:
        return rec
    tvti = np.asarray(raw['hw_tvti'])
    known = np.where(~np.isnan(tvti))[0]
    if known.size == 0:
        return rec
    k_last = int(known[-1])                       # last pre-PS row index
    n = len(tvti)
    n_post = n - 1 - k_last
    n_drop = int(frac * n_post)
    if n_drop <= 0 or n_post - n_drop < 1:
        return rec
    n_keep = n - n_drop
    cropped = {k: (np.asarray(v)[:n_keep] if k.startswith('hw_') else v)
               for k, v in raw.items()}           # TW log untouched
    out = build_image_from_arrays(**cropped)
    out['well_id'] = rec.get('well_id'); out['_is_synth'] = rec.get('_is_synth', False)
    out['_raw'] = cropped
    return out


def render_gr_aug(rec, gain, offset):
    """Affine GR calibration jitter on the HW log only: gr' = mu + gain*(gr - mu) + offset,
    mu = nanmean(hw_gr). TW untouched. Geometry (labels, pins, masks, vertical grid) is
    bit-identical by construction; per-well z-scoring makes the hw SHAPE channel exactly
    invariant to a positive affine, so the perturbation lands on the raw-difference cost
    channel. Returns a fresh rec carrying transformed raws so ToeCrop/Phase compose
    downstream. No-op (returns rec) for the identity transform or missing raws."""
    raw = rec.get('_raw')
    if raw is None or (gain == 1.0 and offset == 0.0):
        return rec
    g = np.asarray(raw['hw_gr'], dtype=np.float64)
    mu = np.nanmean(g)
    if not np.isfinite(mu):
        return rec
    raw2 = dict(raw)
    raw2['hw_gr'] = mu + gain * (g - mu) + offset
    out = build_image_from_arrays(**raw2)
    out['well_id'] = rec.get('well_id'); out['_is_synth'] = rec.get('_is_synth', False)
    out['_raw'] = raw2
    return out


# Build real images for all 773 wells (every well appears in train of 4 folds, val of 1)
print(f'Building all {len(well_ids_all)} real images...')
well_images_real = {}
for wid in tqdm(sorted(well_ids_all), desc='real'):
    hw = pd.read_csv(well_hw_path(wid))
    tw = pd.read_csv(well_tw_path(wid))
    rec = build_image_real(hw, tw)
    rec['well_id'] = wid; rec['_is_synth'] = False
    well_images_real[wid] = rec
print(f'Built {len(well_images_real)} real images')


# === PhaseRender FOUNDATION smoke test ===
def _smoke_test_phase_render():
    # (1) FOUNDATION: phase (0,0) must reproduce the cached build BIT-EXACTLY for every
    # real well — this assert licenses every render-stage aug built on this infra.
    import time
    t0 = time.time(); n = 0
    for wid in tqdm(sorted(well_images_real), desc='phase(0,0) bit-exact'):
        rec = well_images_real[wid]
        r0 = render_phase(rec, 0.0, 0.0)
        assert np.array_equal(r0['image_in'], rec['image_in']), f'image mismatch: {wid}'
        assert np.array_equal(r0['truth_row'], rec['truth_row']), f'truth mismatch: {wid}'
        assert np.array_equal(r0['hint_row'], rec['hint_row']), f'hint mismatch: {wid}'
        assert np.array_equal(r0['post_ps_mask'], rec['post_ps_mask']), f'mask mismatch: {wid}'
        n += 1
    per_ms = (time.time() - t0) / max(1, n) * 1000
    # (2) phase behaviour on one well
    rec = well_images_real[sorted(well_images_real)[0]]
    ra = render_phase(rec, 0.5, 0.5)
    assert not np.array_equal(ra['image_in'], rec['image_in']), 'phase (0.5,0.5) changed nothing'
    pins = ra['image_in'][2]
    assert set(np.unique(pins)).issubset({0.0, 1.0}) and (pins.sum(axis=0) <= 1.0 + 1e-6).all(), \
        'phase render broke one-hot pins'
    # (3) dv=1.0 must equal the integer grid shift: truth rows drop by exactly 1 in-window
    r1 = render_phase(rec, 0.0, 1.0)
    a, b = rec['truth_row'], r1['truth_row']
    ok = (a >= 1) & (b >= 0)
    assert ok.any() and np.array_equal(b[ok], a[ok] - 1), 'dv=1.0 is not the integer row shift'
    # (4) genuine re-interpolation: half-phase TW must differ from BOTH integer renders
    tw0 = rec['image_in'][0][:, 0]; twh = ra['image_in'][0][:, 0]; tw1 = r1['image_in'][0][:, 0]
    assert (not np.array_equal(twh, tw0)) and (not np.array_equal(twh, tw1)), \
        'half-phase TW equals an integer render — re-interp not happening'
    # (5) determinism
    assert np.array_equal(ra['image_in'], render_phase(rec, 0.5, 0.5)['image_in']), \
        'render_phase not deterministic'
    print(f'PhaseRender smoke: {n} wells bit-exact at (0,0) OK ({per_ms:.0f} ms/render), '
          f'one-hot pins OK, dv=1.0==integer-shift OK, half-phase re-interp OK, deterministic OK')

_smoke_test_phase_render()


# === ToeCrop render smoke test ===
def _smoke_test_toecrop_render():
    rec = well_images_real[sorted(well_images_real)[0]]
    raw = rec['_raw']; tvti = np.asarray(raw['hw_tvti'])
    k_last = int(np.where(~np.isnan(tvti))[0][-1]); n = len(tvti)
    rc = render_toecrop(rec, 0.25)
    # (1) row accounting: int(0.25 * n_post) rows dropped, all from the tail
    n_post = n - 1 - k_last
    assert len(rc['_raw']['hw_md']) == n - int(0.25 * n_post), 'row accounting wrong'
    # (2) pre-PS preserved exactly (same rows, same last-known index)
    ct = np.asarray(rc['_raw']['hw_tvti'])
    assert np.array_equal(ct[:k_last + 1], tvti[:k_last + 1]) \
       and int(np.where(~np.isnan(ct))[0][-1]) == k_last, 'pre-PS not preserved'
    # (3) vertical geometry invariant (depth_center is pre-PS => identical window)
    assert rc['tvt_lo'] == rec['tvt_lo'] and rc['depth_center'] == rec['depth_center'], \
        'vertical grid moved under ToeCrop'
    # (4) image changed; PS boundary shifts right (fewer post-PS columns); pins one-hot
    assert not np.array_equal(rc['image_in'], rec['image_in']), 'crop changed nothing'
    assert rc['post_ps_mask'].sum() < rec['post_ps_mask'].sum(), 'post-PS columns did not shrink'
    pins = rc['image_in'][2]
    assert set(np.unique(pins)).issubset({0.0, 1.0}) and (pins.sum(axis=0) <= 1 + 1e-6).all(), \
        'ToeCrop broke one-hot pins'
    # (5) no-op path returns the cached rec object
    assert render_toecrop(rec, 0.0) is rec, 'frac=0 is not a no-op'
    # (6) composition: the phase(0,0) foundation property holds on the CROPPED well too
    assert np.array_equal(render_phase(rc, 0.0, 0.0)['image_in'], rc['image_in']), \
        'phase(0,0) not bit-exact on cropped rec'
    # (7) deterministic
    assert np.array_equal(render_toecrop(rec, 0.25)['image_in'], rc['image_in']), 'not deterministic'
    print('ToeCrop render smoke: row accounting OK, pre-PS preserved OK, vertical grid invariant OK, '
          'post-PS cols shrink OK, pins one-hot OK, no-op OK, phase(0,0)-on-cropped OK, deterministic OK')

_smoke_test_toecrop_render()


# === GR_AUG render smoke test ===
def _smoke_test_gr_aug_render():
    rec = well_images_real[sorted(well_images_real)[0]]
    g0 = np.asarray(rec['_raw']['hw_gr'], dtype=np.float64)
    mu = np.nanmean(g0)
    rg = render_gr_aug(rec, 1.12, -6.0)
    g1 = np.asarray(rg['_raw']['hw_gr'])
    # (1) transform math exact; NaN mask preserved
    fin = np.isfinite(g0)
    assert np.array_equal(fin, np.isfinite(g1)), 'NaN mask changed'
    assert np.allclose(g1[fin], mu + 1.12 * (g0[fin] - mu) - 6.0), 'affine math wrong'
    # (2) geometry bit-identical: labels, pins, masks, vertical grid, TW channel
    assert np.array_equal(rg['truth_row'], rec['truth_row']) \
       and np.array_equal(rg['post_ps_mask'], rec['post_ps_mask']) \
       and rg['tvt_lo'] == rec['tvt_lo'] and rg['depth_center'] == rec['depth_center'], \
        'GR_AUG moved geometry'
    assert np.array_equal(rg['image_in'][2], rec['image_in'][2]), 'pins changed'
    assert np.array_equal(rg['image_in'][0], rec['image_in'][0]), 'TW channel changed'
    # (3) z-invariance of the hw SHAPE channel; cost channel must change
    assert np.allclose(rg['image_in'][1], rec['image_in'][1], atol=1e-5), \
        'hw shape channel not z-invariant under affine'
    assert not np.allclose(rg['image_in'][3], rec['image_in'][3]), 'cost channel unchanged'
    # (4) no-op identity; (5) deterministic; (6) composes with ToeCrop and Phase
    assert render_gr_aug(rec, 1.0, 0.0) is rec, 'identity not a no-op'
    assert np.array_equal(render_gr_aug(rec, 1.12, -6.0)['image_in'], rg['image_in']), 'not deterministic'
    rc = render_toecrop(rg, 0.2)
    assert len(rc['_raw']['hw_md']) < len(g0) and np.isfinite(np.asarray(rc['_raw']['hw_gr'])).any(), \
        'GR_AUG -> ToeCrop composition broken'
    assert np.array_equal(render_phase(rg, 0.0, 0.0)['image_in'], rg['image_in']), \
        'phase(0,0) not bit-exact on GR-augmented rec'
    print('GR_AUG render smoke: affine exact OK, geometry bit-identical OK, shape-channel '
          'z-invariance OK, cost-channel dose OK, no-op OK, deterministic OK, composition OK')

_smoke_test_gr_aug_render()



# Build synth images per fold — both pools (joint + pretrain)
well_images_synth = {fold: {} for fold in ALL_FOLDS}          # joint arm pool
well_images_synth_pretrain = {fold: {} for fold in ALL_FOLDS}  # pretrain arm pool
for fold in ALL_FOLDS:


    print(f'Building PRETRAIN synth images for fold {fold}...')
    for syn in tqdm(synth_pools_pretrain[fold], desc=f'pretrain f{fold}'):
        try:
            rec = build_image_synth(syn)
            rec['well_id'] = syn['_synth_id']; rec['_is_synth'] = True
            well_images_synth_pretrain[fold][syn['_synth_id']] = rec
        except Exception:
            pass
    print(f'  fold {fold} pretrain: {len(well_images_synth_pretrain[fold])} synth images')


In [ ]:
# ============================================================================
# GATE 1 — BINDINGS for this kernel (run IMMEDIATELY BEFORE the Gate-1 cell).
# Pins: pre-PS = TVT_input.notna(); tw.TVT is ABSOLUTE feet (no TW_PAD here);
#       well list = well_ids_all. Fills the three providers the cell resolves.
# ============================================================================
import os, numpy as np, pandas as pd
g = globals()

# 1) well-id list -----------------------------------------------------------
WELL_IDS = list(well_ids_all)                      # 773, canonical order
assert len(WELL_IDS) == 773, len(WELL_IDS)

# 2) per-well HW frame fetch -------------------------------------------------
#    Find whatever this kernel uses to load one well's HW CSV/frame. We probe a
#    few shapes; if none fit, we assert with instructions.
_hw_fetch = None
if 'well_hw_path' in g or 'well_csv_path' in g:            # a path template/dict
    _pathobj = g.get('well_hw_path', g.get('well_csv_path'))
    def _hw_fetch(wid):
        p = _pathobj[wid] if not callable(_pathobj) else _pathobj(wid)
        return pd.read_csv(p)
elif 'load_well' in g and callable(g['load_well']):
    _hw_fetch = lambda wid: g['load_well'](wid)
elif 'DATA_DIR' in g or 'HW_DIR' in g or 'TRAIN_DIR' in g:
    _dir = g.get('DATA_DIR', g.get('HW_DIR', g.get('TRAIN_DIR')))
    def _hw_fetch(wid):
        for ext in ('.csv', '.parquet'):
            fp = os.path.join(_dir, f"{wid}{ext}")
            if os.path.exists(fp):
                return pd.read_parquet(fp) if ext == '.parquet' else pd.read_csv(fp)
        raise FileNotFoundError(wid)
assert _hw_fetch is not None, (
    "Gate1-bind: could not find an HW-frame fetch. Set:\n"
    "    _hw_fetch = lambda wid: <frame for that well with TVT_input, TVT, GR>\n"
    "using whatever this kernel already uses to load one well.")

def load_prePS_hw_tvt_pairs(wid):
    df = _hw_fetch(wid)
    pre = df[df['TVT_input'].notna()]              # pre-PS = known section (ledger convention, confirmed)
    tvt = pre['TVT'].to_numpy(float)               # truth TVT on known rows (given => test-safe)
    gr  = pre['GR'].to_numpy(float)
    m = np.isfinite(tvt) & np.isfinite(gr)
    return tvt[m], gr[m]

# 3) per-well typewell fetch (ABSOLUTE TVT axis) ----------------------------
_tw_fetch = None
if 'well_tw_path' in g:
    _twp = g['well_tw_path']
    def _tw_fetch(wid):
        p = _twp[wid] if not callable(_twp) else _twp(wid)
        return pd.read_csv(p)
elif 'tw_wid' in g and callable(g.get('load_tw', None)):
    _tw_fetch = lambda wid: g['load_tw'](wid)
assert _tw_fetch is not None, (
    "Gate1-bind: could not find a per-well typewell fetch. Set:\n"
    "    _tw_fetch = lambda wid: <tw frame with TVT (absolute ft) and GR>\n"
    "using well_tw_path / tw_wid / whatever maps a well to its typewell.")

def load_typewell(wid):
    t = _tw_fetch(wid)
    tv = t['TVT'].to_numpy(float); gr = t['GR'].to_numpy(float)
    o = np.argsort(tv)                              # sort by absolute TVT; NO TW_PAD (abs axis)
    return tv[o], gr[o]

# 4) per-well OFFSET error (current-line OOF) — OPTIONAL --------------------
#    If you have the current-line OOF residuals loaded, bind offset_err here so
#    clause 2 (mechanism) fires. Otherwise leave it; the cell runs and still
#    separates A/C from B on the |delta| distribution alone.
if 'offset_err' not in g:
    _cand = next((n for n in ('per_well_offset','well_offset_err','oof_offset') if n in g), None)
    if _cand:
        offset_err = g[_cand]
    else:
        print("[bind] no offset-error provider bound -> clause 2 will be NaN "
              "(|delta| distribution read still decides A/C vs B). "
              "To enable it: offset_err = {wid: signed_post_ps_mean_residual}.")

# smoke: fit ONE well end-to-end so a bad fetch fails HERE, not 773 rows deep
_t0, _g0 = load_prePS_hw_tvt_pairs(WELL_IDS[0])
_tt, _tg = load_typewell(WELL_IDS[0])
assert _t0.size > 0 and _tt.size > 0, "empty pre-PS or typewell for well 0 — check the fetch"
print(f"[bind OK] well0={WELL_IDS[0]}  prePS_rows={_t0.size}  tw_rows={_tt.size}  "
      f"tw.TVT[{_tt[0]:.0f},{_tt[-1]:.0f}]  hw.TVT[{_t0.min():.0f},{_t0.max():.0f}]")
print("[bind OK] providers: load_prePS_hw_tvt_pairs, load_typewell"
      + (", offset_err" if 'offset_err' in g else " (offset_err NOT bound)"))
print("Now run the Gate-1 cell.")

## 4. Dataset + jitter

In [ ]:
def make_jittered_image(rec, k):
    """Render a training frame at anchor-jitter offset k. Sparse hint encoding (the locked baseline)."""
    tw_gr_per_row_new = rec['tw_gr_per_row_ext'][K_MAX + k : K_MAX + k + H]
    ch_tw_raw_new = np.broadcast_to(tw_gr_per_row_new[:, None], (H, W)).copy()
    ch_hw_raw_new = np.broadcast_to(rec['hw_gr_per_col'][None, :], (H, W)).copy()
    ch_hint_new = np.zeros((H, W), dtype=np.float32)
    hint_new = rec['hint_row'] - k
    for c in range(W):
        if not rec['post_ps_mask'][c] and 0 <= hint_new[c] < H:
            ch_hint_new[hint_new[c], c] = 1.0
    cost_vol = (ch_hw_raw_new - ch_tw_raw_new) ** 2
    hw_mask = rec['hw_mask']
    if hw_mask.any():
        mean_cost = float(cost_vol[:, hw_mask].mean())
    else:
        mean_cost = 0.0
    for c in range(W):
        if not hw_mask[c]:
            cost_vol[:, c] = mean_cost
    ch_tw_new = ch_tw_raw_new.copy()
    m, s = ch_tw_new.mean(), ch_tw_new.std()
    if s > 1e-6: ch_tw_new = (ch_tw_new - m) / s
    ch_hw_new = ch_hw_raw_new.copy()
    m, s = ch_hw_new.mean(), ch_hw_new.std()
    if s > 1e-6: ch_hw_new = (ch_hw_new - m) / s
    cm, cs = cost_vol.mean(), cost_vol.std()
    if cs > 1e-6:
        cost_vol_norm = (cost_vol - cm) / cs
    else:
        cost_vol_norm = np.zeros_like(cost_vol)
    cost_vol_norm = np.clip(cost_vol_norm, -3.0, 3.0)
    # --- dZ channel is jitter-invariant (per-column scalar; doesn't shift with row jitter k) ---
    ch_dz_new = _dz_channel(rec['dz_per_col'], hw_mask)
    image_new = np.stack([ch_tw_new, ch_hw_new, ch_hint_new, cost_vol_norm, ch_dz_new], axis=0).astype(np.float32)
    truth_new = rec['truth_row'] - k
    main_mask = rec['post_ps_mask'] & (rec['truth_row'] >= 0) & (truth_new >= 0) & (truth_new < H)
    truth_safe = np.where(main_mask, truth_new, 0).astype(np.int64)
    return image_new, truth_safe, main_mask




class TrainDataset(Dataset):
    """FT26 lock v2 — uses per-sample deterministic RNG (determinism patch v1).

    base_seed: per-run seed, passed at construction
    epoch: must be set by training loop via .set_epoch(e) before each epoch starts

    Per-sample RNG eliminates the noise from dataloader worker scheduling.
    Anchor jitter for sample idx in epoch e is determined by
    (base_seed, epoch, idx) — fully reproducible regardless of worker order.
    """
    def __init__(self, real_wids, synth_recs=None, base_seed=0,
                 h_flip_p=0.0, hint_drop_p=0.0,
                 md_warp_strength=0.0, md_warp_autocorr=128, md_warp_p=1.0,
                 shear_std_rows=0.0, shear_autocorr=128, shear_p=1.0, shear_max_rows=9,
                 bedwarp_strength=0.0, bedwarp_autocorr=64, bedwarp_p=1.0,
                 phase_jitter_p=0.0,
                 toecrop_p=0.0, toecrop_max_frac=0.30,
                 gr_aug_p=0.0, gr_aug_gain_s=0.15, gr_aug_offset_s=10.0):
        self.entries = []
        for wid in real_wids:
            self.entries.append(('real', wid))
        if synth_recs:
            for syn_id in synth_recs:
                self.entries.append(('synth', syn_id))
        self.synth_recs = synth_recs or {}
        self.base_seed = int(base_seed) & 0xFFFFFFFF
        self.epoch = 0
        # Augmentation knobs (both default to 0.0 = FT39l baseline). Wired via
        # train_one_run so pretrain / finetune can pass independent values.
        # - h_flip_p:    horizontal (MD-axis) flip probability per sample.
        # - hint_drop_p: per-pinned-column probability of zeroing the hint pin.
        # Each uses a derived RNG stream with a distinct hash constant so toggling
        # one does not perturb the other's draws (clean independent ablation).
        self.h_flip_p    = float(h_flip_p)
        self.hint_drop_p = float(hint_drop_p)
        # MD elastic warp knobs (finetune-only; strength=0 => baseline, no RNG draw).
        self.md_warp_strength = float(md_warp_strength)
        self.md_warp_autocorr = int(md_warp_autocorr)
        self.md_warp_p        = float(md_warp_p)
        # Vertical shear knobs (finetune-only; std=0 => baseline, no RNG draw).
        self.shear_std_rows = float(shear_std_rows)
        self.shear_autocorr = int(shear_autocorr)
        self.shear_p        = float(shear_p)
        self.shear_max_rows = int(shear_max_rows)
        # BedWarp knobs (finetune-only; strength=0 => baseline, no RNG draw).
        self.bedwarp_strength = float(bedwarp_strength)
        self.bedwarp_autocorr = int(bedwarp_autocorr)
        self.bedwarp_p        = float(bedwarp_p)
        # PhaseJitter knob (finetune real wells only; 0.0 => no RNG draw, bit-exact lock).
        self.phase_jitter_p   = float(phase_jitter_p)
        # ToeCrop knobs (finetune real wells only; 0.0 => no RNG draw, bit-exact).
        self.toecrop_p        = float(toecrop_p)
        self.toecrop_max_frac = float(toecrop_max_frac)
        # GR_AUG knobs (finetune real wells only; 0.0 => no RNG draw, bit-exact).
        self.gr_aug_p        = float(gr_aug_p)
        self.gr_aug_gain_s   = float(gr_aug_gain_s)
        self.gr_aug_offset_s = float(gr_aug_offset_s)

    def set_epoch(self, epoch):
        """MUST be called by training loop before each epoch's iteration."""
        self.epoch = int(epoch)

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        kind, wid = self.entries[idx]
        rec = well_images_real[wid] if kind == 'real' else self.synth_recs[wid]
        # Per-sample deterministic RNG — replaces global np.random
        rng_seed = (self.base_seed * 1_000_003 + self.epoch * 7919 + idx) & 0xFFFFFFFF
        rng = np.random.RandomState(rng_seed)
        k = rng.randint(-K_MAX, K_MAX + 1)
        # === GR_AUG (render-stage) — per-well GR calibration jitter (gain+offset, HW only) ===
        # Affine on raw hw GR around the well mean; shape channels z-invariant by construction,
        # dose lands on the cost channel. Runs FIRST in the render-stage chain
        # (GR_AUG -> ToeCrop -> Phase compose via carried raws). Own stream (hash 2166136261),
        # independent of all other streams; p=0.0 => no RandomState, bit-exact.
        if self.gr_aug_p > 0.0 and kind == 'real' and rec.get('_raw') is not None:
            ga_rng = np.random.RandomState((rng_seed * 2166136261) & 0xFFFFFFFF)
            if ga_rng.random_sample() < self.gr_aug_p:
                _gain   = 1.0 + (2.0 * ga_rng.random_sample() - 1.0) * self.gr_aug_gain_s
                _offset = (2.0 * ga_rng.random_sample() - 1.0) * self.gr_aug_offset_s
                rec = render_gr_aug(rec, _gain, _offset)
        # === ToeCrop (render-stage) — crop the toe, re-bin to 512 (global ft/bin aug) ===
        # Drops the last c ~ U(0, max_frac) of POST-PS raw rows, then re-renders; pre-PS
        # anchors always kept, vertical grid invariant. Runs BEFORE PhaseJitter: the
        # cropped rec carries cropped raws, so the phase draw re-renders the cropped well.
        # Own stream (hash 3571428571); independent of the phase stream => arming/disarming
        # either knob leaves the other's draws bit-identical (clean ablation).
        if self.toecrop_p > 0.0 and kind == 'real' and rec.get('_raw') is not None:
            tc_rng = np.random.RandomState((rng_seed * 3571428571) & 0xFFFFFFFF)
            if tc_rng.random_sample() < self.toecrop_p:    # gate: crop+re-render (else cached)
                _frac = float(tc_rng.random_sample()) * self.toecrop_max_frac
                rec = render_toecrop(rec, _frac)
        # === PhaseJitter (render-stage) — runs BEFORE the whole image-space stack ===
        # Re-render this sample from RAW logs at a sub-bin grid phase: dh ~ U(0,1) MD bins,
        # dv ~ U(0,1) TVT ft, own stream (hash 2463534243). Anchor-jitter k then acts as the
        # integer part of a continuous vertical placement; HFlip/HintDrop/MDwarp/BedWarp
        # apply unchanged downstream. Real wells only (synth recs carry no raws).
        if self.phase_jitter_p > 0.0 and kind == 'real' and rec.get('_raw') is not None:
            ph_rng = np.random.RandomState((rng_seed * 2463534243) & 0xFFFFFFFF)
            if ph_rng.random_sample() < self.phase_jitter_p:   # gate: re-render (else cached)
                _dh = float(ph_rng.random_sample())
                _dv = float(ph_rng.random_sample())
                rec = render_phase(rec, phase_h=_dh, phase_v=_dv)
        image, truth_safe, main_mask = make_jittered_image(rec, k)
        # FT4-revisit: horizontal (MD-axis) flip. Composed transformation:
        #   1. reverse W axis on all 5 channels  (image[:, :, ::-1])
        #   2. sign-flip the dz channel (idx 4) — dZ/dMD negates under MD reversal
        #   3. reverse truth_safe and main_mask along W
        # Uses a derived RNG stream so jitter draw above is bit-identical between
        # h_flip_p=0.0 and h_flip_p=0.5 (clean ablation).
        if self.h_flip_p > 0.0:
            flip_rng = np.random.RandomState((rng_seed * 3266489917) & 0xFFFFFFFF)
            if flip_rng.random_sample() < self.h_flip_p:
                image = image[:, :, ::-1].copy()
                image[4] *= -1                       # dz sign-flip under MD reversal
                truth_safe = truth_safe[::-1].copy()
                main_mask  = main_mask[::-1].copy()
        # === HintDrop knob: zero hint pins on a random subset of pinned columns ===
        # Applied AFTER any HFlip — operates on the canonical channel layout regardless
        # of whether the image was flipped. Uses a separate derived RNG stream so the
        # HFlip and jitter draws above are unperturbed (clean independent ablation).
        # `has_pin` restricts the per-column draw to columns that actually carry a pin,
        # so HINT_DROP_P is interpreted as 'fraction of available pins to zero per sample'
        # rather than 'fraction of columns to zero' (the latter is W-dependent).
        if self.hint_drop_p > 0.0:
            hint_rng = np.random.RandomState((rng_seed * 2654435761) & 0xFFFFFFFF)
            has_pin = (image[2] > 0).any(axis=0)            # (W,) bool
            drop_decisions = hint_rng.random_sample(W) < self.hint_drop_p
            cols_to_drop = has_pin & drop_decisions
            if cols_to_drop.any():
                image[2, :, cols_to_drop] = 0.0
        # === MD-axis elastic warp: shared nearest-neighbor column reindex (apparent-dip-rate aug) ===
        # Applied AFTER HFlip/hint-drop on the canonical layout. ONE src[c] map reindexes every
        # channel + truth + mask identically, so output col c == source col src[c] across all of
        # them (cost/hint/truth stay aligned by construction). Own RNG stream so jitter/HFlip/
        # hint-drop draws above are bit-identical whether this is on or off; strength=0 skips it.
        if self.md_warp_strength > 0.0:
            warp_rng = np.random.RandomState((rng_seed * 1234567891) & 0xFFFFFFFF)
            if warp_rng.random_sample() < self.md_warp_p:   # gate: warp this sample (else canonical)
                g = make_correlated_noise(W, autocorr_length=self.md_warp_autocorr,
                                          target_std=self.md_warp_strength, rng=warp_rng)
                cum = np.cumsum(np.exp(g)); cum -= cum[0]
                if cum[-1] > 0:
                    src = np.clip(np.round(cum / cum[-1] * (W - 1)).astype(np.int64), 0, W - 1)
                    image = image[:, :, src]          # reindex all channels along W (consistent)
                    truth_safe = truth_safe[src]       # truth row travels with its column
                    main_mask = main_mask[src]         # loss mask travels with its column
        # === Per-column vertical shear — smooth elastic generalization of anchor jitter ===
        # Output row r of column c takes source row r - d[c] (edge-replicate at the window
        # boundary, never wrap), so content moves DOWN by d[c]; truth_row[c] moves with it.
        # Self-consistent across every channel: shifted cost col = (hw[c] - tw[r-d])^2; pins
        # travel with their columns; dz (row-constant) is bit-invariant. Applied AFTER the warp
        # block — both are label-consistent geometrics and compose freely (independent gates).
        # Own RNG stream (hash 2971215073) so all other draws are bit-identical whether this is
        # on or off; std=0 skips the block entirely (no RandomState construction) => bit-exact
        # warp-locked baseline. Truth shifted out of [0, H) is masked (anchor-jitter convention).
        if self.shear_std_rows > 0.0:
            shear_rng = np.random.RandomState((rng_seed * 2971215073) & 0xFFFFFFFF)
            if shear_rng.random_sample() < self.shear_p:   # gate: shear this sample (else canonical)
                d = make_correlated_noise(W, autocorr_length=self.shear_autocorr,
                                          target_std=self.shear_std_rows, rng=shear_rng)
                d = np.clip(np.round(d), -self.shear_max_rows, self.shear_max_rows).astype(np.int64)
                if np.any(d != 0):
                    r_src = np.clip(np.arange(H)[:, None] - d[None, :], 0, H - 1)
                    image = image[:, r_src, np.arange(W)[None, :]]   # per-column vertical shift, all channels
                    shifted = truth_safe + d                          # truth travels with its column content
                    inside = main_mask & (shifted >= 0) & (shifted < H)
                    truth_safe = np.where(inside, shifted, 0).astype(truth_safe.dtype)
                    main_mask = inside
        # === BedWarp — smooth monotone TVT-axis (row) deformation: bed thickness jitter ===
        # Cost is POINTWISE in tw(row) — cost[r,c] = (hw(c)-tw(r))^2 — so a monotone row
        # reindex of the rendered image IS the stratigraphy-space deformation exactly: ch_tw
        # rows become the deformed TW log, cost transforms pointwise, ch_hw/ch_dz are
        # row-constant (bit-invariant), hint pins ride their rows. truth_row maps through the
        # INVERSE warp (continuous interp, rounded). Endpoint-pinned (src[0]=0, src[-1]=H-1)
        # so truth stays in-window: main_mask untouched (unlike shear). Own RNG stream
        # (hash 3203431781) so all other draws are bit-identical whether this is on or off;
        # strength=0 skips the block entirely (no RandomState) => bit-exact warp-locked lock.
        if self.bedwarp_strength > 0.0:
            bw_rng = np.random.RandomState((rng_seed * 3203431781) & 0xFFFFFFFF)
            if bw_rng.random_sample() < self.bedwarp_p:   # gate: deform this sample (else canonical)
                gh = make_correlated_noise(H, autocorr_length=self.bedwarp_autocorr,
                                           target_std=self.bedwarp_strength, rng=bw_rng)
                cumh = np.cumsum(np.exp(gh)); cumh -= cumh[0]
                if cumh[-1] > 0:
                    src_f = cumh / cumh[-1] * (H - 1)
                    src_r = np.clip(np.round(src_f).astype(np.int64), 0, H - 1)
                    image = image[:, src_r, :]             # reindex all channels along H (consistent)
                    t_new = np.interp(truth_safe.astype(np.float64), src_f,
                                      np.arange(H, dtype=np.float64))
                    truth_safe = np.clip(np.round(t_new), 0, H - 1).astype(truth_safe.dtype)
        return {'image': torch.from_numpy(image).clone(),
                'truth_row': torch.from_numpy(truth_safe).long(),
                'main_mask': torch.from_numpy(main_mask).clone(),
                'well_id': wid, 'is_synth': kind == 'synth'}


class RealValDataset(Dataset):
    """Val dataset. Both arms use sparse hint here — no arm-conditional behavior needed.
    (FT21 bug class cannot apply by construction.)
    Uses cached image_in (5ch incl. dZ) from cell 12."""
    def __init__(self, well_ids):
        self.well_ids = well_ids
    def __len__(self): return len(self.well_ids)
    def __getitem__(self, idx):
        wid = self.well_ids[idx]
        rec = well_images_real[wid]
        image = rec['image_in']
        truth_row = rec['truth_row']
        post_ps = rec['post_ps_mask']
        main_mask = post_ps & (truth_row >= 0)
        truth_safe = np.where(main_mask, truth_row, 0).astype(np.int64)
        return {'image': torch.from_numpy(image).clone(),
                'truth_row': torch.from_numpy(truth_safe).long(),
                'main_mask': torch.from_numpy(main_mask).clone(),
                'well_id': wid}


# === H-flip transformation smoke test ===
# Verifies the L-R + dz-sign-flip is involutive (flip-twice = identity) and that
# a single flip actually negates dz. Catches missing transformations at load time.
def _smoke_test_hflip():
    test_wid = sorted(well_images_real)[0]
    rec = well_images_real[test_wid]
    img0, tr0, mk0 = make_jittered_image(rec, 0)
    # Single flip
    img1 = img0[:, :, ::-1].copy(); img1[4] *= -1
    tr1  = tr0[::-1].copy();        mk1  = mk0[::-1].copy()
    # Flip again — should bit-match original
    img2 = img1[:, :, ::-1].copy(); img2[4] *= -1
    tr2  = tr1[::-1].copy();        mk2  = mk1[::-1].copy()
    assert np.array_equal(img0, img2), 'H-flip image round-trip failed'
    assert np.array_equal(tr0, tr2),  'H-flip truth_safe round-trip failed'
    assert np.array_equal(mk0, mk2),  'H-flip main_mask round-trip failed'
    # Single-flip should actually negate dz vs reversed original
    assert np.allclose(img1[4], -img0[4, :, ::-1]), 'dz sign-flip not applied correctly'
    # And non-dz channels should NOT be sign-flipped
    for ch in (0, 1, 2, 3):
        assert np.allclose(img1[ch], img0[ch, :, ::-1]), f'channel {ch} unexpectedly sign-flipped'
    print('H-flip smoke test: round-trip identity OK, dz sign-flip OK, non-dz channels unchanged OK')

_smoke_test_hflip()


# === HintDrop smoke test ===
# Verifies p=0.0 is a no-op, p=1.0 drops every pinned column, and p=0.5 produces
# a binomially-plausible drop count. Catches off-by-one or aliasing bugs at load.
def _smoke_test_hint_drop():
    test_wid = sorted(well_images_real)[0]
    rec = well_images_real[test_wid]
    img0, _, _ = make_jittered_image(rec, 0)
    has_pin = (img0[2] > 0).any(axis=0)
    n_pinned = int(has_pin.sum())
    assert n_pinned > 0, 'smoke test well has no hint pins — cannot validate dropout'

    # p=0 → no drops
    img_p0 = img0.copy()
    rng = np.random.RandomState(12345)
    drop_decisions = rng.random_sample(W) < 0.0
    assert not (has_pin & drop_decisions).any(), 'p=0.0 selected columns to drop'
    assert np.array_equal(img_p0[2], img0[2]), 'p=0 should be a no-op on the hint channel'

    # p=1 → every pinned column dropped, hint channel becomes all-zero
    img_p1 = img0.copy()
    rng = np.random.RandomState(12345)
    drop_decisions = rng.random_sample(W) < 1.0
    cols = has_pin & drop_decisions
    assert cols.sum() == n_pinned, f'p=1.0 should drop all {n_pinned} pinned cols, got {int(cols.sum())}'
    img_p1[2, :, cols] = 0.0
    assert (img_p1[2] == 0).all(), 'p=1.0 + drop should zero the hint channel entirely'

    # p=0.5 → statistically plausible drop count (3-sigma binomial band)
    rng = np.random.RandomState(20260601)
    drop_decisions = rng.random_sample(W) < 0.5
    n_dropped = int((has_pin & drop_decisions).sum())
    expected   = 0.5 * n_pinned
    std_dev    = (n_pinned * 0.25) ** 0.5
    assert abs(n_dropped - expected) < 4 * std_dev, (
        f'p=0.5 dropped {n_dropped}/{n_pinned}; expected ≈ {expected:.0f} ± {4*std_dev:.0f}')

    # Independence from HFlip: a hint-drop-only pass should not touch dz or other channels
    img_check = img0.copy()
    cols_dummy = has_pin & (np.random.RandomState(7).random_sample(W) < 0.5)
    img_check[2, :, cols_dummy] = 0.0
    for ch in (0, 1, 3, 4):
        assert np.array_equal(img_check[ch], img0[ch]), f'hint-drop unexpectedly touched channel {ch}'
    print('HintDrop smoke test: p=0 no-op OK, p=1 drops all pins OK, p=0.5 statistical OK, '
          'channel isolation OK')

_smoke_test_hint_drop()


# === MD elastic warp smoke test ===
# Verifies the src map is monotonic with fixed endpoints, that every output column is an exact
# copy of its source column across all channels + truth + mask (cost/hint/truth stay aligned),
# that an identity reindex is a no-op, and that the warp is deterministic for a fixed seed.
def _smoke_test_md_warp():
    rec = well_images_real[sorted(well_images_real)[0]]
    img0, tr0, mk0 = make_jittered_image(rec, 0)
    def _src(seed):
        rng = np.random.RandomState((seed * 1234567891) & 0xFFFFFFFF)
        g = make_correlated_noise(W, autocorr_length=128, target_std=0.15, rng=rng)
        cum = np.cumsum(np.exp(g)); cum -= cum[0]
        return np.clip(np.round(cum / cum[-1] * (W - 1)).astype(np.int64), 0, W - 1)
    src = _src(13579)
    assert src[0] == 0 and src[-1] == W - 1, 'warp endpoints not fixed'
    assert np.all(np.diff(src) >= 0), 'warp src not monotonic'
    imgw, trw, mkw = img0[:, :, src], tr0[src], mk0[src]
    c = W // 3
    assert np.array_equal(imgw[:, :, c], img0[:, :, src[c]]), 'output col not an exact source copy'
    assert trw[c] == tr0[src[c]] and mkw[c] == mk0[src[c]], 'truth/mask not aligned with image'
    assert np.array_equal(imgw[3], img0[3][:, src]) and np.array_equal(imgw[1], img0[1][:, src]), \
        'cost/hw not reindexed identically'
    assert np.array_equal(img0[:, :, np.arange(W)], img0), 'identity reindex changed image'
    assert np.array_equal(src, _src(13579)), 'warp not deterministic for fixed seed'
    print(f'MD-warp smoke test: monotonic+endpoints OK, per-col consistency OK, cost/hw aligned OK, '
          f'identity no-op OK, deterministic OK ({int((np.diff(src)==0).sum())} dup / '
          f'{int((np.diff(src)>1).sum())} skip cols at strength 0.15)')

_smoke_test_md_warp()


def _smoke_test_shear():
    rec = well_images_real[sorted(well_images_real)[0]]
    img0, tr0, mk0 = make_jittered_image(rec, 0)
    def _d(seed):
        rng = np.random.RandomState((seed * 2971215073) & 0xFFFFFFFF)
        rng.random_sample()  # consume the gate draw exactly as the dataset path does
        d = make_correlated_noise(W, autocorr_length=128, target_std=4.0, rng=rng)
        return np.clip(np.round(d), -9, 9).astype(np.int64)
    d = _d(11223)
    assert np.abs(d).max() <= 9, 'shear field exceeds clip'
    r_src = np.clip(np.arange(H)[:, None] - d[None, :], 0, H - 1)
    imgs = img0[:, r_src, np.arange(W)[None, :]]
    c = W // 3
    src_r = np.clip(np.arange(H) - d[c], 0, H - 1)
    assert np.array_equal(imgs[:, :, c], img0[:, src_r, c]), 'column not shifted consistently'
    assert np.array_equal(imgs[4], img0[4]), 'dz channel changed under shear (must be bit-invariant)'
    t = int(tr0[c])
    if mk0[c] and 9 <= t < H - 9 and 0 <= t + d[c] < H:
        assert np.array_equal(imgs[:4, t + d[c], c], img0[:4, t, c]), 'truth-row content did not travel with shift'
    z = np.zeros(W, dtype=np.int64)
    assert np.array_equal(img0[:, np.clip(np.arange(H)[:, None] - z[None, :], 0, H - 1),
                               np.arange(W)[None, :]], img0), 'identity shear changed image'
    assert np.array_equal(d, _d(11223)), 'shear not deterministic for fixed seed'
    assert not np.array_equal(d, _d(11224)), 'shear field identical across different seeds'
    print(f'shear smoke test: per-col shift OK, dz invariant OK, truth-travel OK, identity OK, '
          f'deterministic OK (|d| max {int(np.abs(d).max())}, realized std {d.std():.2f} rows)')

_smoke_test_shear()


# === BedWarp smoke test ===
def _smoke_test_bedwarp():
    wid = sorted(well_images_real)[0]
    rec = well_images_real[wid]
    img0, tr0, mk0 = make_jittered_image(rec, 0)
    rng = np.random.RandomState(24680)
    st = BEDWARP_STRENGTH if BEDWARP_STRENGTH > 0 else 0.12
    gh = make_correlated_noise(H, autocorr_length=BEDWARP_AUTOCORR, target_std=st, rng=rng)
    cumh = np.cumsum(np.exp(gh)); cumh -= cumh[0]
    src_f = cumh / cumh[-1] * (H - 1)
    src_r = np.clip(np.round(src_f).astype(np.int64), 0, H - 1)
    assert src_r[0] == 0 and src_r[-1] == H - 1 and np.all(np.diff(src_r) >= 0), \
        'row map not monotone / endpoint-pinned'
    imgw = img0[:, src_r, :]
    assert np.array_equal(imgw[1], img0[1]) and np.array_equal(imgw[4], img0[4]), \
        'row-constant channel (hw/dz) moved under row reindex'
    t_new = np.clip(np.round(np.interp(tr0.astype(np.float64), src_f, np.arange(H))), 0, H - 1)
    # pins: surviving pins must sit within 1 row of the inverse-mapped position
    pr, pc = np.where(img0[2] > 0)
    survived, consistent = 0, 0
    for r, c in zip(pr, pc):
        nr = np.where(imgw[2][:, c] > 0)[0]
        if nr.size:
            survived += 1
            if np.min(np.abs(nr - np.interp(r, src_f, np.arange(H)))) <= 1.0:
                consistent += 1
    assert pr.size == 0 or consistent == survived, 'surviving pin off its inverse-mapped row'
    disp = np.abs(t_new[mk0] - tr0[mk0].astype(np.float64)) if mk0.any() else np.zeros(1)
    # determinism + gate rate on the dataset stream
    fires = 0
    for t in range(400):
        sd = (1000003 * 999 + 7919 + t) & 0xFFFFFFFF
        if np.random.RandomState((sd * 3203431781) & 0xFFFFFFFF).random_sample() < BEDWARP_P:
            fires += 1
    assert BEDWARP_STRENGTH == 0 or abs(fires / 400 - BEDWARP_P) < 0.07, f'gate rate {fires/400:.3f}'
    print(f'BedWarp smoke: monotone+pinned OK, hw/dz invariant OK, pins {survived}/{pr.size} survive '
          f'(drops = rounding, HintDrop-like) all consistent OK, |dTruth| med={np.median(disp):.1f} '
          f'max={disp.max():.1f} rows, gate {fires}/400 ~ {BEDWARP_P} OK')

_smoke_test_bedwarp()


# === PhaseJitter dataset-gate smoke test ===
def _smoke_test_phase_gate():
    wids = sorted(well_images_real)[:2]
    ds0 = TrainDataset(wids, base_seed=4242)
    dsP = TrainDataset(wids, base_seed=4242, phase_jitter_p=0.0)
    ds0.set_epoch(0); dsP.set_epoch(0)
    a, b = ds0[0], dsP[0]
    assert torch.equal(a['image'], b['image']) and torch.equal(a['truth_row'], b['truth_row']), \
        'phase_jitter_p=0.0 is not bit-exact'
    ds1 = TrainDataset(wids, base_seed=4242, phase_jitter_p=1.0)
    ds1.set_epoch(0)
    c1, c2 = ds1[0], ds1[0]
    assert torch.equal(c1['image'], c2['image']), 'phase-jittered sample not deterministic'
    assert not torch.equal(c1['image'], a['image']), 'phase_jitter_p=1.0 produced canonical sample'
    fires = sum(np.random.RandomState((((4242 * 1_000_003 + 7919 + t) & 0xFFFFFFFF)
                * 2463534243) & 0xFFFFFFFF).random_sample() < 0.5 for t in range(400))
    assert abs(fires / 400 - 0.5) < 0.07, f'gate rate {fires/400:.3f}'
    print(f'PhaseJitter gate smoke: p=0 bit-exact OK, p=1 deterministic+non-canonical OK, '
          f'gate {fires}/400 ~ 0.5 OK')

_smoke_test_phase_gate()


# === ToeCrop dataset-gate smoke test ===
def _smoke_test_toecrop_gate():
    wids = sorted(well_images_real)[:2]
    base = TrainDataset(wids, base_seed=515)
    off  = TrainDataset(wids, base_seed=515, toecrop_p=0.0)
    base.set_epoch(0); off.set_epoch(0)
    a, b = base[1], off[1]
    assert torch.equal(a['image'], b['image']) and torch.equal(a['truth_row'], b['truth_row']), \
        'toecrop_p=0.0 is not bit-exact'
    on = TrainDataset(wids, base_seed=515, toecrop_p=1.0, toecrop_max_frac=0.30)
    on.set_epoch(0)
    c1, c2 = on[1], on[1]
    assert torch.equal(c1['image'], c2['image']), 'toecropped sample not deterministic'
    # frac ~ U(0,0.3): a single draw can round to a 0-row crop, so require a change on ANY
    # of the first samples rather than a specific index
    assert any(not torch.equal(on[i]['image'], base[i]['image']) for i in range(len(wids))), \
        'toecrop_p=1.0 never produced a cropped sample'
    fires = sum(np.random.RandomState((((515 * 1_000_003 + 7919 + t) & 0xFFFFFFFF)
                * 3571428571) & 0xFFFFFFFF).random_sample() < 0.5 for t in range(400))
    assert abs(fires / 400 - 0.5) < 0.07, f'gate rate {fires/400:.3f}'
    print(f'ToeCrop gate smoke: p=0 bit-exact OK, p=1 deterministic+cropping OK, gate {fires}/400 ~ 0.5 OK')

_smoke_test_toecrop_gate()


# === GR_AUG dataset-gate smoke test ===
def _smoke_test_gr_aug_gate():
    wids = sorted(well_images_real)[:2]
    base = TrainDataset(wids, base_seed=515)
    off  = TrainDataset(wids, base_seed=515, gr_aug_p=0.0)
    base.set_epoch(0); off.set_epoch(0)
    a, b = base[1], off[1]
    assert torch.equal(a['image'], b['image']), 'gr_aug_p=0.0 is not bit-exact'
    on = TrainDataset(wids, base_seed=515, gr_aug_p=1.0)
    on.set_epoch(0)
    c1, c2 = on[1], on[1]
    assert torch.equal(c1['image'], c2['image']), 'gr-augmented sample not deterministic'
    assert any(not torch.equal(on[i]['image'], base[i]['image']) for i in range(len(wids))), \
        'gr_aug_p=1.0 never changed a sample'
    assert torch.equal(c1['truth_row'], a['truth_row']), 'GR_AUG moved labels'
    fires = sum(np.random.RandomState((((515 * 1_000_003 + 7919 + t) & 0xFFFFFFFF)
                * 2166136261) & 0xFFFFFFFF).random_sample() < 0.5 for t in range(400))
    assert abs(fires / 400 - 0.5) < 0.07, f'gate rate {fires/400:.3f}'
    print(f'GR_AUG gate smoke: p=0 bit-exact OK, p=1 deterministic+active OK, labels fixed OK, '
          f'gate {fires}/400 ~ 0.5 OK')

_smoke_test_gr_aug_gate()




# === Dataset shape smoke — baseline 5-channel path ===
def _smoke_test_dataset():
    wid = well_ids_all[0]
    ds = TrainDataset([wid], base_seed=1234, h_flip_p=0.5, hint_drop_p=0.3)
    b = ds[0]
    assert b['image'].shape[0] == IN_CH_BASE == 5, f"expected 5 input channels, got {b['image'].shape[0]}"
    assert set(b.keys()) == {'image', 'truth_row', 'main_mask', 'well_id', 'is_synth'}, b.keys()
    a = ds[0]['image'].numpy(); c = ds[0]['image'].numpy()
    assert np.array_equal(a, c), 'dataset non-deterministic for fixed seed/epoch'
    print(f'Dataset smoke: IN_CH={b["image"].shape[0]}, keys OK, deterministic OK')
_smoke_test_dataset()


## 5. Model + loss

In [ ]:
def _apply_drop_connect(unet, rate):
    """Override SMP native-EfficientNet stochastic-depth (drop_connect) rate in place.
    rate=None -> untouched (bit-exact). No-op (with warning) for encoders that don't expose it
    (resnet/timm 'tu-' encoders set drop_path_rate at build, not post-hoc)."""
    if rate is None:
        return
    enc = unet.encoder
    if hasattr(enc, '_drop_connect_rate'):
        enc._drop_connect_rate = float(rate)
    else:
        import warnings
        warnings.warn(f"{type(enc).__name__} has no _drop_connect_rate; "
                      f"DROP_CONNECT_RATE={rate} ignored (native smp efficientnet-bX only).")




class UNetGeneric(nn.Module):
    def __init__(self, encoder_name='resnet34', in_ch=5):
        super().__init__()
        self.smp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                            in_channels=in_ch, classes=1,
                            decoder_channels=DECODER_CHANNELS,
                            decoder_interpolation=DECODER_INTERP)
        _apply_drop_connect(self.smp, DROP_CONNECT_RATE)
    def forward(self, x):
        logits = self.smp(x).squeeze(1)
        return logits

def _assert_droppath_active(unet, rate, tag):
    """Fail LOUD if drop_path_rate was requested but the backbone built NO active
    stochastic-depth layers. timm HRNet swallows drop_path_rate via **kwargs->cfg.update
    and applies nothing (silent no-op); efficientnet/resnet/convnext/regnet honor it."""
    if not rate:
        return
    from timm.layers import DropPath
    live = [getattr(m, "drop_prob", 0.0) for m in unet.modules() if isinstance(m, DropPath)]
    live = [p for p in live if p and p > 0.0]
    assert live, (
        f"drop_path_rate={rate} requested but {tag} built 0 active DropPath layers -> "
        f"this backbone IGNORES it (silent no-op). Wire stochastic depth explicitly, or use "
        f"a backbone that honors drop_path (efficientnet/resnet/convnext/regnet).")


class PosteriorMDDilation(nn.Module):
    """Residual MD-only dilated 1-D stack on the stage-1 posterior plane. post1 is (B, H, W): treat the H
    rows as the CHANNEL dim and MD (W) as the 1-D conv axis, so dilations grow the MD receptive field
    (~+-31 cols at default rates) WITHOUT ever convolving across the row (TVT) axis -> long-range trajectory
    context at full vertical resolution. proj_in/out bottleneck rows->embed->rows. The output projection is
    ZERO-initialised so the module returns the posterior unchanged at init (bit-identical handoff), then
    learns a de-drift / consistency correction as a residual."""
    def __init__(self, n_rows=H, embed=POST_DIL_EMBED, dilations=POST_DIL_RATES, k=3):
        super().__init__()
        self.proj_in = nn.Conv1d(n_rows, embed, kernel_size=1)            # rows -> embed (per MD column)
        trunk = []
        for d in dilations:
            trunk += [nn.Conv1d(embed, embed, k, dilation=d, padding=((k - 1) // 2) * d), nn.GELU()]
        self.trunk = nn.Sequential(*trunk)
        self.proj_out = nn.Conv1d(embed, n_rows, kernel_size=1)           # embed -> rows
        nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)   # zero-init -> identity at start
    def forward(self, post1):                      # (B, H, W) row-posterior, H rows as channels
        z = self.proj_in(post1)                    # (B, embed, W)
        z = self.trunk(z)                          # dilated MD context along the well
        delta = self.proj_out(z)                   # (B, H, W) correction in posterior space
        return post1 + delta                       # residual; zero-init -> starts == post1

def _freeze_posterior_dilation(model):
    nfrozen = 0
    for nm, p in model.named_parameters():
        if "post_md_dilation" in nm:
            p.requires_grad_(False); nfrozen += p.numel()
    if nfrozen: print(f"  [POST-DIL] froze {nfrozen/1e6:.3f}M synth-pretrained posterior-dilation params for finetune.")
    return model


def _add_multiscale_post_injection(unet, n_planes=1):
    """Re-supply the stage-1 row-posterior to EVERY stage-2 decoder block (not only the stem). Each
    decoder block's conv1 first-conv is WIDENED by n_planes input channels (the extra weights ZERO-INIT,
    so the block starts bit-identical to baseline) and its forward is wrapped to AREA-pool the carried
    posterior to that block's output resolution and concat it onto conv1's input. Area-pool (not bilinear)
    marginalises row mass over each pooled bin -> the distribution survives at every scale. The posterior
    is read from unet._ms_post, set by StackedUNet.forward before the stage-2 pass (the SAME already-
    augmented post1 that enters the stem -> row coordinates stay consistent with the truth/image transforms
    by construction). smp decoder conv1 is Conv2dReLU == Sequential[Conv2d, BN, ReLU]; we swap conv1[0]."""
    unet._ms_post = None
    n_aug = 0
    for b in unet.decoder.blocks:
        old = b.conv1[0]
        assert isinstance(old, nn.Conv2d), 'unexpected smp decoder conv1 structure (expected Conv2d at conv1[0])'
        new = nn.Conv2d(old.in_channels + n_planes, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=(old.bias is not None))
        with torch.no_grad():
            nn.init.zeros_(new.weight)
            new.weight[:, :old.in_channels].copy_(old.weight)     # copy baseline weights; extension stays 0
            if old.bias is not None:
                new.bias.copy_(old.bias)
        b.conv1[0] = new                                          # registered under the SAME state_dict key (wider)
        n_aug += 1
        def _patched(feature_map, target_height, target_width, skip_connection=None, _b=b, _unet=unet):
            fm = F.interpolate(feature_map, size=(target_height, target_width), mode=_b.interpolation_mode)
            if skip_connection is not None:
                fm = torch.cat([fm, skip_connection], dim=1)
                fm = _b.attention1(fm)
            post = _unet._ms_post                                 # (B, n_planes, H, W) carried row-posterior
            post_s = F.interpolate(post, size=(target_height, target_width), mode='area')   # marginalise mass
            fm = torch.cat([fm, post_s], dim=1)                   # supply the posterior at THIS scale
            fm = _b.conv1(fm)
            fm = _b.conv2(fm)
            fm = _b.attention2(fm)
            return fm
        b.forward = _patched
    print(f'  [MS-INJECT] widened {n_aug} stage-2 decoder block(s) by {n_planes} posterior plane(s) '
          f'(zero-init -> starts == baseline; area-pooled posterior per scale).')
    return unet


def _load_reuse_into_model(model, state):
    """Load a (possibly baseline-architecture) pretrain state into `model`, adapting decoder conv1
    weights WIDENED by multi-scale injection: copy the checkpoint's narrower weight into the leading
    input channels and keep the model's zero-init extension. Equal-shape keys load exactly (== strict).
    A genuinely incompatible shape (not a pure input-channel widening) fails LOUD. Returns
    (n_shape_adapted, missing_keys, unexpected_keys)."""
    msd = model.state_dict()
    merged = dict(msd)                                            # model's own values -> all keys present
    adapted = 0
    for k, sv in state.items():
        if k not in msd:
            continue
        mv = msd[k]
        if sv.shape == mv.shape:
            merged[k] = sv
        elif (sv.dim() == mv.dim() and sv.dim() >= 2 and sv.shape[0] == mv.shape[0]
              and sv.shape[1] < mv.shape[1] and sv.shape[2:] == mv.shape[2:]):
            nw = mv.clone(); nw[:, :sv.shape[1]] = sv             # copy overlap; keep zero extension
            merged[k] = nw; adapted += 1
        else:
            raise RuntimeError(f'reuse load: incompatible shape for {k}: ckpt {tuple(sv.shape)} '
                               f'vs model {tuple(mv.shape)} (not an input-channel widening).')
    missing    = [k for k in msd if k not in state]
    unexpected = [k for k in state if k not in msd]
    model.load_state_dict(merged, strict=True)
    return adapted, missing, unexpected


class StackedUNet(nn.Module):
    def __init__(self, encoder_name='resnet34', in_ch=5, detach_handoff=False):
        super().__init__()
        self.detach_handoff = detach_handoff
        self.stage1 = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                               in_channels=in_ch, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=DECODER_INTERP,
                               drop_path_rate=DROP_PATH_RATE)
        self.stage2 = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                               in_channels=in_ch + 1, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=DECODER_INTERP,
                               drop_path_rate=DROP_PATH_RATE)
        _apply_drop_connect(self.stage1, DROP_CONNECT_RATE)   # set on BOTH encoders
        _apply_drop_connect(self.stage2, DROP_CONNECT_RATE)
        _assert_droppath_active(self.stage1, DROP_PATH_RATE, f"{encoder_name} stage1")
        _assert_droppath_active(self.stage2, DROP_PATH_RATE, f"{encoder_name} stage2")
        if POSTERIOR_DILATION:
            self.post_md_dilation = PosteriorMDDilation()
        if MULTISCALE_POST_INJECT:
            _add_multiscale_post_injection(self.stage2, n_planes=1)

    def forward(self, x, return_aux=False):
        logits1 = self.stage1(x).squeeze(1)                 # (B, H, W)
        handoff = logits1.detach() if self.detach_handoff else logits1
        post1   = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)  # (B,H,W) row-posterior; /T softens the handoff (train+eval)
        if POSTERIOR_DILATION:
            post1 = self.post_md_dilation(post1)            # residual MD-consistency pass (zero-init: starts == post1)
        if MULTISCALE_POST_INJECT:
            self.stage2._ms_post = post1.unsqueeze(1)       # carry the SAME post1 into every stage-2 decoder block
        x2      = torch.cat([x, post1.unsqueeze(1)], dim=1) # (B, IN_CH+1, H, W)
        logits2 = self.stage2(x2).squeeze(1)                # (B, H, W)
        if MULTISCALE_POST_INJECT:
            self.stage2._ms_post = None                     # release ref (graph already built; backward unaffected)
        if return_aux:
            return logits2, logits1                          # logits1 ATTACHED -> deep-sup CE1
        return logits2

    def _stage2_from_post(self, x, post):
        """One stage-2 pass given the input image x and a row-posterior `post` (B,H,W). Mirrors the
        post1 handling in forward() (multi-scale injection ref) so an unrolled round is identical to
        the first round except for which belief is fed. Returns stage-2 logits (B,H,W)."""
        if MULTISCALE_POST_INJECT:
            self.stage2._ms_post = post.unsqueeze(1)
        x2 = torch.cat([x, post.unsqueeze(1)], dim=1)
        logits2 = self.stage2(x2).squeeze(1)
        if MULTISCALE_POST_INJECT:
            self.stage2._ms_post = None
        return logits2

    def unroll_forward(self, x, k, return_aux=False, deep_sup_rounds=False):
        """Detached random-k unrolling. Round 1 == baseline forward (stage-1 -> post1 -> stage-2).
        Rounds 2..k re-feed stage-2's OWN softmax posterior, DETACHED, so gradients flow only through
        the final (k-th) pass. k>=1; k==1 is bit-identical to forward(). `return_aux` returns the
        stage-1 logits (ATTACHED, for deep-sup CE1) exactly like forward(). deep_sup_rounds returns
        the list of intermediate stage-2 logits (rounds 2..k-1, attached-at-their-round) for optional
        per-round supervision; unused when False."""
        assert k >= 1
        logits1 = self.stage1(x).squeeze(1)
        handoff = logits1.detach() if self.detach_handoff else logits1
        post = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)   # same handoff temperature as forward()
        if POSTERIOR_DILATION:
            post = self.post_md_dilation(post)
        # round 1
        logits2 = self._stage2_from_post(x, post)
        round_logits = []
        # rounds 2..k: re-feed detached stage-2 posterior. Only the FINAL pass keeps its graph.
        for _r in range(2, k + 1):
            if deep_sup_rounds and _r < k + 1:
                round_logits.append(logits2)
            post = F.softmax(logits2.detach() / REFEED_TRAIN_T, dim=1)   # DETACH + /T softens the re-fed belief (no backprop through prior rounds)
            logits2 = self._stage2_from_post(x, post)
        if return_aux and deep_sup_rounds:
            return logits2, logits1, round_logits
        if return_aux:
            return logits2, logits1
        return logits2

def _unroll_sample_k(gen):
    """Sample (do_unroll, k) for one training step from the UNROLL_* config using a torch.Generator
    `gen` (per-step, determinism-safe). do_unroll=False -> k=1 (baseline path)."""
    if not UNROLL_ENABLE or UNROLL_P <= 0.0:
        return False, 1
    if float(torch.rand((), generator=gen)) >= UNROLL_P:
        return False, 1
    w = torch.tensor(UNROLL_K_WEIGHTS, dtype=torch.float32)
    idx = int(torch.multinomial(w, 1, generator=gen))
    return True, int(UNROLL_K_CHOICES[idx])

def unroll_train_forward(model, img, gen, return_aux=False):
    """TRAIN-time forward dispatcher. Samples (do_unroll,k); routes StackedUNet through unroll_forward
    when unrolling, else the baseline forward. Single UNet or k==1 -> baseline path (bit-identical).
    Returns the SAME shapes as model(img[,return_aux]) so the loss code is unchanged. The sampled k is
    attached as the last element ONLY when deep-sup-rounds is on (handled by caller); here we keep the
    baseline-compatible return so identity holds when OFF."""
    do_unroll, k = _unroll_sample_k(gen)
    if (not do_unroll) or (STACK_N < 2):
        return model(img, return_aux=return_aux)
    return model.unroll_forward(img, k, return_aux=return_aux,
                                deep_sup_rounds=UNROLL_DEEP_SUP_ROUNDS)   # False -> final-only (bit-identical); True -> also return rounds 2..k-1

def build_model(encoder_name=None, in_ch=None):
    """Single source of truth for model construction (training AND the eval/OOF
    reconstruction in section C). STACK_N>=2 -> StackedUNet, else the bit-exact single
    UNetGeneric. Reads the globals at call time so one STACK_N toggle switches every
    reconstruction site consistently."""
    enc = ENCODER if encoder_name is None else encoder_name
    ich = IN_CH   if in_ch       is None else in_ch
    model = (StackedUNet(encoder_name=enc, in_ch=ich, detach_handoff=STACK_DETACH)
             if STACK_N >= 2 else UNetGeneric(encoder_name=enc, in_ch=ich))
    return model

def hard_ce_loss(logits, truth_row, mask):
    B, Hh, Ww = logits.shape
    log_probs = F.log_softmax(logits, dim=1).permute(0, 2, 1).reshape(B*Ww, Hh)
    tgt = truth_row.reshape(B*Ww); msk = mask.reshape(B*Ww)
    nll = -log_probs.gather(1, tgt.unsqueeze(1)).squeeze(1)
    return (nll * msk.float()).sum() / msk.float().sum().clamp(min=1.0)


def exp_row_loss(logits, truth_row, mask, delta=8.0):
    """Integral-regression term: Huber between the softmax-EXPECTED row (the eval
    decode) and the true row, over valid columns. Same quantity the metric scores."""
    B, Hh, Ww = logits.shape
    rows = torch.arange(Hh, device=logits.device, dtype=logits.dtype)   # (H,)
    probs = F.softmax(logits, dim=1)                                    # (B,H,W)
    er = (probs * rows[None, :, None]).sum(dim=1)                       # (B,W) expected row
    e  = (er - truth_row.to(logits.dtype)).abs()                       # (B,W)
    huber = torch.where(e < delta, 0.5 * e**2, delta * (e - 0.5 * delta))
    m = mask.to(logits.dtype)
    return (huber * m).sum() / m.sum().clamp(min=1.0)

def constant_baseline_pooled(val_ids):
    all_errs_sq = []
    for wid in val_ids:
        hw = pd.read_csv(well_hw_path(wid), usecols=['TVT_input','TVT'])
        post = hw['TVT_input'].isna().values
        if post.sum() == 0: continue
        last_known = hw['TVT_input'].dropna().iloc[-1]
        true_post = hw['TVT'].values[post]
        errs = true_post - last_known
        all_errs_sq.append(errs**2)
    return float(np.sqrt(np.concatenate(all_errs_sq).mean()))

fold_const = {f: constant_baseline_pooled(fold_splits[f]['val']) for f in ALL_FOLDS}
for f in ALL_FOLDS:
    print(f'Fold {f} constant: {fold_const[f]:.3f} ft')


# === EMA state (lifted from FT17c) ===
class EMAState:
    """Exponential moving average of model parameters.
    Tracks all float-dtype state_dict entries (params + BN running stats).
    """
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
    def swap_into(self, model):
        live = {}
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in self.shadow:
                    live[k] = v.detach().clone()
                    v.copy_(self.shadow[k])
        return live
    def restore(self, model, live):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in live:
                    v.copy_(live[k])

def make_lr_lambda(warmup_epochs, total_epochs):
    """Linear warmup then cosine."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * min(1.0, max(0.0, progress))))
    return lr_lambda

def make_lr_lambda_warm_restarts(warmup_epochs, total_epochs, T_0, T_mult):
    """Linear warmup, then SGDR-style cosine annealing with warm restarts.

    After warmup, divides remaining epochs into restart cycles of length
    T_0, T_0*T_mult, T_0*T_mult^2, ...  At the start of each cycle the LR
    jumps back to its max value, then cosine-anneals to ~0 over that cycle.

    Examples (T_0=10):
      T_mult=1: cycles at ep [0-10, 10-20, 20-30, 30-40] (4 equal kicks)
      T_mult=2: cycles at ep [0-10, 10-30, 30-70]        (1 restart, then long anneal)
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        e = epoch - warmup_epochs
        cycle_start = 0; T_cur = T_0
        while cycle_start + T_cur <= e:
            cycle_start += T_cur
            T_cur *= T_mult
        progress = (e - cycle_start) / T_cur
        return 0.5 * (1.0 + np.cos(np.pi * min(1.0, max(0.0, progress))))
    return lr_lambda

def warm_restart_cycle_epochs(warmup_epochs, total_epochs, T_0, T_mult):
    """Absolute (0-based) epoch indices that BEGIN a cycle (restarts) and that
    END a cycle (low-LR settle points), matching make_lr_lambda_warm_restarts."""
    restarts, ends = set(), set()
    n = total_epochs - warmup_epochs
    cs, T = 0, T_0
    while cs < n:
        restarts.add(warmup_epochs + cs)
        ends.add(warmup_epochs + min(cs + T - 1, n - 1))
        cs += T; T *= T_mult
    return restarts, ends

# === RESIDUAL-PRIOR smoke test — pure math, no model build ===


## 5b. Teacher-input channels — REMOVED (KD stripped)

In [ ]:
# === Teacher-input channels: REMOVED ==========================================
# KD wiring stripped after §31.54 (the fold-4 KD gain was the OOF estimator edge, not
# knowledge transfer: nested 5.55 / shuffle 5.66 / LBO 5.63 all killed it vs baseline
# 5.605). This notebook is the clean 5-channel baseline; nothing here builds planes.
assert IN_CH == IN_CH_BASE == 5, f'IN_CH must be 5 with KD stripped, got IN_CH={IN_CH}'
assert N_TEACH == 0, 'noKD notebook: N_TEACH must be 0'
print(f'Teacher path removed (N_TEACH={N_TEACH}, IN_CH={IN_CH}).')


## 6. Train + eval

In [ ]:
row_idx_t = torch.arange(H, device=DEVICE, dtype=torch.float32)

def _decode_expected_err(er_b_np, rec, hw_cache):
    """Decode one well's per-column expected row -> per-row post-PS squared errors.
    Shared by stage-2 and stage-1 so both decode identically (single source of truth)."""
    n_hw, bin_edges, post, true_post, ok = hw_cache
    pred_col = rec['tvt_lo'] + (er_b_np + 0.5) * BIN_SIZE
    pred_full = np.zeros(n_hw, dtype=np.float32)
    for c in range(W): pred_full[bin_edges[c]:bin_edges[c+1]] = pred_col[c]
    return (true_post[ok] - pred_full[post][ok])**2


def _eval_iter_logits(model, img, kmax):
    """Eval-time iteration MATCHING DEPLOYMENT for the fixed-T arm: stage-1 -> handoff (softened by
    HANDOFF_TRAIN_T, same path as forward) -> stage-2, then re-feed the belief softened by
    REFEED_TRAIN_T for rounds 2..kmax (the model was TRAINED to consume that softened belief, so
    inference must feed the same). Returns {k: logits2} for k=1..kmax and the stage-1 logits. No grad.
    Uses the model's own _stage2_from_post so decode == forward at k=1."""
    logits1 = model.stage1(img).squeeze(1)
    handoff = logits1.detach() if model.detach_handoff else logits1
    post = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)
    if POSTERIOR_DILATION:
        post = model.post_md_dilation(post)
    out = {}
    logits2 = model._stage2_from_post(img, post)
    out[1] = logits2
    for k in range(2, kmax + 1):
        post = F.softmax(logits2 / REFEED_TRAIN_T, dim=1)   # refeed softened by REFEED_TRAIN_T (deploy == train)
        logits2 = model._stage2_from_post(img, post)
        out[k] = logits2
    return out, logits1

def evaluate_model(model, loader, k_list=None):
    """Eval -> softmax-expected row -> post-PS TVT-RMSE + masked val CE (stage-2 = headline).

    For the stacked model (STACK_N>=2) ALSO decode the STAGE-1 posterior in the SAME pass and
    return its standalone TVT-RMSE + CE (base vs refinement split).

    Returns (rmse2, ce2, rmse1, ce1) as before -> best-state/early-stop are UNCHANGED (they use the
    first two, which are the k=1 stage-2 headline). If k_list is given (e.g. [2,3,4]) and stacked, a
    5th return `multik` is a dict {k: (rmse_k, ce_k)} for each requested depth, using deployment-
    matched iteration (refeed_T + handoff_T applied). k_list is observational only.
    """
    model.eval()
    stacked = (STACK_N >= 2)
    want_k = sorted(set(int(k) for k in (k_list or []) if int(k) >= 2)) if stacked else []
    kmax = max([1] + want_k)
    errs2, errs1 = [], []
    errs_k = {k: [] for k in want_k}
    loss2_sum, loss1_sum, n_batches = 0.0, 0.0, 0
    lossk_sum = {k: 0.0 for k in want_k}
    with torch.no_grad():
        for batch in loader:
            img = batch['image'].to(DEVICE)
            val_tgt = batch['truth_row'].to(DEVICE); val_msk = batch['main_mask'].to(DEVICE)
            if want_k:
                kl, logits1 = _eval_iter_logits(model, img, kmax)
                logits = kl[1]                                   # k=1 stage-2 == forward() headline
            elif stacked:
                logits, logits1 = model(img, return_aux=True)
            else:
                logits, logits1 = model(img), None
            loss2_sum += hard_ce_loss(logits, val_tgt, val_msk).item()
            er2 = (F.softmax(logits, dim=1) * row_idx_t[None, :, None]).sum(dim=1)
            if stacked:
                loss1_sum += hard_ce_loss(logits1, val_tgt, val_msk).item()
                er1 = (F.softmax(logits1, dim=1) * row_idx_t[None, :, None]).sum(dim=1)
            for k in want_k:
                lossk_sum[k] += hard_ce_loss(kl[k], val_tgt, val_msk).item()
            n_batches += 1
            for b in range(img.size(0)):
                wid = batch['well_id'][b]; rec = well_images_real[wid]
                hw = pd.read_csv(well_hw_path(wid), usecols=['TVT','TVT_input'])
                n_hw = len(hw); bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
                post = hw['TVT_input'].isna().values
                true_post = hw['TVT'].values[post]; ok = ~np.isnan(true_post)
                hw_cache = (n_hw, bin_edges, post, true_post, ok)
                errs2.append(_decode_expected_err(er2[b].cpu().numpy(), rec, hw_cache))
                if stacked:
                    errs1.append(_decode_expected_err(er1[b].cpu().numpy(), rec, hw_cache))
                for k in want_k:
                    er_k = (F.softmax(kl[k], dim=1) * row_idx_t[None, :, None]).sum(dim=1)
                    errs_k[k].append(_decode_expected_err(er_k[b].cpu().numpy(), rec, hw_cache))
    rmse2 = float(np.sqrt(np.concatenate(errs2).mean()))
    ce2   = float(loss2_sum / max(n_batches, 1))
    rmse1 = float(np.sqrt(np.concatenate(errs1).mean())) if stacked else None
    ce1   = float(loss1_sum / max(n_batches, 1)) if stacked else None
    if want_k:
        multik = {k: (float(np.sqrt(np.concatenate(errs_k[k]).mean())),
                      float(lossk_sum[k] / max(n_batches, 1))) for k in want_k}
        return rmse2, ce2, rmse1, ce1, multik
    return rmse2, ce2, rmse1, ce1


def _autocast_ctx():
    # no-op context when USE_AMP=False (bit-identical fp32); autocast on the conv/matmul path when True.
    _dt = torch.bfloat16 if AMP_DTYPE == 'bfloat16' else torch.float16
    return torch.cuda.amp.autocast(enabled=USE_AMP, dtype=_dt)

def _train_one_phase(model, ema, train_loader, val_loader, opt, sched,
                     n_epochs, patience, run_label, phase_tag,
                     do_real_val=True, do_early_stop=True,
                     warmup_epochs=0,
                     ema_cycle_aware=False, restart_epochs=frozenset(),
                     cycle_end_epochs=frozenset(), ema_lr_frac=0.5, peak_lr=None):
    """Inner training loop. Returns (best_rmse, best_state, best_epoch, final_epoch, val_traj).
    If do_real_val=False (synth pretrain), val_rmse is NOT measured and best-state tracking is skipped.
    """
    best_rmse = float('inf'); best_state = None; best_epoch = -1
    val_traj = []
    val_loss_traj = []
    epochs_since = 0
    final_ep = 0
    ema_armed = not ema_cycle_aware      # legacy: always armed; cycle-aware: arm in LR tail
    # GradScaler only does work for fp16 AMP; disabled -> scale()/unscale_()/step()/update() are passthroughs,
    # so the step below is bit-identical to plain fp32 when USE_AMP=False (and for bf16, which needs no scaling).
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == 'float16'))
    # Per-step RNG for unrolling k/gate draws (finetune-only; deterministic, distinct stream).
    # Seeded from UNROLL_RNG_SEED + a phase/label salt so pretrain and finetune never share draws and
    # a resumed run is reproducible. When unrolling is OFF this generator is simply never consulted.
    _unroll_gen = torch.Generator(device='cpu')
    _unroll_gen.manual_seed((UNROLL_RNG_SEED + hash((run_label, phase_tag)) ) & 0x7FFFFFFF)
    _unroll_active = (UNROLL_ENABLE and UNROLL_P > 0.0 and phase_tag == 'finetune' and STACK_N >= 2)
    _mk_active = (phase_tag == 'finetune' and STACK_N >= 2 and bool(DEPLOY_K_LOG))
    _mk_ks = sorted(set(int(k) for k in (DEPLOY_K_LOG or []) if int(k) >= 2)) if _mk_active else []
    for epoch in range(n_epochs):
        final_ep = epoch + 1
        # Propagate epoch to dataset (per-sample RNG depends on it). Guarded so this
        # also works with datasets that don't have set_epoch.
        if hasattr(train_loader.dataset, 'set_epoch'):
            train_loader.dataset.set_epoch(epoch)
        # Reset EMA shadow at end of warmup so it starts from the live weights
        # (avoids contaminating the shadow with LR-warmup weight drift).
        # warmup_epochs == 0 makes this condition never fire (no-op).
        if ema is not None and epoch == warmup_epochs and warmup_epochs > 0:
            ema.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                          if v.dtype.is_floating_point}
            print(f'  [{run_label}|{phase_tag}] EMA shadow reset at end of warmup (ep {epoch+1})')
        # [EMA] warm-restart handling: disarm at each restart, then (re)seed the shadow from
        # live weights once this cycle's LR enters the settling band and accumulate only the
        # low-LR tail. Each cycle-end shadow is a clean within-cycle average, not a cross-basin smear.
        epoch_lr = opt.param_groups[0]['lr']           # LR this epoch actually trains at
        if ema is not None and ema_cycle_aware:
            if epoch in restart_epochs:
                ema_armed = False
            if (not ema_armed) and (peak_lr is not None) and (epoch_lr <= ema_lr_frac * peak_lr):
                ema.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                              if v.dtype.is_floating_point}
                ema_armed = True
                print(f'  [{run_label}|{phase_tag}] EMA re-seeded for cycle tail (ep {epoch+1}, lr={epoch_lr:.2e})')
        in_warmup = epoch < warmup_epochs
        # Integral-regression weight for THIS epoch (planned-schedule function; finetune-only).
        _exp_on = (phase_tag == 'finetune' and _EXP_ARMED)
        _exp_w  = exp_loss_weight_at(epoch + 1) if _exp_on else 0.0
        model.train(); sm, sm2, sm1, nb_ = 0.0, 0.0, 0.0, 0
        exp_sum = 0.0        # per-epoch mean of the RAW (unweighted) Huber term, armed runs only
        for batch in train_loader:
            img = batch['image'].to(DEVICE)
            tgt = batch['truth_row'].to(DEVICE)
            msk = batch['main_mask'].to(DEVICE)
            with _autocast_ctx():
                if STACK_N >= 2 and STACK_DEEP_SUP_WEIGHT > 0:
                    # Deep supervision: CE on the refined (stage-2) logits + lambda * CE on the
                    # stage-1 logits. Eval/inference still use stage-2 only (model(img) default).
                    if _unroll_active and UNROLL_DEEP_SUP_ROUNDS:
                        # NEW round-deep-sup path: unroll_forward returns (final, stage1, [rounds 2..k-1]).
                        # Each round CE trains that round as a one-step refiner of a DETACHED belief
                        # (its input posterior was detached at the prior round boundary) — no backprop
                        # across rounds, matching the detached-unroll design. round_logits is EMPTY when
                        # the sampled k==2 (no intermediate rounds); we average so the effective weight
                        # does not scale with the sampled k.
                        # PER-STEP arity: the dispatcher gates unrolling per step (prob UNROLL_P).
                        # Baseline-path steps return (logits, aux1); unrolled steps return
                        # (logits, aux1, round_logits). Unpack by arity, not by run-level flag.
                        _out = unroll_train_forward(model, img, _unroll_gen, return_aux=True)
                        if len(_out) == 3:
                            logits, aux1_logits, _round_logits = _out
                        else:
                            logits, aux1_logits = _out
                            _round_logits = []
                        ce2 = hard_ce_loss(logits, tgt, msk)
                        ce1 = hard_ce_loss(aux1_logits, tgt, msk)
                        loss = ce2 + STACK_DEEP_SUP_WEIGHT * ce1
                        if _round_logits:
                            _rce = torch.stack([hard_ce_loss(rl, tgt, msk) for rl in _round_logits]).mean()
                            loss = loss + UNROLL_ROUND_SUP_WEIGHT * _rce
                    else:
                        logits, aux1_logits = (unroll_train_forward(model, img, _unroll_gen, return_aux=True)
                                              if _unroll_active else model(img, return_aux=True))
                        ce2 = hard_ce_loss(logits, tgt, msk)
                        ce1 = hard_ce_loss(aux1_logits, tgt, msk)
                        loss = ce2 + STACK_DEEP_SUP_WEIGHT * ce1
                else:
                    logits = (unroll_train_forward(model, img, _unroll_gen)
                              if _unroll_active else model(img))
                    ce2 = hard_ce_loss(logits, tgt, msk)
                    ce1 = None
                    loss = ce2
                if _exp_on:
                    # Compute the raw term whenever ARMED (even at a mid-ramp w=0 epoch) so the log
                    # shows the magnitude the ramp is about to weight; add to loss only when w>0.
                    _exp_raw = exp_row_loss(logits, tgt, msk, delta=EXP_LOSS_DELTA)
                    if _exp_w > 0:
                        loss = loss + _exp_w * _exp_raw
                    exp_sum += float(_exp_raw.detach())
            if not torch.isfinite(loss):
                raise RuntimeError(f'{run_label} [{phase_tag}]: diverged ep {epoch}')
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                          # unscale before clip (no-op when AMP off)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            if ema is not None and not in_warmup and ema_armed:
                ema.update(model)
            sm += loss.item(); sm2 += ce2.item(); nb_ += 1
            if ce1 is not None: sm1 += ce1.item()
        sched.step()
        cur_lr = opt.param_groups[0]['lr']
        if do_real_val:
            # Eval with EMA weights swapped in (or live weights if EMA is off
            # or still in warmup, where the shadow is stale).
            if ema is not None and not in_warmup and ema_armed:
                live = ema.swap_into(model)
                _ev = evaluate_model(model, val_loader, k_list=_mk_ks)
                ema.restore(model, live)
            else:
                _ev = evaluate_model(model, val_loader, k_list=_mk_ks)
            if _mk_active:
                val_rmse, val_loss, val_rmse1, val_loss1, _multik = _ev
            else:
                val_rmse, val_loss, val_rmse1, val_loss1 = _ev; _multik = None
            val_traj.append(val_rmse)
            val_loss_traj.append(val_loss)
            # Selection metric: CHECKPOINT_K RMSE when armed+available (deployment-matched, from the
            # multi-k eval), else the k=1 headline. patience counter + best-state both key on this.
            if CHECKPOINT_K > 1 and _multik is not None and CHECKPOINT_K in _multik:
                sel_rmse = _multik[CHECKPOINT_K][0]
            else:
                sel_rmse = val_rmse                       # k=1 (identity path / pretrain / no multik)
            improved = sel_rmse < best_rmse
            if improved:
                best_rmse = sel_rmse
                src_state = ema.shadow if (ema is not None and not in_warmup and ema_armed) else model.state_dict()
                best_state = {k: v.detach().cpu().clone() for k, v in src_state.items()}
                for k, v in model.state_dict().items():
                    if k not in best_state:
                        best_state[k] = v.detach().cpu().clone()
                best_epoch = epoch + 1; epochs_since = 0
            else: epochs_since += 1
            marker = (' *' if improved else '') + ('' if CHECKPOINT_K == 1 else f'(sel@k{CHECKPOINT_K})' if improved else '')
            stop_marker = ' [STOP next]' if (do_early_stop and epochs_since >= patience) else ''
            # Stacked diagnostic: split train CE into stage-2-only (CE2) and stage-1 (CE1), and
            # report stage-1's STANDALONE val TVT/CE next to stage-2's (the headline). Both empty
            # when STACK_N==1 -> the single-model line is byte-identical to the lock.
            _tr = f' (CE2={sm2/nb_:.3f} CE1={sm1/nb_:.3f})' if STACK_N >= 2 else ''
            # Separate integral-regression slot (train-only): raw per-epoch mean, this epoch's
            # weight, and the weighted contribution — so raw-vs-CE ratio and ramp position are
            # both readable per epoch. Absent (byte-identical line) when the loss is unarmed.
            _tr += (f'  EXP={exp_sum/nb_:.2f}*{_exp_w:.3f}={exp_sum/nb_*_exp_w:.3f}'
                    if _exp_on and nb_ > 0 else '')
            _s1 = (f'  | s1 TVT={val_rmse1:.3f}'
                   if (STACK_N >= 2 and val_rmse1 is not None) else '')
            # k=1 is the selection metric (marked *). Deploy depths logged side by side when armed.
            _mk = ''
            if _multik:
                _mk = '  | ' + ' '.join(f'k{k}:{_multik[k][0]:.3f}({_multik[k][1]:.3f})' for k in _mk_ks)
            print(f'  [{run_label}|{phase_tag}] ep {epoch+1:2d}  lr={cur_lr:.2e}  '
                  f'L_CE={sm/nb_:.3f}{_tr}  k1_TVT={val_rmse:.3f}(CE{val_loss:.3f}){_s1}{_mk}{marker}{stop_marker}')
            if do_early_stop and epochs_since >= patience:
                print(f'  [{run_label}|{phase_tag}] EARLY STOP at ep {epoch+1} '
                      f'(patience={patience}, best={best_rmse:.3f} at ep {best_epoch})')
                break
        else:
            val_traj.append(None)
            val_loss_traj.append(None)
            print(f'  [{run_label}|{phase_tag}] ep {epoch+1:2d}  lr={cur_lr:.2e}  '
                  f'L_CE={sm/nb_:.3f}  (no real-val in {phase_tag} phase)')
    # --- synth-pretrain checkpoint (saved only for the last 2 pretrain epochs) ---
        if phase_tag == 'pretrain' and epoch >= n_epochs - 2:
             _sd = {k: v.detach().cpu() for k, v in model.state_dict().items()}
             _fn = f'pretrain_{run_label}_ep{epoch+1}.pt'
             torch.save(_sd, os.path.join(CKPT_DIR, _fn))
             if IN_COLAB:
                 torch.save(_sd, os.path.join(DRIVE_CKPT, _fn))
             print(f'  [{run_label}|{phase_tag}] saved {_fn}')
    return best_rmse, best_state, best_epoch, final_ep, val_traj, val_loss_traj


def train_one_run(fold, seed, run_label,
                   lr=1e-3, wd=1e-3, batch_size=8, patience=10,
                   reuse_pt_state=None, capture_pt=False):
    """Single deterministic recipe: synth pretrain -> real finetune (ARMS removed).
    Determinism patch v1 wired in: seed_everything + deterministic dataloader workers
    + per-sample RNG in TrainDataset (via base_seed + set_epoch).
    """
    seed_everything(seed)
    train_real_wids = fold_splits[fold]['train']
    val_wids = fold_splits[fold]['val']
    encoder_name = ENCODER
    use_pretrain = USE_PRETRAIN

    val_ds = RealValDataset(val_wids)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4,
                            pin_memory=True, worker_init_fn=seed_worker)

    # Build model + EMA once; reuse across phases for arm 2
    model = build_model(encoder_name=encoder_name, in_ch=IN_CH).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Recipe (encoder={encoder_name}, in_ch={IN_CH}, EMA={EMA_DECAY}, use_pretrain={use_pretrain})')
    print(f'  Total params: {n_params/1e6:.2f}M')
    # EMA is created fresh at finetune start (see below), NOT here. The pretrain
    # phase runs EMA-free so the finetune shadow isn't contaminated with pretrain
    # weight history.

    # === ARM 2: pretrain on synth-only, then finetune on real-only ===
    # [REUSE_PT] If a donor pretrained state is provided, SKIP the entire pretrain phase
    # and load those weights instead (see the REUSE_PT note in the config cell). The synth
    # pretrain pool is keyed to the GLOBAL seed, so the donor only fixes the shared init/basin;
    # the finetune phase below still uses THIS seed's own RNG (ft_base_seed=seed+1e6 augs +
    # per-seed loader generator + per-seed drop_path), so finetune seed-diversity is preserved
    # -- only pretrain-path + decoder-init diversity is dropped.
    if reuse_pt_state is None:
        pretrain_synth_recs = well_images_synth_pretrain[fold]
        # === Composition probe: pretrain uses H_FLIP_P_PRETRAIN, no hint dropout ===
        # HFlip during pretrain breaks the canonical-orientation bias of synth features
        # so finetune doesn't carry the entire invariance burden (motivated by the
        # 0.449 ft canonical/flipped pooled asymmetry under HFlip-finetune-only).
        # HintDrop is finetune-only: pretrain hint_drop_p is fixed at 0.0.
        pretrain_ds = TrainDataset([], pretrain_synth_recs, base_seed=seed,
                                   h_flip_p=H_FLIP_P_PRETRAIN,
                                   hint_drop_p=0.0)
        # Defensive (FT21c lesson): pretrain dataset must contain ONLY synth, no real
        assert all(e[0] == 'synth' for e in pretrain_ds.entries), 'pretrain dataset leaked real wells'
        print(f'  PRETRAIN phase ({PRETRAIN_EPOCHS} ep): {len(pretrain_synth_recs)} synth wells')
        pretrain_loader = DataLoader(pretrain_ds, batch_size=batch_size, shuffle=True,
                                     num_workers=4, pin_memory=True,
                                     worker_init_fn=seed_worker,
                                     generator=make_loader_generator(seed))
        opt_pre = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched_pre = torch.optim.lr_scheduler.LambdaLR(opt_pre, lr_lambda=make_lr_lambda(0, PRETRAIN_EPOCHS))
        # No real-val during pretrain (no synth-val either — fixed budget by design)
        _, _, _, pre_final_ep, _, _ = _train_one_phase(
            model, None, pretrain_loader, val_loader, opt_pre, sched_pre,
            n_epochs=PRETRAIN_EPOCHS, patience=patience, run_label=run_label,
            phase_tag='pretrain', do_real_val=True, do_early_stop=False,
            warmup_epochs=0,
        )
        print(f'  PRETRAIN done (EMA off). Fresh EMA + optimizer + cosine for finetune.')
        pt_state_out = ({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                        if capture_pt else None)
        pt_reused = False
    else:
        # Reuse pre-trained weights (in-memory donor OR an external per-(fold,seed) checkpoint);
        # finetune starts from here and the synth pretrain phase is skipped. Decoder conv1s that
        # multi-scale injection WIDENED are auto-adapted (baseline weights -> leading input channels,
        # zero extension kept) -> reusing a BASELINE pretrain makes the injection start as a zero-init
        # residual == baseline finetune start. Any OTHER shape mismatch fails LOUD.
        try:
            _n_adapt, _miss, _unexp = _load_reuse_into_model(model, reuse_pt_state)
        except RuntimeError as _e:
            raise RuntimeError(
                f'{run_label}: reuse/pretrained state did not match the current architecture. The '
                f'external pretrained checkpoint must be compatible (same encoder / decoder_channels / '
                f'STACK_N / POSTERIOR_DILATION; conv1 input-channel widening from injection is auto-adapted). '
                f'Original error:\n{_e}')
        if _n_adapt or _miss or _unexp:
            print(f'  [reuse] adapted {_n_adapt} widened conv1 tensor(s); missing={len(_miss)} unexpected={len(_unexp)}'
                  + ('  (injection starts as a zero-init residual on the reused baseline pretrain)' if _n_adapt else ''))
        pre_final_ep = PRETRAIN_EPOCHS
        pt_state_out = None
        pt_reused = True
        print(f'  PRETRAIN reused (skipped the {PRETRAIN_EPOCHS}-ep pretrain phase).')

    # Fresh EMA from post-pretrain weights, so the shadow starts equal to where
    # finetune begins rather than carrying any pretrain history. EMA is then
    # skipped during the first WARMUP_EPOCHS of finetune (handled inside the phase).
    ema = EMAState(model, decay=EMA_DECAY) if USE_EMA else None
    if USE_EMA:
        print(f'  EMA: on (decay={EMA_DECAY}, fresh at finetune start, '
              f'skipped during {WARMUP_EPOCHS} warmup ep)')
    else:
        print(f'  EMA: off')

    # Finetune phase: real ONLY (no synth — that is the design)
    # Offset finetune base_seed so finetune RNG is distinct from pretrain's
    # (otherwise the same idx in pretrain epoch 0 and finetune epoch 0 would
    # draw the same anchor jitter k).
    ft_base_seed = (seed + 1_000_000) & 0xFFFFFFFF
    # === Composition probe: finetune uses H_FLIP_P + HINT_DROP_P ===
    # Both augmentations active here. Independent RNG streams in TrainDataset
    # ensure toggling either knob does not perturb the other's draws.
    finetune_ds = TrainDataset(train_real_wids, synth_recs=None,
                                base_seed=ft_base_seed,
                                h_flip_p=H_FLIP_P,
                                hint_drop_p=HINT_DROP_P,
                                md_warp_strength=MD_WARP_STRENGTH,
                                md_warp_autocorr=MD_WARP_AUTOCORR,
                                md_warp_p=MD_WARP_P,
                                shear_std_rows=SHEAR_STD_ROWS,
                                shear_autocorr=SHEAR_AUTOCORR,
                                shear_p=SHEAR_P,
                                shear_max_rows=SHEAR_MAX_ROWS,
                                bedwarp_strength=BEDWARP_STRENGTH,
                                bedwarp_autocorr=BEDWARP_AUTOCORR,
                                bedwarp_p=BEDWARP_P,
                                phase_jitter_p=PHASE_JITTER_P,
                                toecrop_p=TOECROP_P,
                                toecrop_max_frac=TOECROP_MAX_FRAC,
                                gr_aug_p=GR_AUG_P,
                                gr_aug_gain_s=GR_AUG_GAIN_S,
                                gr_aug_offset_s=GR_AUG_OFFSET_S)
    # Defensive: finetune dataset must contain ONLY real, no synth
    assert all(e[0] == 'real' for e in finetune_ds.entries), 'finetune dataset leaked synth wells'
    print(f'  FINETUNE phase ({FINETUNE_EPOCHS} ep): {len(train_real_wids)} real wells (no synth)')
    finetune_loader = DataLoader(finetune_ds, batch_size=batch_size, shuffle=True,
                                 num_workers=4, pin_memory=True,
                                 worker_init_fn=seed_worker,
                                 generator=make_loader_generator(ft_base_seed))
    # Fresh optimizer (independent cosine restart), same EMA object (shadow already populated from pretrain)
    if POSTERIOR_DILATION and POSTERIOR_DILATION_FREEZE:
        _freeze_posterior_dilation(model)
    opt_ft = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=wd)
    if USE_WARM_RESTARTS:
        sched_ft = torch.optim.lr_scheduler.LambdaLR(
            opt_ft,
            lr_lambda=make_lr_lambda_warm_restarts(
                WARMUP_EPOCHS, FINETUNE_EPOCHS, WARM_RESTART_T0, WARM_RESTART_T_MULT))
        ft_patience = max(patience, WARM_RESTART_T0 + 5)   # don't early-stop within a cycle
        ft_restarts, ft_cycle_ends = warm_restart_cycle_epochs(
            WARMUP_EPOCHS, FINETUNE_EPOCHS, WARM_RESTART_T0, WARM_RESTART_T_MULT)
        ft_cycle_aware = USE_EMA            # cycle-aware EMA only meaningful when EMA is on
    else:
        sched_ft = torch.optim.lr_scheduler.LambdaLR(
            opt_ft, lr_lambda=make_lr_lambda(WARMUP_EPOCHS, FINETUNE_EPOCHS))
        ft_patience = patience
        ft_restarts, ft_cycle_ends = frozenset(), frozenset()
        ft_cycle_aware = False
    best_rmse, best_state, best_epoch, final_ep, val_traj, val_loss_traj = _train_one_phase(
        model, ema, finetune_loader, val_loader, opt_ft, sched_ft,
        n_epochs=FINETUNE_EPOCHS, patience=ft_patience, run_label=run_label,
        phase_tag='finetune', do_real_val=True, do_early_stop=True,
        warmup_epochs=WARMUP_EPOCHS,
        ema_cycle_aware=ft_cycle_aware, restart_epochs=ft_restarts,
        cycle_end_epochs=ft_cycle_ends, ema_lr_frac=0.5, peak_lr=lr,
    )
    return {'best_rmse': best_rmse, 'best_state': best_state,
            'best_epoch': best_epoch, 'final_epoch': final_ep,
            'val_traj': val_traj,
            'val_loss_traj': val_loss_traj,
            'pt_state': pt_state_out,
            'phase_info': {'mode': 'pretrain_then_finetune',
                           'pretrain_epochs': pre_final_ep,
                           'finetune_epochs': final_ep,
                           'pt_reused': pt_reused,
                           # integral-regression schedule provenance (per-seed stratifiers):
                           # w at the SELECTED checkpoint vs w when training stopped — tells you
                           # whether the chosen epoch ever lived under the full-weight regime.
                           'exp_w_start': _EXP_W0, 'exp_w_end': _EXP_W1,
                           'exp_ramp': [START_EXP_RAMP, END_EXP_RAMP] if _EXP_RAMP_ON else None,
                           'exp_w_at_best': (exp_loss_weight_at(best_epoch) if (_EXP_ARMED and best_epoch > 0) else 0.0),
                           'exp_w_at_stop': (exp_loss_weight_at(final_ep) if (_EXP_ARMED and final_ep > 0) else 0.0)}}


## §B. Full Training Loop

In [ ]:
# === Aug knobs (HFlip / HintDrop locked; MD-warp + dz-jitter set in cell 4) ===
# NOTE: commentary below is historical (FT39l-era CV numbers). Live references:
# b7/reduced_C baseline 6.789 (LEDGER S20); plain MD-warp 6.680. ARM C = dz-jitter only.
#
# H_FLIP_P_PRETRAIN: HFlip rate during synth pretrain (wired in cell 18 pretrain_ds).
#   - 0.0  = canonical-only pretrain (HFlip-finetune-only baseline, CV 8.123)
#   - 0.25 = dose-tested midpoint; safer than 0.5 if pretrain features need a
#            canonical bootstrap before learning flip invariance
#   - 0.5  = full-dose; observed finetune-ep1 ≈ 16 vs 14 baseline (regression
#            symptomatic of pretrain capacity-vs-variety bottleneck)
#
# H_FLIP_P: HFlip rate during real finetune (wired in cell 18 finetune_ds).
#   - 0.5 = HFlip-finetune-only baseline value (the PASS-≫ configuration).
#
# HINT_DROP_P: per-pinned-column hint dropout in finetune (cell 18 finetune_ds).
#   - 0.0 = no hint dropout
#   - 0.3 = HD30 reference value (NULL alone vs FT39l, motivated as composition
#           with HFlip because the mechanisms are orthogonal: HD30 regularizes
#           hint-position overfitting, HFlip forces hint-channel reading)
#
# Pretrain hint_drop_p is fixed at 0.0 by construction (cell 18). HintDrop is a
# finetune-only intervention.
#
# RNG streams: HFlip 3266489917, HintDrop 2654435761, MD-warp 1234567891,
# shear 2971215073 (DISARMED this arm), BedWarp 3203431781, PhaseJitter 2463534243, ToeCrop 3571428571, GR_AUG 2166136261
# (dz-jitter 1618033989 RETIRED with ARM C); all derived
# from the per-sample rng_seed via multiplicative-hash modulo 2^32. Distinct
# constants → toggling one knob does not perturb the other's draws → clean
# multi-axis ablation. Pretrain (base_seed=seed) and finetune (base_seed=seed+1e6)
# also have disjoint RNG streams.
H_FLIP_P_PRETRAIN = 0.0
H_FLIP_P          = 0.5
HINT_DROP_P       = 0.3

BASE_HP = {'lr': 1e-3, 'wd': 1e-3, 'patience': 15, 'batch_size': 8}

# Defensive: pretrain pool exists for all 5 folds at correct size
for fold in ALL_FOLDS:
    assert fold in well_images_synth_pretrain and len(well_images_synth_pretrain[fold]) > 0, \
        f'pretrain synth pool missing for fold {fold}'
    actual_size = len(well_images_synth_pretrain[fold])
    #assert actual_size == N_SYNTH_PRETRAIN, \
     #   f'fold {fold} pretrain pool has {actual_size} wells, expected {N_SYNTH_PRETRAIN}'

print(f'Base HP: {BASE_HP}')
print(f'EMA: decay={EMA_DECAY}, warmup={WARMUP_EPOCHS}')
print(f'Pretrain {PRETRAIN_EPOCHS} ep / Finetune {FINETUNE_EPOCHS} ep, lr=1e-3 cosine')
print(f'Synth pretrain pool: {N_SYNTH_PRETRAIN}/fold')
print(f'Recipe: encoder={ENCODER}, in_ch={IN_CH}, use_pretrain={USE_PRETRAIN} (dZ-channel probe)')
print(f'H-flip: pretrain p={H_FLIP_P_PRETRAIN}, finetune p={H_FLIP_P}')
print(f'Hint dropout (finetune only): p={HINT_DROP_P}')

# === Launch guard (run-1 trained B7 on a stale config; verify BEFORE the long run) ===
print('=' * 74)
print('LAUNCH CONFIG — verify these are intended BEFORE the run starts:')
print(f'  ENCODER         = {ENCODER}   (ENCODER_SHORT={ENCODER_SHORT})')
print(f'  STACK_N         = {STACK_N}   deep_sup_lambda = {STACK_DEEP_SUP_WEIGHT}   detach_handoff = {STACK_DETACH}')
print(f'  CORRUPTION_MODE = {CORRUPTION_MODE}   patience = {BASE_HP["patience"]}')
print(f'  EXPERIMENT_NAME = {EXPERIMENT_NAME}')
print('  REUSE_PT        = ' + str(REUSE_PT) + '   '
      + ('(pretrain ONCE/fold, reuse across seeds)' if REUSE_PT
         else '(independent pretrain per seed = original)'))
print(f'  PRETRAINED_DIR  = {PRETRAINED_DIR}'
      + ('   (per-(fold,seed) pretrain reuse -> pretrain SKIPPED where found)' if PRETRAINED_DIR else '   (none -> pretrain fresh)'))
print(f'  MULTISCALE_POST_INJECT = {MULTISCALE_POST_INJECT}'
      + ('   (post re-supplied to every stage-2 decoder block; reused baseline conv1s auto-adapted)' if MULTISCALE_POST_INJECT else '   (stem-only posterior = baseline)'))
print(f'  DECODER_INTERP  = {DECODER_INTERP}   (optimizer = AdamW; PixelShuffle / DecoyLevel / SAM removed)')
print(f'  UNROLL          = {UNROLL_ENABLE}   p={UNROLL_P} k~{UNROLL_K_CHOICES} (finetune-only, detached, final-only; eval NEVER unrolled)'
      + ('' if UNROLL_ENABLE else '   [OFF -> bit-identical baseline]'))
print(f'  TRAIN_TEMP       = refeed_T {REFEED_TRAIN_T}  handoff_T {HANDOFF_TRAIN_T}'
      + ('' if (REFEED_TRAIN_T==1.0 and HANDOFF_TRAIN_T==1.0) else '   [ARMED]'))
print('=' * 74)

assert f'stack{STACK_N}' in EXPERIMENT_NAME, \
    f'EXPERIMENT_NAME {EXPERIMENT_NAME!r} out of sync with STACK_N={STACK_N} — stale config? re-run cell 3.'
jobs = [(fold, seed) for fold in ALL_FOLDS for seed in SEEDS]
print(f'\nTotal jobs: {len(jobs)}')

import shutil
def mirror_to_drive(path_local):
    if IN_COLAB and DRIVE_ARTIFACT_DIR:
        try:
            shutil.copy(path_local, DRIVE_ARTIFACT_DIR)
        except Exception as e:
            print(f'  drive mirror failed: {e}')

# Simple filename: f{fold}_s{seed} (no arm tag needed — single arm)
def _label(fold, seed):
    base = f'f{fold}_s{seed}'
    return base + ('_reusePT' if REUSE_PT else '')   # tag reuse artifacts (avoid cross-mode collision)

# [REUSE_PT] Per-fold donor pretrain cache. The first seed in a fold that actually RUNS (not
# resumed) pretrains once; its post-pretrain weights are cached here AND mirrored to disk for
# restart-robustness. Every later seed in the fold loads them and skips pretrain. On a fresh run
# the donor is SEEDS[0] -> 'train the first seed, reuse for the rest.'
_pt_reuse_cache = {}
def _reuse_ckpt_local(fold):  return os.path.join(CKPT_DIR, f'pretrain_reuse_f{fold}.pt')
def _reuse_ckpt_drive(fold):  return os.path.join(DRIVE_CKPT, f'pretrain_reuse_f{fold}.pt') if IN_COLAB else None
def _donor_seed(fold):
    # donor = first seed that runs for this fold (matches the per-epoch ckpt filename)
    return next(s for (f, s) in jobs if f == fold)
def _load_reuse_state(fold):
    """Donor post-pretrain CPU state_dict for this fold, or None if no donor has run yet."""
    if fold in _pt_reuse_cache:
        return _pt_reuse_cache[fold]
    p = _reuse_ckpt_local(fold)
    if not os.path.exists(p):
        dp = _reuse_ckpt_drive(fold)
        if dp and os.path.exists(dp):
            shutil.copy(dp, p)
    if os.path.exists(p):
        st = torch.load(p, map_location='cpu')
        _pt_reuse_cache[fold] = st
        return st
    # [REUSE_PT ckpt fallback] reuse the donor's saved ep{PRETRAIN_EPOCHS} pretrain checkpoint -> skip pretrain.
    if REUSE_PRETRAIN_EPOCH_CKPT:
        fn = f'pretrain_{_label(fold, _donor_seed(fold))}_ep{PRETRAIN_EPOCHS}.pt'
        for d in PRETRAIN_CKPT_SEARCH_DIRS:
            cand = os.path.join(d, fn)
            if os.path.exists(cand):
                st = torch.load(cand, map_location='cpu')
                _pt_reuse_cache[fold] = st
                print(f'  [REUSE_PT] loaded pretrain ep{PRETRAIN_EPOCHS} ckpt {fn} from {d} -- SKIPPING pretrain.')
                return st
    return None
def _save_reuse_state(fold, state):
    _pt_reuse_cache[fold] = state
    torch.save(state, _reuse_ckpt_local(fold))
    dp = _reuse_ckpt_drive(fold)
    if dp:
        try: torch.save(state, dp)
        except Exception as e: print(f'  reuse-ckpt drive mirror failed: {e}')

# === EXTERNAL per-(fold,seed) pretrained-checkpoint loader (PRETRAINED_DIR, cell 2) ===
# Loads 'pretrain_f{fold}_s{seed}_ep{PRETRAIN_EPOCHS}.pt' if present, to be passed as the finetune
# init (train_one_run then SKIPS pretrain). Per-(fold,seed) -> exact reuse with full seed diversity;
# takes PRIORITY over REUSE_PT. None/absent -> fall through to REUSE_PT / fresh pretrain.
_ext_pt_warned = set()
def _external_pretrain_state(fold, seed):
    if not PRETRAINED_DIR:
        return None
    fn = f'pretrain_f{fold}_s{seed}_ep{PRETRAIN_EPOCHS}.pt'
    cand = os.path.join(PRETRAINED_DIR, fn)
    if os.path.exists(cand):
        st = torch.load(cand, map_location='cpu')
        print(f'  [PRETRAINED_DIR] loaded {fn} -> SKIP the {PRETRAIN_EPOCHS}-ep pretrain phase.')
        return st
    if (fold, seed) not in _ext_pt_warned:
        _ext_pt_warned.add((fold, seed))
        print(f'  [PRETRAINED_DIR] {fn} NOT found in {PRETRAINED_DIR} -> pretrain fresh for this job.')
    return None

results = {}
t_total = time.time()
for i, (fold, seed) in enumerate(jobs, 1):
    label = _label(fold, seed)
    print(f'\n--- [{i}/{len(jobs)}] {label} ---')
    ckpt_path = os.path.join(LOCAL_ARTIFACT_DIR, f'ckpt_{label}.pt')
    result_path = os.path.join(LOCAL_ARTIFACT_DIR, f'result_{label}.npz')
    drive_ckpt_path = os.path.join(DRIVE_ARTIFACT_DIR, f'ckpt_{label}.pt') if IN_COLAB else None
    drive_result_path = os.path.join(DRIVE_ARTIFACT_DIR, f'result_{label}.npz') if IN_COLAB else None
    if IN_COLAB and not os.path.exists(result_path) and drive_result_path and os.path.exists(drive_result_path):
        shutil.copy(drive_result_path, result_path)
        if os.path.exists(drive_ckpt_path):
            shutil.copy(drive_ckpt_path, ckpt_path)
        print(f'  Resumed from Drive.')
    if os.path.exists(result_path) and os.path.exists(ckpt_path):
        d = np.load(result_path, allow_pickle=True)
        results[(fold, seed)] = {
            'best_rmse': float(d['best_rmse']),
            'best_epoch': int(d['best_epoch']),
            'final_epoch': int(d['final_epoch']),
            'fold': fold, 'seed': seed, 'ckpt_path': ckpt_path,
            'val_traj': d['val_traj'].tolist() if 'val_traj' in d else [],
            'pretrain_val_traj': d['pretrain_val_traj'].tolist() if 'pretrain_val_traj' in d else None,
        }
        print(f'  [RESUMED] best_rmse={d["best_rmse"]:.3f} at ep {int(d["best_epoch"])}')
        continue
    ext_state = _external_pretrain_state(fold, seed)
    if ext_state is not None:
        res = train_one_run(fold, seed, label, reuse_pt_state=ext_state, **BASE_HP)
    elif REUSE_PT:
        reuse_state = _load_reuse_state(fold)
        if reuse_state is None:
            print(f'  [REUSE_PT] fold {fold} DONOR (seed {seed}): pretrain once, reuse for the rest.')
            res = train_one_run(fold, seed, label, capture_pt=True, **BASE_HP)
            if res.get('pt_state') is not None:
                _save_reuse_state(fold, res['pt_state'])
                res['pt_state'] = None   # free the big dict (cached on disk + in _pt_reuse_cache)
        else:
            print(f'  [REUSE_PT] fold {fold} seed {seed}: REUSING donor pretrain (skip pretrain phase).')
            res = train_one_run(fold, seed, label, reuse_pt_state=reuse_state, **BASE_HP)
    else:
        res = train_one_run(fold, seed, label, **BASE_HP)
    margin = fold_const[fold] - res['best_rmse']
    print(f'  >> TVT-RMSE={res["best_rmse"]:.3f} ft (best ep {res["best_epoch"]}/{res["final_epoch"]}), '
          f'fold_const={fold_const[fold]:.3f}, margin={margin:+.3f}')
    if 'phase_info' in res:
        print(f'     phase_info: {res["phase_info"]}')
    torch.save(res['best_state'], ckpt_path)
    # Light in-memory record: scalars + path only.
    results[(fold, seed)] = {
        'best_rmse': res['best_rmse'], 'best_epoch': res['best_epoch'],
        'final_epoch': res['final_epoch'], 'fold': fold, 'seed': seed,
        'ckpt_path': ckpt_path,
        'val_traj': res['val_traj'], 'pretrain_val_traj': res.get('pretrain_val_traj'),
        'phase_info': res.get('phase_info'),
    }
    val_traj_safe = np.array([float('nan') if v is None else float(v) for v in res['val_traj']],
                             dtype=np.float32)
    pre_traj = res.get('pretrain_val_traj') or []
    pretrain_val_traj_safe = np.array([float('nan') if v is None else float(v) for v in pre_traj],
                                       dtype=np.float32)
    np.savez(result_path,
             best_rmse=res['best_rmse'], best_epoch=res['best_epoch'], final_epoch=res['final_epoch'],
             val_traj=val_traj_safe,
             pretrain_val_traj=pretrain_val_traj_safe)
    mirror_to_drive(ckpt_path)
    mirror_to_drive(result_path)
print(f'\nAll {len(jobs)} jobs complete in {(time.time()-t_total)/60:.1f} min.')


## §C. Aggregates + ensemble + diagnostics

In [ ]:
rows = []
for (fold, seed), r in results.items():
    rows.append({'fold': fold, 'seed': seed,
                 'rmse': r['best_rmse'], 'best_ep': r['best_epoch'], 'final_ep': r['final_epoch'],
                 'const': fold_const[fold], 'margin': fold_const[fold] - r['best_rmse']})
cells_df = pd.DataFrame(rows).sort_values(['fold','seed'])
print('=== All cells (n=%d) ===\n' % len(cells_df))
print(cells_df.round(3).to_string(index=False))

print('\n=== Per-fold single-seed mean + std (n=3) ===')
print(cells_df.groupby('fold')['margin'].agg(['mean', 'std']).round(3).to_string())

print(f'\n=== Per-fold const baseline ===')
for fold in ALL_FOLDS:
    print(f'  fold {fold}: const = {fold_const[fold]:.3f} ft, n_val = {len(fold_splits[fold]["val"])}')

print(f'\n=== Per-fold cross-seed std ===')
for fold in ALL_FOLDS:
    s = float(cells_df[cells_df.fold==fold]['margin'].std())
    print(f'  fold {fold}: std = {s:.3f} (n={len(cells_df[cells_df.fold==fold])})')

print(f'\n=== Best-epoch distribution (finetune) ===')
for fold in ALL_FOLDS:
    eps = cells_df[cells_df.fold == fold]['best_ep'].values
    print(f'  fold {fold}: best eps {sorted(eps.tolist())}, mean {np.mean(eps):.1f}')
all_eps = cells_df['best_ep'].values
print(f'  pooled: mean {np.mean(all_eps):.1f}, range [{min(all_eps)}, {max(all_eps)}]')



In [ ]:


# ============================================================================
# §C.1  Ensemble eval — seed-only (5-seed best_state average; the headline CV)
# ============================================================================
POSITION_BINS = 20

def _eval_model_pool(fold, ckpt_paths):
    """Average decoded expected_row over an arbitrary pool of checkpoints, each iterated to
    CHECKPOINT_K (deployment-matched refeed/handoff temperature). k=1 -> single forward (identity).
    Returns (rmse, pos_rmse, well_residuals, errs_sq)."""
    models = []
    for p in ckpt_paths:
        state = torch.load(p, map_location=DEVICE)
        m = build_model(encoder_name=ENCODER, in_ch=IN_CH).to(DEVICE)
        m.load_state_dict(state); m.eval(); models.append(m)
    val_ds = RealValDataset(fold_splits[fold]['val'])
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4,
                            worker_init_fn=seed_worker)
    all_errs_sq = []
    pos_sum_sq = np.zeros(POSITION_BINS, dtype=np.float64)
    pos_count  = np.zeros(POSITION_BINS, dtype=np.int64)
    well_residuals = {}
    with torch.no_grad():
        for batch in val_loader:
            wids = batch['well_id']; B = batch['image'].size(0)
            img = batch['image'].to(DEVICE)
            ers = []
            for m in models:
                # iterate to CHECKPOINT_K with the SAME deployment-matched refeed/handoff temps
                # used for selection; k==1 -> _eval_iter_logits(...)[1] == m(img) (identity).
                if CHECKPOINT_K > 1:
                    _kl, _ = _eval_iter_logits(m, img, CHECKPOINT_K)
                    logits = _kl[CHECKPOINT_K]
                else:
                    logits = m(img)
                probs = F.softmax(logits, dim=1)
                er = (probs * row_idx_t[None, :, None]).sum(dim=1).cpu().numpy()
                ers.append(er)
            er_mean = np.stack(ers, axis=0).mean(axis=0)          # == decode(mean probs)
            for b in range(B):
                wid = wids[b]; rec = well_images_real[wid]
                pred_col = rec['tvt_lo'] + (er_mean[b] + 0.5) * BIN_SIZE
                hw = pd.read_csv(well_hw_path(wid), usecols=['TVT','TVT_input'])
                n_hw = len(hw); bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
                pred_full = np.zeros(n_hw, dtype=np.float32)
                for c in range(W): pred_full[bin_edges[c]:bin_edges[c+1]] = pred_col[c]
                post = hw['TVT_input'].isna().values
                true_post = hw['TVT'].values[post]; pred_post = pred_full[post]
                ok = ~np.isnan(true_post); resid = true_post[ok] - pred_post[ok]
                all_errs_sq.append(resid**2)
                if len(resid) > 0:
                    pos_norm = np.linspace(0, 1, len(resid), endpoint=False) + 0.5/len(resid)
                    bin_idx = np.clip((pos_norm * POSITION_BINS).astype(int), 0, POSITION_BINS-1)
                    for kk in range(POSITION_BINS):
                        sel = bin_idx == kk
                        if sel.any():
                            pos_sum_sq[kk] += float(np.sum(resid[sel]**2)); pos_count[kk] += int(sel.sum())
                well_residuals[wid] = resid
    all_errs_sq_concat = np.concatenate(all_errs_sq)
    rmse = float(np.sqrt(all_errs_sq_concat.mean()))
    pos_rmse = np.sqrt(pos_sum_sq / np.maximum(pos_count, 1))
    return rmse, pos_rmse, well_residuals, all_errs_sq_concat


def _seed_ckpt_paths(fold):
    """Current behavior: one best_state per seed."""
    return [results[(fold, seed)]['ckpt_path'] for seed in SEEDS]

def ensemble_eval(fold):
    """UNCHANGED: 5-seed best_state ensemble (the current headline CV)."""
    return _eval_model_pool(fold, _seed_ckpt_paths(fold))

# ============================================================================
# Headline block — UNCHANGED (seed-only). Preserves downstream globals
# (ens_df, full_cv_rmse, pos_per_fold, all_residuals, errs_sq_per_fold, verdict).
# ============================================================================
print(f'=== Per-fold ensemble TVT-RMSE (seed-only, n={len(SEEDS)}, k={CHECKPOINT_K}) ===\n')
ens_rows = []; pos_per_fold = {}; all_residuals = {}; errs_sq_per_fold = {}
for fold in ALL_FOLDS:
    rmse, pos_rmse, well_res, errs_sq = ensemble_eval(fold)
    margin = fold_const[fold] - rmse
    ens_rows.append({'fold': fold, 'ens_rmse': rmse, 'ens_margin': margin})
    pos_per_fold[fold] = pos_rmse; all_residuals.update(well_res); errs_sq_per_fold[fold] = errs_sq
    print(f'  fold {fold}: ens TVT-RMSE = {rmse:.3f}, margin = {margin:+.3f}, fold_const = {fold_const[fold]:.3f}')
ens_df = pd.DataFrame(ens_rows)

all_errs_sq_5fold = np.concatenate([errs_sq_per_fold[f] for f in ALL_FOLDS])
full_cv_rmse = float(np.sqrt(all_errs_sq_5fold.mean()))
n_total_post_ps_rows = len(all_errs_sq_5fold)
n_total_wells = sum(len(fold_splits[f]['val']) for f in ALL_FOLDS)
all_const_errs_sq = []
for f in ALL_FOLDS:
    for wid in fold_splits[f]['val']:
        hw = pd.read_csv(well_hw_path(wid), usecols=['TVT', 'TVT_input'])
        known = hw.loc[~hw['TVT_input'].isna(), 'TVT_input'].values
        if len(known) == 0: continue
        last_known = float(known[-1])
        post = hw['TVT_input'].isna().values
        true_post = hw['TVT'].values[post]; ok = ~np.isnan(true_post)
        all_const_errs_sq.append((true_post[ok] - last_known)**2)
full_cv_const = float(np.sqrt(np.concatenate(all_const_errs_sq).mean()))
full_cv_margin = full_cv_const - full_cv_rmse

print(f'\n=== FULL-CV ENSEMBLE (seed-only headline, k={CHECKPOINT_K}) ===')
print(f'  Wells: {n_total_wells}   Post-PS rows: {n_total_post_ps_rows:,}')
print(f'  Const baseline: {full_cv_const:.3f} ft')
print(f'  5-fold ens RMSE: {full_cv_rmse:.3f} ft   margin {full_cv_margin:+.3f}')

cv_delta = CV_TO_BEAT - full_cv_rmse
worst_fold_rmse = float(ens_df['ens_rmse'].max())
worst_fold_regress = worst_fold_rmse - CV_TO_BEAT
print(f'\n=== verdict — vs CV_TO_BEAT = {CV_TO_BEAT:.2f} ft ===')
print(f'  Δ full-CV: {cv_delta:+.3f} ft   worst-fold RMSE: {worst_fold_rmse:.3f} (regress vs CV_TO_BEAT: {worst_fold_regress:+.3f})')
if cv_delta >= CV_PASS_MARGIN and worst_fold_regress <= CV_FOLD_REGRESS_TOL:
    verdict = 'PASS'
elif cv_delta <= -CV_PASS_MARGIN:
    verdict = 'NULL-<<'
else:
    verdict = 'NULL-~'
print(f'  VERDICT: {verdict}')

### §C.2 · Save per-seed OOF predictions

Writes one file per `(fold, seed)` — `oof_f{fold}_s{seed}.npz` — alongside the checkpoints (`ckpt_f{fold}_s{seed}.pt`), each seed scored on its own fold's held-out val wells at the deployment-matched depth (`CHECKPOINT_K`). This is the per-seed decode that the §C.1 ensemble averages away. Predictions are also kept in memory as `oof_preds[(fold, seed)]`. Run after the training cell has populated `results`.

In [ ]:
# ============================================================================
# §C.2  PER-SEED OOF PREDICTIONS — save one file per (fold, seed), mirroring the
#       checkpoint naming (ckpt_f{fold}_s{seed}.pt  ->  oof_f{fold}_s{seed}.npz).
# ----------------------------------------------------------------------------
# Each seed is scored ONLY on its own fold's held-out val wells (true OOF), at the
# same deployment-matched depth the checkpoint was selected on (CHECKPOINT_K, with
# REFEED_TRAIN_T / HANDOFF_TRAIN_T). The per-well decoded prediction is exactly the
# `er` that _eval_model_pool averages away — here we keep the individual seed.
#
# Saved per file (npz):
#   well_ids     (N,)            str  — val well order
#   er_row       object[N]       per-well softmax-expected ROW in [0,H) (grid space, len=W=512)
#   pred_col     object[N]       per-well TVT prediction per MD column (len=W=512)
#   pred_post    object[N]       per-well TVT prediction on POST-PS raw rows (decoded to native HW length)
#   resid_post   object[N]       true_post - pred_post on known post-PS rows (the OOF error signal)
#   rmse         scalar          this seed's OOF RMSE on its fold (sanity; matches the log)
#   fold, seed, checkpoint_k, refeed_T, handoff_T   scalars (provenance)
# Object arrays hold ragged per-well vectors (wells have different native HW lengths).
# The in-memory dict `oof_preds[(fold, seed)]` holds the same content for immediate use.
# ============================================================================
import os, shutil

# Write OOF files FLAT into LOCAL_ARTIFACT_DIR, exactly like the checkpoints
# (ckpt_f{fold}_s{seed}.pt). A subdir would break the final flat Drive-mirror loop,
# which shutil.copy's every listdir entry and cannot copy a directory.
OOF_DIR = LOCAL_ARTIFACT_DIR
os.makedirs(OOF_DIR, exist_ok=True)

# One-time migration: if a previous run wrote into a nested 'oof/' folder, move those
# already-computed npz files up (no re-inference) and remove the stale dir so the mirror
# cell stops crashing. Drive copies were already flat (per-file mirror), so nothing is lost.
_stale_oof = os.path.join(LOCAL_ARTIFACT_DIR, 'oof')
if os.path.isdir(_stale_oof):
    for _fn in os.listdir(_stale_oof):
        _src = os.path.join(_stale_oof, _fn); _dst = os.path.join(LOCAL_ARTIFACT_DIR, _fn)
        if os.path.exists(_dst): os.remove(_src)
        else: shutil.move(_src, _dst)
    try: os.rmdir(_stale_oof)
    except OSError: pass
    print(f'  [migrated] flattened stale oof/ subdir -> {LOCAL_ARTIFACT_DIR}')

def _decode_seed_oof(fold, ckpt_path, k=None):
    """Single-checkpoint OOF over `fold`'s val wells. Returns a per-well dict. Mirrors
    _eval_model_pool's decode EXACTLY but for ONE model (no seed averaging)."""
    k = int(CHECKPOINT_K if k is None else k)
    state = torch.load(ckpt_path, map_location=DEVICE)
    m = build_model(encoder_name=ENCODER, in_ch=IN_CH).to(DEVICE)
    m.load_state_dict(state); m.eval()
    val_ds = RealValDataset(fold_splits[fold]['val'])
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4,
                            worker_init_fn=seed_worker)
    out = {}                                   # wid -> (er_row, pred_col, pred_post, resid_post)
    with torch.no_grad():
        for batch in val_loader:
            wids = batch['well_id']; B = batch['image'].size(0)
            img = batch['image'].to(DEVICE)
            # deployment-matched depth: k>1 self-refeed, k==1 single forward (identity)
            if k > 1:
                _kl, _ = _eval_iter_logits(m, img, k)
                logits = _kl[k]
            else:
                logits = m(img)
            er = (F.softmax(logits, dim=1) * row_idx_t[None, :, None]).sum(dim=1).cpu().numpy()  # (B,W)
            for b in range(B):
                wid = wids[b]; rec = well_images_real[wid]
                pred_col = rec['tvt_lo'] + (er[b] + 0.5) * BIN_SIZE                # (W,) TVT per MD col
                hw = pd.read_csv(well_hw_path(wid), usecols=['TVT', 'TVT_input'])
                n_hw = len(hw); bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
                pred_full = np.zeros(n_hw, dtype=np.float32)
                for c in range(W): pred_full[bin_edges[c]:bin_edges[c+1]] = pred_col[c]
                post = hw['TVT_input'].isna().values
                true_post = hw['TVT'].values[post]; pred_post = pred_full[post]
                ok = ~np.isnan(true_post); resid = true_post[ok] - pred_post[ok]
                out[wid] = (er[b].astype(np.float32), pred_col.astype(np.float32),
                            pred_post.astype(np.float32), resid.astype(np.float32))
    del m
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return out

def _save_seed_oof(fold, seed, per_well, path_local):
    wids = list(per_well.keys())
    er_row     = np.empty(len(wids), dtype=object)
    pred_col   = np.empty(len(wids), dtype=object)
    pred_post  = np.empty(len(wids), dtype=object)
    resid_post = np.empty(len(wids), dtype=object)
    sq = []
    for i, wid in enumerate(wids):
        er_row[i], pred_col[i], pred_post[i], resid_post[i] = per_well[wid]
        sq.append(resid_post[i] ** 2)
    rmse = float(np.sqrt(np.concatenate(sq).mean())) if sq else float('nan')
    np.savez(path_local,
             well_ids=np.array(wids), er_row=er_row, pred_col=pred_col,
             pred_post=pred_post, resid_post=resid_post,
             rmse=rmse, fold=fold, seed=seed,
             checkpoint_k=int(CHECKPOINT_K),
             refeed_T=float(REFEED_TRAIN_T), handoff_T=float(HANDOFF_TRAIN_T))
    return rmse

oof_preds = {}      # (fold, seed) -> per_well dict, in memory for downstream use
print(f'=== Saving per-seed OOF (k={CHECKPOINT_K}, refeed_T={REFEED_TRAIN_T}, '
      f'handoff_T={HANDOFF_TRAIN_T}) -> {OOF_DIR} ===\n')
for (fold, seed) in sorted(results.keys()):
    ckpt_path = results[(fold, seed)]['ckpt_path']
    label = f'f{fold}_s{seed}'                                 # matches ckpt_f{fold}_s{seed}.pt
    oof_local = os.path.join(OOF_DIR, f'oof_{label}.npz')
    if os.path.exists(oof_local):
        print(f'  [skip] oof_{label}.npz exists'); continue
    per_well = _decode_seed_oof(fold, ckpt_path)
    oof_preds[(fold, seed)] = per_well
    rmse = _save_seed_oof(fold, seed, per_well, oof_local)
    mirror_to_drive(oof_local)                                 # reuse cell-26 Drive mirror
    # sanity: this seed's single-model OOF RMSE (the ensemble RMSE in §C.1 will be <= this)
    print(f'  fold {fold} seed {seed}: {len(per_well)} wells  OOF RMSE={rmse:.3f}  -> oof_{label}.npz')
print(f'\nDone. {len(oof_preds)} seed-OOF files written to {OOF_DIR} '
      f'(+ Drive mirror if Colab). In-memory: oof_preds[(fold, seed)].')


In [ ]:
# === DATUM vs DRIFT decomposition at full CV — mechanism check for the corruption arm ===
# Row-weighted orthogonal split (matches the 5.80 datum + 5.30 drift convention):
#   datum = per-well MEAN residual ; drift = within-well residual after removing it.
# Hypothesis: reduced_B (removing the tvt_offset misalignment) lets the model pin absolute position from
# the GR<->TW match, so DATUM should shrink more than DRIFT vs the locked baseline. Compare to baseline's split.
_b_rows, _w_rows = [], []
for _wid, _resid in all_residuals.items():
    _r = np.asarray(_resid, dtype=np.float64)
    if _r.size == 0: continue
    _bw = _r.mean()
    _b_rows.append(np.full(_r.size, _bw)); _w_rows.append(_r - _bw)
_ball = np.concatenate(_b_rows); _wall = np.concatenate(_w_rows)
_datum_rms = float(np.sqrt((_ball**2).mean())); _drift_rms = float(np.sqrt((_wall**2).mean()))
_pwb = np.array([np.asarray(_r).mean() for _r in all_residuals.values() if np.asarray(_r).size])
print('=== datum / drift decomposition (arm=%s, full 5-fold CV) ===' % CORRUPTION_MODE)
print(f'  pooled CV RMSE     = {full_cv_rmse:.3f} ft')
print(f'  DATUM  (per-well)  = {_datum_rms:.3f} ft   <- corruption-hypothesis target bucket (baseline ~5.80)')
print(f'  DRIFT  (within)    = {_drift_rms:.3f} ft   (baseline ~5.30)')
print(f'  check sqrt(d^2+w^2) = {np.sqrt(_datum_rms**2 + _drift_rms**2):.3f} ft  (~= pooled)')
print(f'  per-well bias: mean={_pwb.mean():+.3f} ft, std={_pwb.std():.3f} ft, n_wells={_pwb.size}')


In [ ]:
# === TAIL metrics — per-well RMSE distribution + catastrophic-miss count ===
# RMSE is tail-dominated, and the match_corr cleanup most plausibly helps the ambiguous/tail wells.
# Pooled CV + datum/drift can look flat while the tail moves; this surfaces that. Reads the 5-seed
# ENSEMBLE (all_residuals), same as datum/drift. Compare these numbers between arms (reduced_B vs _C).
_wids   = [w for w, r in all_residuals.items() if np.asarray(r).size]
_rmse_w = np.array([float(np.sqrt((np.asarray(all_residuals[w], dtype=np.float64)**2).mean())) for w in _wids])
_sse_w  = np.array([float((np.asarray(all_residuals[w], dtype=np.float64)**2).sum()) for w in _wids])
_n_w    = np.array([np.asarray(all_residuals[w]).size for w in _wids])

print('=== TAIL metrics (arm=%s, 5-fold ensemble, n_wells=%d) ===' % (CORRUPTION_MODE, len(_wids)))
print('  per-well RMSE percentiles (ft):')
for _p in (50, 75, 90, 95, 99):
    print(f'    p{_p:<2d} = {np.percentile(_rmse_w, _p):6.3f}')
print(f'    max  = {_rmse_w.max():6.3f}   mean = {_rmse_w.mean():6.3f}')
print('  catastrophic-miss wells (per-well RMSE > threshold):')
for _t in (8, 10, 12, 15):
    _c = int((_rmse_w > _t).sum())
    print(f'    > {_t:2d} ft : {_c:3d} wells  ({100*_c/len(_wids):4.1f}%)')
_order = np.argsort(_rmse_w)[::-1]; _tot_sse = _sse_w.sum()
for _frac in (0.05, 0.10, 0.20):
    _k = max(1, int(_frac * len(_wids)))
    print(f'  worst {int(_frac*100):2d}% of wells ({_k:3d}) carry {100*_sse_w[_order[:_k]].sum()/_tot_sse:4.1f}% of pooled squared error')
print(f'  (pooled RMSE check: {np.sqrt(_tot_sse/_n_w.sum()):.3f} ft  ~= full_cv_rmse {full_cv_rmse:.3f})')

In [ ]:
pos_pooled = np.sqrt(np.mean([pos_per_fold[f]**2 for f in ALL_FOLDS], axis=0))
ft15a_pos_ref = np.array([3.05, 5.85, 6.90, 8.10, 8.95, 9.55, 10.50, 11.05, 11.65, 11.65,
                          11.50, 12.05, 12.50, 12.95, 13.05, 13.60, 14.00, 14.40, 14.80, 15.60])
x = np.linspace(0, 1, POSITION_BINS, endpoint=False) + 0.5/POSITION_BINS

plt.figure(figsize=(11, 5))
plt.plot(x, pos_pooled, marker='s', lw=2, color='#ff7f0e',
         label='5ch dZ (pre+ft, n=3 seeds × 5 folds)')
plt.plot(x, ft15a_pos_ref, 'k--', marker='v', alpha=0.4, label='FT15a 5-fold ref')
plt.xlabel('normalized post-PS position (0=heel, 1=toe)'); plt.ylabel('TVT pooled RMSE (ft)')
plt.title('Per-position RMSE: dZ-channel probe (5-fold pooled)')
plt.legend(loc='upper left', fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

h = pos_pooled[:4].mean(); t = pos_pooled[-4:].mean()
print(f'\nHeel/toe (pooled): heel = {h:.2f}  toe = {t:.2f}  gap = {t-h:.2f}')
print(f'FT15a ref:          heel = {ft15a_pos_ref[:4].mean():.2f}  toe = {ft15a_pos_ref[-4:].mean():.2f}')


In [ ]:
print('=== Per-fold heel/toe ===')
for fold in ALL_FOLDS:
    pos = pos_per_fold[fold]
    h = pos[:4].mean(); t = pos[-4:].mean()
    print(f'  fold {fold}: heel = {h:.2f}  toe = {t:.2f}  gap = {t-h:.2f}')


In [ ]:
def project_modes(residuals, n_modes=8, n_resample=200):
    well_ids_with_data = [w for w, r in residuals.items() if len(r) >= 10]
    coeffs = np.zeros((len(well_ids_with_data), n_modes)); energies = np.zeros(len(well_ids_with_data))
    for i, w in enumerate(well_ids_with_data):
        r = residuals[w]
        x_orig = np.linspace(0, 1, len(r), endpoint=False) + 0.5/len(r)
        x_new = np.linspace(0, 1, n_resample, endpoint=False) + 0.5/n_resample
        r_resampled = np.interp(x_new, x_orig, r)
        modes = np.zeros((n_modes, n_resample))
        for k in range(1, n_modes+1):
            modes[k-1] = np.sqrt(2) * np.sin((2*k - 1) * np.pi * x_new / 2)
        coeffs[i] = (modes @ r_resampled) / n_resample
        energies[i] = (r_resampled**2).sum() / n_resample
    return well_ids_with_data, coeffs, energies

wids, coeffs, energies = project_modes(all_residuals, n_modes=8)
c_sq = coeffs**2
pool_mode1 = c_sq[:, 0].sum() / energies.sum()
pool_modes_1_8 = c_sq.sum(axis=1).sum() / energies.sum()
print(f'5ch dZ: pool_mode1={pool_mode1*100:.1f}%  pool_modes_1-8={pool_modes_1_8*100:.1f}%')
print('Reference: FT24 arm 2 (pre+ft) pool_mode1 was ~58.5% in 2-fold probe.')


## §D. Baseline lock + Kaggle inference artifact preservation

In [ ]:
# === Arm summary — labels DERIVED from live knob values (METHODOLOGY §12: no stale text) ===
_arm_bits = []
if STACK_N >= 2: _arm_bits.append(f'Stack{STACK_N}(deep_sup={STACK_DEEP_SUP_WEIGHT}, posterior-handoff)')
if MD_WARP_STRENGTH > 0: _arm_bits.append(f'MDwarp(p={MD_WARP_P}, s={MD_WARP_STRENGTH}, ac={MD_WARP_AUTOCORR})')
if SHEAR_STD_ROWS   > 0: _arm_bits.append(f'shear(p={SHEAR_P}, std={SHEAR_STD_ROWS}r, ac={SHEAR_AUTOCORR}, max={SHEAR_MAX_ROWS})')
if BEDWARP_STRENGTH > 0: _arm_bits.append(f'BedWarp(p={BEDWARP_P}, s={BEDWARP_STRENGTH}, ac={BEDWARP_AUTOCORR}r)')
if PHASE_JITTER_P   > 0: _arm_bits.append(f'PhaseJitter(p={PHASE_JITTER_P})')
if TOECROP_P        > 0: _arm_bits.append(f'ToeCrop(p={TOECROP_P}, maxfrac={TOECROP_MAX_FRAC})')
if GR_AUG_P         > 0: _arm_bits.append(f'GR_AUG(p={GR_AUG_P}, gain_s={GR_AUG_GAIN_S}, off_s={GR_AUG_OFFSET_S})')
if UNROLL_ENABLE and UNROLL_P > 0: _arm_bits.append(f'unroll(p={UNROLL_P}, k={tuple(UNROLL_K_CHOICES)})')
if EVENT_PACK == 'v3_expA': _arm_bits.append(f'EventPackA(dipbrk={DIPBREAK_PROB}, repeat={REPEAT_PROB}, pinch-dormant)')
if FZ_ENABLE: _arm_bits.append(f'FaultZone(growth_floor={FZ_GROWTH_FLOOR if FZ_GROWTH_ENABLE else None}, pair={FZ_PAIR_PROB}, sharp_p={FZ_RAMP_SHARP_P}, w={FZ_RAMP_W_MIN}-{FZ_RAMP_W_MAX})')
if REFEED_TRAIN_T != 1.0:  _arm_bits.append(f'refeedT={REFEED_TRAIN_T}')
if HANDOFF_TRAIN_T != 1.0: _arm_bits.append(f'handoffT={HANDOFF_TRAIN_T}')
ARM_LABEL = ' + '.join(_arm_bits) if _arm_bits else 'canonical (no warp / no dz-jitter)'
_tag_bits = [t for t, on in ((f'stack{STACK_N}', STACK_N >= 2),
                                  ('mdwarp', MD_WARP_STRENGTH > 0), ('shear', SHEAR_STD_ROWS > 0),
                                  ('bedwarp', BEDWARP_STRENGTH > 0),
                                  ('phase', PHASE_JITTER_P > 0),
                                  ('toecrop', TOECROP_P > 0),
                                  ('gr_aug', GR_AUG_P > 0),
                                  ('unroll', UNROLL_ENABLE and UNROLL_P > 0),
                                  ('expa', EVENT_PACK == 'v3_expA'),
                                  ('fz', FZ_ENABLE)) if on]
ARM_TAG = '_'.join(_tag_bits) if _tag_bits else 'canonical'

print(f'=== Arm summary: {ARM_LABEL} ===')
print(f'  encoder={ENCODER}, in_ch={IN_CH}, corruption={CORRUPTION_MODE}, '
      f'HFlip p={H_FLIP_P} (pretrain {H_FLIP_P_PRETRAIN}), HintDrop p={HINT_DROP_P} (finetune-only)')
print(f'  Recipe: synth pretrain ({PRETRAIN_EPOCHS} ep, {N_SYNTH_PRETRAIN} wells) -> real finetune ({FINETUNE_EPOCHS} ep)')
print(f'  Total checkpoints: {len(ALL_FOLDS) * len(SEEDS)} ({len(ALL_FOLDS)} folds x {len(SEEDS)} seeds)')

print(f'\n=== Per-fold metrics ===')
print(f'  {"fold":<5} {"ss_mean":>9} {"ss_std":>7} {"ens_rmse":>9} {"ens_marg":>9} {"const":>8} {"n_val":>6}')
for fold in ALL_FOLDS:
    ss_mean = float(cells_df[cells_df.fold==fold]['margin'].mean())
    ss_std  = float(cells_df[cells_df.fold==fold]['margin'].std())
    ens_margin = float(ens_df[ens_df.fold==fold]['ens_margin'].iloc[0])
    ens_rmse = float(ens_df[ens_df.fold==fold]['ens_rmse'].iloc[0])
    const = fold_const[fold]; n_val = len(fold_splits[fold]['val'])
    print(f'  {fold:<5} {ss_mean:>+9.3f} {ss_std:>7.3f} {ens_rmse:>9.3f} {ens_margin:>+9.3f} '
          f'{const:>8.3f} {n_val:>6}')

print(f'\n=== Headline metric ===')
print(f'  {IN_CH}ch full-CV ensemble TVT-RMSE: {full_cv_rmse:.3f} ft  (margin {full_cv_margin:+.3f} vs const {full_cv_const:.3f})')

print(f'\n=== Verdict vs CV_TO_BEAT ({CV_TO_BEAT:.3f} ft) — pre-reg ladder only ===')
print(f'  delta = {cv_delta:+.3f} ft   ->   {verdict}')
print(f'  (PASS <= {CV_TO_BEAT - CV_PASS_MARGIN:.3f} / NULL-~ band +/-{CV_PASS_MARGIN} / '
      f'worst-fold regress tol {CV_FOLD_REGRESS_TOL})')
print(f'  Reference: lock 6.541; ARCHITECTURE-family rule binds -> watch the worst-fold regress (FT39q died here) and the tail.')
print(f'  No auto-recommendation by design — route via the ledger, per-fold coherence, and the')
print(f'  tail co-primary (worst-10% share / >12 ft count in the TAIL cell above).')


In [ ]:
import json as _json

# ============================================================================
# Diagnostics serialization (FT: tail co-primaries are the actual verdict basis)
# Per-fold AND pooled datum/drift + tail + heel/toe, computed from the 5-seed
# ENSEMBLE residuals (all_residuals) and per-position RMSE (pos_per_fold) produced
# in section C. Pooled values match the printed cells 24/25/26 by construction;
# the PER-FOLD split is added here (cells 24/25 only ever print pooled), so a wash
# can be read against mechanism (datum-limited folds vs drift folds) and arms
# (e.g. AdamW baseline vs SAM) can be diffed from the saved JSON, not by eye.
# No retraining needed: re-running section C -> here on existing ckpts is enough.
_TAIL_THRESH = (8, 10, 12, 15)        # catastrophic per-well RMSE thresholds (ft)
_WORST_FRACS = (0.05, 0.10, 0.20)     # worst-k% of wells -> share of pooled squared error
_PCTLS       = (50, 75, 90, 95, 99)

def _dd_split(res_dict, wids):
    """Row-weighted orthogonal split: datum = per-well MEAN residual, drift = within-well remainder
    (the 5.80 datum + 5.30 drift convention from cell 24)."""
    b, w, pwb = [], [], []
    for wid in wids:
        r = np.asarray(res_dict.get(wid, []), dtype=np.float64)
        if r.size == 0: continue
        mu = r.mean(); b.append(np.full(r.size, mu)); w.append(r - mu); pwb.append(mu)
    if not b: return None
    b = np.concatenate(b); w = np.concatenate(w); pwb = np.asarray(pwb)
    return {"datum_rms": float(np.sqrt((b**2).mean())),
            "drift_rms": float(np.sqrt((w**2).mean())),
            "datum_share_of_var": float((b**2).mean() / max((b**2).mean() + (w**2).mean(), 1e-12)),
            "per_well_bias_mean": float(pwb.mean()), "per_well_bias_std": float(pwb.std()),
            "n_wells": int(pwb.size)}

def _tail_stats(res_dict, wids):
    rmse_w, sse_w, n_w = [], [], []
    for wid in wids:
        r = np.asarray(res_dict.get(wid, []), dtype=np.float64)
        if r.size == 0: continue
        rmse_w.append(float(np.sqrt((r**2).mean()))); sse_w.append(float((r**2).sum())); n_w.append(int(r.size))
    if not rmse_w: return None
    rmse_w = np.asarray(rmse_w); sse_w = np.asarray(sse_w); n_w = np.asarray(n_w)
    order = np.argsort(rmse_w)[::-1]; tot = float(sse_w.sum())
    return {"n_wells": int(rmse_w.size),
            "rmse_pct": {f"p{p}": float(np.percentile(rmse_w, p)) for p in _PCTLS},
            "rmse_max": float(rmse_w.max()), "rmse_mean": float(rmse_w.mean()),
            "catastrophic_counts": {f"gt_{t}ft": int((rmse_w > t).sum()) for t in _TAIL_THRESH},
            "catastrophic_frac":   {f"gt_{t}ft": float((rmse_w > t).mean()) for t in _TAIL_THRESH},
            "worst_frac_sse_share": {f"worst_{int(fr*100)}pct":
                float(sse_w[order[:max(1, int(fr*rmse_w.size))]].sum() / max(tot, 1e-12)) for fr in _WORST_FRACS},
            "pooled_rmse": float(np.sqrt(tot / max(int(n_w.sum()), 1)))}

def _heel_toe(pos_rmse):
    pr = np.asarray(pos_rmse, dtype=np.float64)
    return {"heel": float(pr[:4].mean()), "toe": float(pr[-4:].mean()), "gap": float(pr[-4:].mean() - pr[:4].mean())}

_pos_pooled = np.sqrt(np.mean([pos_per_fold[f]**2 for f in ALL_FOLDS], axis=0))
_diag_per_fold = {}
for f in ALL_FOLDS:
    _fw = [w for w in fold_splits[f]['val'] if w in all_residuals]
    _diag_per_fold[f"fold{f}"] = {
        "ens_rmse":    float(ens_df[ens_df.fold == f]['ens_rmse'].iloc[0]),
        "datum_drift": _dd_split(all_residuals, _fw),
        "tail":        _tail_stats(all_residuals, _fw),
        "heel_toe":    _heel_toe(pos_per_fold[f]),
    }
diagnostics = {
    "note": "5-seed ensemble residual diagnostics. Per-fold tail/datum-drift included so a "
            "per-fold wash can be read against mechanism. Diff these between arms.",
    "pooled": {"full_cv_rmse": full_cv_rmse,
               "datum_drift": _dd_split(all_residuals, list(all_residuals.keys())),
               "tail":        _tail_stats(all_residuals, list(all_residuals.keys())),
               "heel_toe":    _heel_toe(_pos_pooled)},
    "per_fold": _diag_per_fold,
    "per_position_rmse": {"bins": int(POSITION_BINS),
                          "pooled": [float(v) for v in _pos_pooled],
                          **{f"fold{f}": [float(v) for v in pos_per_fold[f]] for f in ALL_FOLDS}},
}
try:
    diagnostics["residual_modes"] = {"pool_mode1": float(pool_mode1), "pool_modes_1_8": float(pool_modes_1_8)}
except NameError:
    pass

# inline per-fold table (so the verdict basis is visible on re-run, not just in the JSON)
print('\n=== Diagnostics: per-fold tail co-primaries + datum/drift (5-seed ensemble) ===')
print(f'  {"fold":<5}{"ens":>7}{"datum":>8}{"drift":>8}{"w10%SSE":>9}{">12ft":>7}{"toe-heel":>9}')
for f in ALL_FOLDS:
    _d = _diag_per_fold[f"fold{f}"]; _dd = _d["datum_drift"]; _tl = _d["tail"]; _ht = _d["heel_toe"]
    print(f'  {f:<5}{_d["ens_rmse"]:>7.2f}{_dd["datum_rms"]:>8.2f}{_dd["drift_rms"]:>8.2f}'
          f'{100*_tl["worst_frac_sse_share"]["worst_10pct"]:>8.0f}%{_tl["catastrophic_counts"]["gt_12ft"]:>7d}'
          f'{_ht["gap"]:>9.2f}')
_pp = diagnostics["pooled"]
print(f'  {"POOL":<5}{full_cv_rmse:>7.2f}{_pp["datum_drift"]["datum_rms"]:>8.2f}{_pp["datum_drift"]["drift_rms"]:>8.2f}'
      f'{100*_pp["tail"]["worst_frac_sse_share"]["worst_10pct"]:>8.0f}%{_pp["tail"]["catastrophic_counts"]["gt_12ft"]:>7d}'
      f'{_pp["heel_toe"]["gap"]:>9.2f}')

summary = {
    "experiment": f"ARM_{ARM_TAG}__{ARM_LABEL}",
    "folds": ALL_FOLDS, "seeds": SEEDS,
    "n_total_runs": len(ALL_FOLDS) * len(SEEDS),
    "recipe": {
        "encoder_name": ENCODER, "in_ch": IN_CH, "uses_pretrain": USE_PRETRAIN,
        "stack_n": STACK_N, "stack_deep_sup_weight": STACK_DEEP_SUP_WEIGHT,
        "stack_handoff": ("softmax_posterior_plane" if STACK_N >= 2 else None),
        "pretrain_epochs": PRETRAIN_EPOCHS, "finetune_epochs": FINETUNE_EPOCHS,
        "synth_pretrain_pool": N_SYNTH_PRETRAIN,
        "lr": 1e-3, "wd": 1e-3, "batch_size": BASE_HP['batch_size'], "patience": BASE_HP['patience'],
        "warmup_epochs": WARMUP_EPOCHS, "ema_decay": EMA_DECAY,
        "h_flip_p": H_FLIP_P, "h_flip_p_pretrain": H_FLIP_P_PRETRAIN, "hint_drop_p": HINT_DROP_P,
        "corruption_mode": CORRUPTION_MODE,
        "md_warp": {"strength": MD_WARP_STRENGTH, "p": MD_WARP_P, "autocorr": MD_WARP_AUTOCORR},
        "shear": {"std_rows": SHEAR_STD_ROWS, "p": SHEAR_P, "autocorr": SHEAR_AUTOCORR, "max_rows": SHEAR_MAX_ROWS},
        "bedwarp": {"strength": BEDWARP_STRENGTH, "p": BEDWARP_P, "autocorr": BEDWARP_AUTOCORR},
        "phase_jitter": {"p": PHASE_JITTER_P},
        "toecrop": {"p": TOECROP_P, "max_frac": TOECROP_MAX_FRAC},
        "gr_aug": {"p": GR_AUG_P, "gain_s": GR_AUG_GAIN_S, "offset_s": GR_AUG_OFFSET_S},
        "dz_synth_dip_frac": DZ_SYNTH_DIP_FRAC,
        "dz_synth_dip_autocorr_frac": DZ_SYNTH_DIP_AUTOCORR_FRAC,
        "dz_synth_noise_std": DZ_SYNTH_NOISE_STD,
        "event_pack": {"mode": EVENT_PACK, "dipbreak_prob": DIPBREAK_PROB,
                       "dipbreak_dslope": [DIPBREAK_DSLOPE_MIN, DIPBREAK_DSLOPE_MAX],
                       "repeat_prob": REPEAT_PROB, "pinch_prob": PINCH_PROB,
                       "max_abs_dip": EXPA_MAX_ABS_DIP, "exc_cap_ft": EXPA_EXC_CAP_FT},
        "fz": {"enabled": FZ_ENABLE, "hash": FZ_HASH,
               "growth": {"enabled": FZ_GROWTH_ENABLE, "floor": FZ_GROWTH_FLOOR},
               "pair": {"prob": FZ_PAIR_PROB, "spacing": [FZ_PAIR_SPACING_MIN, FZ_PAIR_SPACING_MAX],
                        "recover": [FZ_PAIR_RECOVER_MIN, FZ_PAIR_RECOVER_MAX]},
               "ramp": {"sharp_p": FZ_RAMP_SHARP_P, "w": [FZ_RAMP_W_MIN, FZ_RAMP_W_MAX],
                        "max_slope": FZ_RAMP_MAX_SLOPE}},
    },
    "image_constants": {"H": H, "W": W, "BIN_SIZE": BIN_SIZE, "K_MAX": K_MAX},
    "input_channels": ["TW_GR_norm", "HW_GR_norm", "sparse_hint", "cost_volume_norm", "dZ_per_col_norm"],
    "results_per_fold": {
        f"fold{f}": {
            "ss_mean_margin": float(cells_df[cells_df.fold==f]['margin'].mean()),
            "ss_std_margin":  float(cells_df[cells_df.fold==f]['margin'].std()),
            "ens_margin":     float(ens_df[ens_df.fold==f]['ens_margin'].iloc[0]),
            "ens_rmse":       float(ens_df[ens_df.fold==f]['ens_rmse'].iloc[0]),
            "fold_const":     float(fold_const[f]),
            "n_val_wells":    len(fold_splits[f]['val']),
        } for f in ALL_FOLDS
    },
    "full_cv": {
        "ens_rmse": full_cv_rmse, "ens_margin": full_cv_margin, "const_rmse": full_cv_const,
        "n_wells": n_total_wells, "n_post_ps_rows": int(n_total_post_ps_rows),
    },
    "probe_verdict": {
        "cv_to_beat": CV_TO_BEAT, "delta": cv_delta, "verdict": verdict,
        "pass_margin": CV_PASS_MARGIN, "worst_fold_rmse": worst_fold_rmse,
    },
    "diagnostics": diagnostics,
}
# Arm-named outputs (CURRENT_BASELINE standing debt: every line since ft39l reused these
# filenames; the 'experiment' field is decisive, but the collision ends here).
summary_path = os.path.join(LOCAL_ARTIFACT_DIR, 'faultzone_summary.json')
with open(summary_path, 'w') as fh: _json.dump(summary, fh, indent=2)

cells_df.to_csv(os.path.join(LOCAL_ARTIFACT_DIR, 'faultzone_cells.csv'), index=False)
ens_df.to_csv(os.path.join(LOCAL_ARTIFACT_DIR, 'faultzone_ensemble.csv'), index=False)

fold_splits_export = {
    f"fold{f}": {"val": fold_splits[f]['val'], "train": fold_splits[f]['train'],
                 "n_val": len(fold_splits[f]['val']), "n_train": len(fold_splits[f]['train']),
                 "fold_const": float(fold_const[f])} for f in ALL_FOLDS
}
with open(os.path.join(LOCAL_ARTIFACT_DIR, 'ft39l_fold_splits.json'), 'w') as fh:
    _json.dump(fold_splits_export, fh, indent=2)

ckpt_list = []
for fold in ALL_FOLDS:
    for seed in SEEDS:
        filename = f'ckpt_f{fold}_s{seed}.pt'
        local_path = os.path.join(LOCAL_ARTIFACT_DIR, filename)
        size_bytes = os.path.getsize(local_path) if os.path.exists(local_path) else -1
        ckpt_list.append({'fold': fold, 'seed': seed, 'filename': filename, 'size_bytes': size_bytes,
                          'best_epoch': int(results[(fold, seed)]['best_epoch']),
                          'best_rmse':  float(results[(fold, seed)]['best_rmse'])})

manifest = {
    "format_version": 1,
    "experiment_id": f"ARM_{ARM_TAG}",
    "recipe": summary['recipe'],
    "image_constants": summary['image_constants'],
    "input_channels": summary['input_channels'],
    "full_cv_metric": {"ens_rmse": full_cv_rmse, "ens_margin": full_cv_margin, "const_rmse": full_cv_const},
    "probe_verdict": summary['probe_verdict'],
    "diagnostics": diagnostics,
    "checkpoints": ckpt_list, "n_checkpoints": len(ckpt_list),
    "inference_strategy": {
        "description": f"Ensemble all {len(ckpt_list)} EMA-shadow checkpoints by averaging softmax-expected rows.",
        "model_kwargs": ({"model": "StackedUNet", "encoder_name": ENCODER, "encoder_weights": None,
                          "in_channels": IN_CH, "stage2_in_channels": IN_CH + 1, "classes": 1,
                          "stack_n": STACK_N, "handoff": "softmax_posterior_plane",
                          "note": "stage1: in_ch->1 logits; softmax over H -> posterior plane; "
                                  "stage2: cat(inputs, posterior) (in_ch+1)->1 logits; eval uses stage2 only"}
                         if STACK_N >= 2 else
                         {"encoder_name": ENCODER, "encoder_weights": None,
                          "in_channels": IN_CH, "classes": 1}),
        "channel_note": (f"{IN_CH} input channels ({IN_CH_BASE} base). "
                         f"Train-time aug arm: {ARM_LABEL}. All augs are finetune-train-only — "
                         "inference is canonical (no warp)."),
    },
    "fold_splits": fold_splits_export,
}
with open(os.path.join(LOCAL_ARTIFACT_DIR, 'ft39l_kaggle_inference_manifest.json'), 'w') as fh:
    _json.dump(manifest, fh, indent=2)

print(f'\n=== Artifacts saved to {LOCAL_ARTIFACT_DIR} ===')
print(f'  ft39l_summary.json / ft39l_cells.csv / ft39l_ensemble.csv')
print(f'  ft39l_fold_splits.json / ft39l_kaggle_inference_manifest.json')
print(f'  ckpt_f{{0..4}}_s{{seeds}}.pt — {len(ckpt_list)} EMA-shadow checkpoints')

if IN_COLAB and DRIVE_ARTIFACT_DIR:
    import shutil as _sh
    for f in os.listdir(LOCAL_ARTIFACT_DIR):
        _p = os.path.join(LOCAL_ARTIFACT_DIR, f)
        if os.path.isdir(_p):        # skip any subdir (flat mirror can't copy dirs)
            continue
        _sh.copy(_p, DRIVE_ARTIFACT_DIR)
    print(f'\nMirrored to {DRIVE_ARTIFACT_DIR}')